<a href="https://colab.research.google.com/github/AngeP02/MonteCarloPerRadioterapia/blob/main/Progetto_HPC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup ambiente di sviluppo

In [ ]:
!nvidia-smi
!nvcc --version
!gcc --version
!python3 --version
!pip install numpy matplotlib

/bin/bash: line 1: nvidia-smi: command not found
/bin/bash: line 1: nvcc: command not found
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Python 3.12.13


Creazione della struttura delle celle

In [64]:
import os

os.makedirs('CPU_Sequenziale', exist_ok=True)
os.makedirs('CPU_Parallelo', exist_ok=True)
os.makedirs('GPU_V1', exist_ok=True)
os.makedirs('GPU_V2', exist_ok=True)
os.makedirs('GPU_V3', exist_ok=True)
os.makedirs('GPU_V4', exist_ok=True)

print("Cartella creata")

Cartella creata


# Nuova sezione

In [88]:
numero_fotoni = 1000000

# Codice CPU Sequenziale

## Implementazione

### File compton.h

Effetto Compton comprende:
1.   Metodo di Kahn che campiona l'angolo di scattering Compton dalla distribuzione di Klein-Nishina attraverso il rejection sampling.
      * Input
        * energy_mev: energia fotone incidente MeV
        * xi_1, xi_2, xi_3: numeri casuali
      * Output
        * cos_theta: cosenzo angolo di scattering
        * energia_scatter: energia fotone diffuso

      Si hanno due fasi:
        * decomposizione in cui viene divisa la sezione d'urto in due rami probabilistici e si sceglie un ramo in base al numero casuale e si genera un valore di tau;
        * criterio di accettazione, in cui questo valore tau viene accettato solo se supera un test statistico basato sulla conservazione del momento e dell'energia. Se il test non viene superato si ricomincia.

2.   Wrapper con loop di rejection che garantisce che la funzione non restituisca mai un valore fisicamente impossivile.
3.   Rotazione della direzione del fotone dopo lo scattering. In input prende il versore attuale, viene calcolato un nuovo angolo e applicata una matrice di rotazione così da ottnere un nuovo versore di direzione che il fotone seguirà nel voxel successivo.

        * Input
            * ux, uy, uz che sono i versori e rappresentano la direzione del fotone prima dell'urto.



In [65]:
%%writefile ./CPU_Sequenziale/compton.h
#pragma once

#include <cmath>
#include "physics.h"

// Kahn
inline void kahn_compton(double energia_mev, double xi_1, double xi_2, double xi_3, double &cos_theta, double &energia_scatter) {
    double alpha = energia_mev / ME_C2;
    double tau_min = 1.0 / (1.0 + 2.0 * alpha);

    double area_ramo_1  = std::log(1.0 / tau_min);
    double area_ramo_2  = (1.0 - tau_min * tau_min) * 0.5;
    double area_totale = area_ramo_1 + area_ramo_2;
    double tau;

    if (xi_1 * area_totale < area_ramo_1) {
        // si campiona ramo logaritmico
        tau = std::pow(tau_min, 1.0 - xi_2);
    } else {
        // si fa interpolazione lineare e si campiona ramo lineare
        double tau_min_quadro = tau_min * tau_min;
        double tau_quadro = tau_min_quadro + xi_2*(1.0 - tau_min_quadro);
        tau = std::sqrt(std::max(tau_quadro, 1e-30));
    }

    tau = std::min(std::max(tau, tau_min), 1.0);

    cos_theta = 1.0 - (1.0 - tau) / (alpha * tau);
    cos_theta = std::min(std::max(cos_theta, -1.0), 1.0);
    energia_scatter = tau * energia_mev;

    // calcolo della probabilità di accettazione
    double sin2_theta = std::max(0.0, 1.0 - cos_theta * cos_theta);
    double termine_correttivo = (tau * sin2_theta) / (1.0 + tau * tau);
    double probabilita_accettazione = 1.0 - termine_correttivo;
    probabilita_accettazione = std::max(0.0, std::min(probabilita_accettazione, 1.0));

    if (xi_3 > probabilita_accettazione) {
        cos_theta = 2.0;
    }
}

// Wrapper con loop rejection
template<typename RNG>
inline void sample_compton(double energia_mev, RNG &rng, double &cos_theta, double &energia_scatter) {
    while(true){ // loop rejection perchè prima o poi un valore valido viene estratto
        double xi_1 = rng();
        double xi_2 = rng();
        double xi_3 = rng();
        kahn_compton(energia_mev, xi_1, xi_2, xi_3, cos_theta, energia_scatter);
        if (cos_theta <= 1.0)
            return;
    }
}

// Rotazione della direzione del fotone dopo lo scattering
inline void rotate_direction(double &ux, double &uy, double &uz, double cos_theta, double phi) {
    double sin_theta = std::sqrt(std::max(0.0, 1.0 - cos_theta * cos_theta));
    double cos_phi = std::cos(phi);
    double sen_phi = std::sin(phi);

    double ux_new, uy_new, uz_new;

    if (std::fabs(uz) > 0.99999) { // se viaggia quasi parallelo all'asse Z
        double segno = 1.0;
        if(uz > 0.0){
          segno = 1.0;
        }else{
          segno = -1.0;
        }
        ux_new = sin_theta * cos_phi;
        uy_new = sin_theta * sen_phi * segno;
        uz_new = cos_theta * segno;
    } else {
        double proiezione_xy = std::sqrt(1.0 - uz * uz);
        ux_new = sin_theta * (ux * uz * cos_phi - uy * sen_phi) / proiezione_xy + ux * cos_theta;
        uy_new = sin_theta * (uy * uz * cos_phi + ux * sen_phi) / proiezione_xy + uy * cos_theta;
        uz_new = -sin_theta * cos_phi * proiezione_xy + uz * cos_theta;
    }

    // Normalizzazione
    double norm = std::sqrt(ux_new * ux_new + uy_new * uy_new + uz_new * uz_new);
    if (norm > 0.0){
       ux = ux_new/norm;
       uy = uy_new/norm;
       uz = uz_new/norm;
    }
}

Writing ./CPU_Sequenziale/compton.h


### File phantom.h

Si occupa della costruzione del phantom voxelizzato. In particolare si hanno:
* Phantom Omogeneo
  * Si considera composto da acqua pura di 30x30x30 cm^3, quindi 100^3 voxel con 3mm a lato
* Phantom acqua + inserto osseo
  * si considera composto oltre che da acqua da un inserto osseo di 5x5x5 cm^3 al centro

Inizializza un volume cubico stadard dove ogni singolo voxel è marcato con acqua e a quello con inserto osseo viene inserito al centro della griglia un cubo di materiale osseo. *Viene, inoltre, considerato che metà lato dell'inserto è pari a 2.5cm / 0.3cm/voxel che è circa uguale e 8 voxel*

In [66]:
%%writefile ./CPU_Sequenziale/phantom.h

#pragma once

#include <cstring>
#include <cstdio>
#include "physics.h"

// Phantom Omogeneo (solo acqua)
inline void build_phantom_water(int *phantom) {
    int total = NX * NY * NZ;
    for (int i = 0; i < total; i++)
        phantom[i] = MAT_WATER;
}

// Phantom acqua + inserto osseo
inline void build_phantom_hetero(int *phantom) {
    build_phantom_water(phantom);

    // Centro del phantom in indici voxel
    int cx = NX / 2;
    int cy = NY / 2;
    int cz = NZ / 2;

    int meta_lato_inserto = (int)(2.5 / VOXEL_CM + 0.5);

    int count = 0;
    for (int iz = cz - meta_lato_inserto; iz < cz + meta_lato_inserto; iz++)
    for (int iy = cy - meta_lato_inserto; iy < cy + meta_lato_inserto; iy++)
    for (int ix = cx - meta_lato_inserto; ix < cx + meta_lato_inserto; ix++) {
        if (ix >= 0 && ix < NX && iy >= 0 && iy < NY && iz >= 0 && iz < NZ) { // controlli per non scrivere fuori dalla memoria del phantom
            phantom[phantom_idx(ix, iy, iz)] = MAT_BONE;
            count++;
        }
    }

    double volume_fisico_totale = count * VOXEL_CM * VOXEL_CM * VOXEL_CM;
    printf("Inserto osseo: %d voxel = %.1f cm³  ( volume teorico 125 cm³)\n", count, volume_fisico_totale);
}

// Inizializzazione dose grid a zero
inline void init_dose(double *dose) {
    memset(dose, 0, NX * NY * NZ * sizeof(double));
}


Writing ./CPU_Sequenziale/phantom.h


### File physics.h

Contiene le costanti fondamentali al calcolo e le tabelle dei coefficienti di interazione derivate dal database ufficiale NIST XCOM (https://physics.nist.gov/PhysRefData/XrayMassCoef/tab4.html), oltre alle funzioni matematiche per l'estrazione delle probabilità fisiche. Vengono definite le soglie ECUT e PCUT, la risoluzione spaziale della griglia voxelizzata e la cnversione fisica tra idnici dei voxel.

La griglia energetica viene considerata come l'insieme dei punti discreti su cui sono stati misurati e tabulati i dati fisici e secondo il db NIST XCOM si considera da 28 punti, quindi da 0.01 MeV a 20 MeV. In questo modo se il fotone ha un'energia che cade su uno di questi punti viene letto direttamente il valore dalla tabella altrimenti si usa la griglia per sapere tra quali punti effettuare l'interpolazione.

In [67]:
%%writefile ./CPU_Sequenziale/physics.h

#pragma once

#include <cmath>
#include <cassert>

// COSTANTI FISIHCHE
static const double ME_C2    = 0.51099895;  // MeV  massa a riposo elettrone
static const double PI       = 3.14159265358979323846;
static const double ECUT     = 0.010;       // MeV  cutoff fotoni  (10 keV)
static const double PCUT     = 0.100;       // MeV  cutoff elettroni

// GEOMETRIA PHANTOM
static const int    NX = 100, NY = 100, NZ = 100;   // voxel per asse
static const double VOXEL_CM = 0.30;                // lato voxel [cm] = 3 mm
static const double PHANTOM_CM = NX * VOXEL_CM;     // 30.0 cm per asse

// INDICI MATERIALI
#define MAT_WATER 0   // acqua  ρ = 1.000 g/cm^3
#define MAT_BONE  1   // osso (ICRU)  ρ = 1.850 g/cm^3
#define MAT_LUNG  2   // polmone (ICRU)  ρ = 0.260 g/cm^3
#define MAT_AIR   3   // aria  ρ = 0.001205 g/cm^3
#define N_MAT     4   // numero materiali disponibili

// DENSITÀ [g/cm^3]
static const double DENSITY[N_MAT] = { 1.000, 1.850, 0.260, 0.001205 };

// GRIGLIA ENERGETICA [MeV]  (28 punti, da 0.01 a 20 MeV)
static const int N_ENERGY = 28;
static const double ENERGY_GRID[N_ENERGY] = {
    0.010, 0.015, 0.020, 0.030, 0.040, 0.050, 0.060, 0.080, 0.100,
    0.150, 0.200, 0.300, 0.400, 0.500, 0.600, 0.800, 1.000, 1.250,
    1.500, 2.000, 3.000, 4.000, 5.000, 6.000, 8.000, 10.000,
    15.000, 20.000
};

// COEFFICIENTI μ/ρ [cm^2/g]  per ogni materiale e processo -> [materiale][bin_energia]
static const double MU_TOTAL[N_MAT][N_ENERGY] = {
    // ACQUA
    { 5.329, 1.673, 0.8096, 0.3756, 0.2683, 0.2269, 0.2059, 0.1837, 0.1707,
      0.1505, 0.1370, 0.1186, 0.1061, 0.09687, 0.09007, 0.07865, 0.07072, 0.06323,
      0.05754, 0.04942, 0.03969, 0.03403, 0.03031, 0.02770, 0.02429, 0.02219,
      0.01941, 0.01813 },
    // OSSO
    { 19.89, 7.131, 3.085, 1.012, 0.5475, 0.3941, 0.3178, 0.2595, 0.2368,
      0.1958, 0.1698, 0.1393, 0.1222, 0.1107, 0.1018, 0.08795, 0.07838, 0.06945,
      0.06283, 0.05351, 0.04257, 0.03624, 0.03209, 0.02913, 0.02536, 0.02296,
      0.01978, 0.01832 },
    // POLMONE
    { 5.208, 1.638, 0.7933, 0.3681, 0.2629, 0.2224, 0.2018, 0.1800, 0.1673,
      0.1475, 0.1342, 0.1162, 0.1040, 0.09493, 0.08827, 0.07708, 0.06930, 0.06196,
      0.05639, 0.04843, 0.03889, 0.03335, 0.02970, 0.02715, 0.02380, 0.02175,
      0.01902, 0.01776 },
    // ARIA
    { 36.01, 12.28, 5.279, 1.625, 0.6918, 0.3954, 0.2885, 0.2085, 0.1875,
      0.1504, 0.1340, 0.1123, 0.09921, 0.09076, 0.08414, 0.07285, 0.06529, 0.05817,
      0.05298, 0.04534, 0.03597, 0.03054, 0.02699, 0.02453, 0.02131, 0.01931,
      0.01673, 0.01551 }
};

// EFFETTO FOTOELETTRICO
static const double MU_PE[N_MAT][N_ENERGY] = {
    // ACQUA
    { 4.944, 1.374, 0.5195, 0.1036, 0.02407, 0.005800, 0.001334, 5.510e-5, 3.998e-5,
      2.799e-6, 2.200e-7, 1.400e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    // OSSO
    { 19.35, 6.833, 2.818, 0.7469, 0.2837, 0.1152, 0.04660, 0.008680, 0.001900,
      1.800e-4, 2.000e-5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    // POLMONE
    { 4.845, 1.346, 0.5091, 0.1015, 0.02359, 0.005684, 0.001307, 5.400e-5, 3.918e-5,
      2.743e-6, 2.156e-7, 1.372e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    // ARIA
    { 35.52, 11.99, 5.012, 1.379, 0.4529, 0.1581, 0.05757, 0.008251, 0.001581,
      8.208e-5, 7.636e-6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 }
};

// SCATTERING COMPTON
static const double MU_COMPTON[N_MAT][N_ENERGY] = {
    // ACQUA
    { 0.3854, 0.2988, 0.2672, 0.2651, 0.2595, 0.2476, 0.2329, 0.1984, 0.1661,
      0.1505, 0.1370, 0.1186, 0.1061, 0.09687, 0.09007, 0.07865, 0.07072, 0.06323,
      0.05754, 0.04942, 0.03969, 0.03403, 0.03031, 0.02770, 0.02429, 0.02219,
      0.01878, 0.01719 },
    // OSSO
    { 0.4869, 0.2684, 0.2503, 0.2465, 0.2429, 0.2310, 0.2172, 0.1848, 0.1548,
      0.1400, 0.1275, 0.1103, 0.09870, 0.09010, 0.08377, 0.07313, 0.06575, 0.05862,
      0.05338, 0.04579, 0.03667, 0.03133, 0.02784, 0.02539, 0.02217, 0.02016,
      0.01702, 0.01552 },
    // POLMONE
    { 0.3777, 0.2928, 0.2619, 0.2598, 0.2543, 0.2426, 0.2282, 0.1944, 0.1628,
      0.1475, 0.1342, 0.1162, 0.1040, 0.09493, 0.08827, 0.07708, 0.06930, 0.06196,
      0.05639, 0.04843, 0.03889, 0.03335, 0.02970, 0.02715, 0.02380, 0.02175,
      0.01840, 0.01684 },
    // ARIA
    { 0.3779, 0.2933, 0.2624, 0.2602, 0.2547, 0.2430, 0.2285, 0.1946, 0.1630,
      0.1477, 0.1344, 0.1163, 0.1041, 0.09516, 0.08844, 0.07723, 0.06942, 0.06207,
      0.05649, 0.04852, 0.03894, 0.03339, 0.02973, 0.02718, 0.02382, 0.02177,
      0.01843, 0.01686 }
};

// PRODUZIONE DI COPPIE
static const double MU_PAIR[N_MAT][N_ENERGY] = {
    // ACQUA
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.000630, 0.000940 },
    // OSSO
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.002760, 0.002800 },
    // POLMONE
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.000617, 0.000921 },
    // ARIA
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.000589, 0.000879 }
};

// INTERPOLAZIONE LINEARE SU GRIGLIA ENERGETICA
inline double interp_mu(double energia_mev, const double tabella[N_ENERGY]) {
    if (energia_mev <= ENERGY_GRID[0])
        return tabella[0];
    if (energia_mev >= ENERGY_GRID[N_ENERGY-1])
        return tabella[N_ENERGY-1];

    int indice_limite_inferiore = 0;
    int indice_imite_superiore = N_ENERGY - 1;

    while (indice_imite_superiore - indice_limite_inferiore > 1) {
        int punto_centrale = (indice_limite_inferiore + indice_imite_superiore) / 2;
        if (ENERGY_GRID[punto_centrale] <= energia_mev){
            indice_limite_inferiore = punto_centrale;
        }else{
            indice_imite_superiore = punto_centrale;
        }
    }

    double fattore_interpolazione = (energia_mev - ENERGY_GRID[indice_limite_inferiore]) / (ENERGY_GRID[indice_imite_superiore] - ENERGY_GRID[indice_limite_inferiore]);
    return tabella[indice_limite_inferiore] * (1.0 - fattore_interpolazione) + tabella[indice_imite_superiore] * fattore_interpolazione;
}

// CALCOLO MU TOTALE
inline double get_mu_total(double energia, int materiale) {   // probabilita di avere un urto
    return interp_mu(energia, MU_TOTAL[materiale]) * DENSITY[materiale];
}
inline double get_mu_pe(double energia, int materiale) {      // probabilita effetto fotoelettrico
    return interp_mu(energia, MU_PE[materiale]) * DENSITY[materiale];
}
inline double get_mu_compton(double energia, int materiale) {   // probabilità di compton
    return interp_mu(energia, MU_COMPTON[materiale]) * DENSITY[materiale];
}
inline double get_mu_pair(double energia, int materiale) {    // probabilita produzione coppie
    return interp_mu(energia, MU_PAIR[materiale]) * DENSITY[materiale];
}

// SELEZIONE TIPO DI INTERAZIONE
// Restituisce: 0=fotoelettrico, 1=Compton, 2=produzione coppie
// xi: numero casuale uniforme in [0,1)
inline int select_interaction(double energia, int materiale, double xi) {
    double mu_totale = get_mu_total(energia, materiale);

    if (mu_totale <= 0.0)
        return 1;

    double probabilita_fotoelettrico = get_mu_pe(energia, materiale) / mu_totale;
    double probabilita_compton = get_mu_compton(energia, materiale) / mu_totale;

    if (xi < probabilita_fotoelettrico)
        return 0;   // fotoelettrico
    if (xi < probabilita_fotoelettrico + probabilita_compton)
        return 1; // compton
    return 2;  // produzione di coppie
}

// INDICE LINEARE PHANTOM CON ROW MAJOR ORDER
inline int phantom_idx(int ix, int iy, int iz) {
    return ix + NX * (iy + NY * iz);
}

// CONTROLLO COORDINATE CONTORNO CUBO
inline bool inside(double x, double y, double z) {
    return x >= 0.0 && x < PHANTOM_CM &&
           y >= 0.0 && y < PHANTOM_CM &&
           z >= 0.0 && z < PHANTOM_CM;
}

// CONTROLLO PSIZIONE IN VOXEL
inline int vox(double coord) {
    int indice_voxel = (int)(coord / VOXEL_CM);
    if (indice_voxel < 0)
        indice_voxel = 0;
    if (indice_voxel >= NX)
        indice_voxel = NX - 1;
    return indice_voxel;
}


Writing ./CPU_Sequenziale/physics.h


### File random.h

Si occupa di generare numeri casuali ad alte prestazioni e campionare l'energia dei fotoni incidenti seguendo lo spettro reale di un acceleratore lineare LINAC. In particolare:
* è stato implementato l'algoritmo xoshiro256
  * viene utilizzato l'algoritmo splotmix64 per garantire che partendo da un singolo seed tutti i bit sono inizializzati in modo correlato
* è stato modellato un fascio reale di radioterapia (basato sui dati della lettetatura) e questo spettro è stato suddiviso in 24 intervalli energetici (bin) da 0.25 MeV fino a 6 MeV.
  * Si fa riferimento allo spettro 6MV del fascio standard Varian 2100C a 6MV, campo 10x10cm^2 a SAD 100cm

In [68]:
%%writefile ./CPU_Sequenziale/random.h

#pragma once

#include <cstdint>
#include <cmath>
#include <cstring>

// XOSHIRO256
struct Xoshiro256 {
    uint64_t s[4];

    // Inizializzazione con un seed a 64 bit usando splitmix64
    explicit Xoshiro256(uint64_t seed) { // con explicit si evitano conversioni automatiche
        auto splitmix = [](uint64_t &x) -> uint64_t {
            x += 0x9e3779b97f4a7c15ULL; // rappresentazione a 64 bit della sezione aurea (2^64/phi)
            uint64_t z = x;
            z = (z ^ (z >> 30)) * 0xbf58476d1ce4e5b9ULL;
            z = (z ^ (z >> 27)) * 0x94d049bb133111ebULL;
            return z ^ (z >> 31);
        };
        s[0] = splitmix(seed);
        s[1] = splitmix(seed);
        s[2] = splitmix(seed);
        s[3] = splitmix(seed);
    }

    // Genera un uint64_t casuale
    uint64_t next() {
        const uint64_t result = rotate_left(s[1] * 5, 7) * 9;
        const uint64_t t = s[1] << 17;
        s[2] ^= s[0]; s[3] ^= s[1]; // ^ indica opertaroe XOR bit a bit
        s[1] ^= s[2]; s[0] ^= s[3];
        s[2] ^= t;
        s[3] = rotate_left(s[3], 45);
        return result;
    }

    // Genera double uniforme in (0, 1) escludendo 0 per evitare log(0)
    double operator()() {
        double r;
        do {
            // usa i 53 bit superiori
            r = (double)(next() >> 11) * (1.0 / (double)(1ULL << 53)); // ULL intero a 64 bit senza segno
        } while (r <= 0.0);
        return r;
    }

private:
    static uint64_t rotate_left(const uint64_t x, int k) {
        return (x << k) | (x >> (64 - k));
    }
};

// SPETTRO 6MV
static const int SPECTRUM_BINS = 24;
static const double SPECTRUM_E[SPECTRUM_BINS] = {
    0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 1.75, 2.00,
    2.25, 2.50, 2.75, 3.00, 3.25, 3.50, 3.75, 4.00,
    4.25, 4.50, 4.75, 5.00, 5.25, 5.50, 5.75, 6.00
};

// Fluenza relativa normalizzata  (somma = 1.0)
static const double SPECTRUM_FLUENCE[SPECTRUM_BINS] = {
    0.0243, 0.0676, 0.0862, 0.0929, 0.0919, 0.0868, 0.0794, 0.0712,
    0.0628, 0.0548, 0.0471, 0.0399, 0.0334, 0.0276, 0.0224, 0.0178,
    0.0138, 0.0104, 0.0075, 0.0052, 0.0034, 0.0020, 0.0010, 0.0004
};

// CDF precalcolata all'inizializzazione
struct Spectrum {
    double cdf[SPECTRUM_BINS];
    double energie[SPECTRUM_BINS];
    double larghezza_intervallo_energetico_bin;

    Spectrum() {
        double somma_fluence = 0.0;
        for (int i = 0; i < SPECTRUM_BINS; i++) {
          somma_fluence += SPECTRUM_FLUENCE[i];
        }

        cdf[0] = SPECTRUM_FLUENCE[0] / somma_fluence;
        for (int i = 1; i < SPECTRUM_BINS; i++)
            cdf[i] = cdf[i-1] + SPECTRUM_FLUENCE[i] / somma_fluence;
        cdf[SPECTRUM_BINS-1] = 1.0;

        for (int i = 0; i < SPECTRUM_BINS; i++)
            energie[i] = SPECTRUM_E[i];

        larghezza_intervallo_energetico_bin = 0.25;
    }

    // Campiona energia con binary search sulla CDF
    double sample(Xoshiro256 &rng) const {
        double xi = rng();
        // Ricerca binaria sulla CDF
        int indice_limite_inferiore = 0;
        int indice_limite_superiore = SPECTRUM_BINS - 1;
        while (indice_limite_inferiore < indice_limite_superiore) {
            int punto_centrale = (indice_limite_inferiore + indice_limite_superiore) / 2;
            if (cdf[punto_centrale] < xi){
                indice_limite_inferiore = punto_centrale + 1;
            }
            else{
                indice_limite_superiore = punto_centrale;
            }
        }

        double energia_centrale = energie[indice_limite_inferiore];
        double offset = (rng() - 0.5) * larghezza_intervallo_energetico_bin;
        double energia = energia_centrale + offset;

        if (energia < 0.01){
           energia = 0.01;
        }
        if (energia > 6.00){
          energia = 6.00;
        }
        return energia;
    }
};


Writing ./CPU_Sequenziale/random.h


### File output.h

Contiene le routine necessarie per elaborare la matrice tridimensionale della dose e generare grafici e statistiche. In particolare:
* PDD o Percent Depht Dose è la misura di come la dose varia man mano che il fascio penetra in profondità nel paziente. Considera:
  * Media spaziale che calcola la media su una finestra di +- 8 voxel per simulare la risposta di una camera a ionizzazione reale;
  * normalizzazione che imposta la dose massima lungo l'asse mentre le altre vengono scalate
  * costruzione di profondità che calcola la coordinata fisica per ogni punto di misura.

* Calcola la funzione che analizza come la dose si distribuisce trasversalmente rispetto alla direzione del fascio
* Esportazione dei dati in csv
* Monitoraggio efficienza
  * numero di particelle al secondo;
  * energia depositata
  * voxel occupati

In [69]:
%%writefile ./CPU_Sequenziale/output.h

#pragma once

#include <cstdio>
#include <cmath>
#include <fstream>
#include <algorithm>
#include "physics.h"

// PDD
inline void compute_pdd(const double *dose, double *pdd, double *profondita, int semiampiezza_della_media = 8) {
    int cx = NX / 2; // centro del fascio in X
    int cy = NY / 2; // centro del fascio in Y

    double max_dose = 0.0;
    for (int iz = 0; iz < NZ; iz++) {
        double valore_dose = 0.0;
        int    num_voxel_sommati = 0;
        for (int ix = cx - semiampiezza_della_media; ix <= cx + semiampiezza_della_media; ix++){
          for (int iy = cy - semiampiezza_della_media; iy <= cy + semiampiezza_della_media; iy++) {
              if (ix >= 0 && ix < NX && iy >= 0 && iy < NY) {
                  valore_dose += dose[phantom_idx(ix, iy, iz)];
                  num_voxel_sommati++;
              }
          }
        }
        if (num_voxel_sommati > 0) {
            pdd[iz] = valore_dose / num_voxel_sommati;
        } else{
            0.0;
        }
        profondita[iz] = (iz + 0.5) * VOXEL_CM;
        if (pdd[iz] > max_dose){
            max_dose = pdd[iz];
        }
    }

    if (max_dose > 0.0)
        for (int iz = 0; iz < NZ; iz++)
            pdd[iz] = pdd[iz] / max_dose * 100.0;
}

// PROFILO LATERALE a profondità fissa (lungo X, centrato su Y)
inline void compute_lateral_profile(const double *dose, double *profilo_dose_relativa, double *coordinate_cm, double profondita_scelta = 10.0, int semiampiezza_media = 2) {
    int iz = std::min((int)(profondita_scelta / VOXEL_CM), NZ - 1);
    int centro_asse_y = NY / 2;

    double dose_massima_profilo = 0.0;
    for (int ix = 0; ix < NX; ix++) {
        double somma_dose_accumulata = 0.0;
        int conteggio_voxel_validi = 0;
        for (int iy = centro_asse_y - semiampiezza_media; iy <= centro_asse_y + semiampiezza_media; iy++) {
            if (iy >= 0 && iy < NY) {
                somma_dose_accumulata += dose[phantom_idx(ix, iy, iz)];
                conteggio_voxel_validi++;
            }
        }
        if (conteggio_voxel_validi > 0){
            profilo_dose_relativa[ix] = somma_dose_accumulata / conteggio_voxel_validi;
        } else{
            profilo_dose_relativa[ix] = 0.0;
        }
        coordinate_cm[ix] = (ix + 0.5) * VOXEL_CM - PHANTOM_CM / 2.0;
        if (profilo_dose_relativa[ix] > dose_massima_profilo) {
            dose_massima_profilo = profilo_dose_relativa[ix];
        }
    }

    if (dose_massima_profilo > 0.0)
        for (int ix = 0; ix < NX; ix++)
            profilo_dose_relativa[ix] = profilo_dose_relativa[ix] / dose_massima_profilo * 100.0;
}

// SALVA PDD SU CSV
inline void save_pdd_csv(const double *vettore_profondita_cm, const double *pdd, const char *filename) {
    std::ofstream f(filename);
    f << "depth_cm,dose_percent\n";
    for (int iz = 0; iz < NZ; iz++)
        f << vettore_profondita_cm[iz] << "," << pdd[iz] << "\n";
    f.close();
    printf("Salvato: %s\n", filename);
}

// SALVA PROFILO LATERALE SU CSV
inline void save_profile_csv(const double *coordinate_cm, const double *profilo_dose_relativa, const char *filename) {
    std::ofstream f(filename);
    f << "position_cm,dose_percent\n";
    for (int ix = 0; ix < NX; ix++)
        f << coordinate_cm[ix] << "," << profilo_dose_relativa[ix] << "\n";
    f.close();
    printf("  Salvato: %s\n", filename);
}

// SALVA SLICE 2D CENTRALE SU CSV (per heatmap)
inline void save_dose_slice_csv(const double *dose, const char *filename) {
    std::ofstream f(filename);
    int iy = NY / 2;
    for (int iz = 0; iz < NZ; iz++) {
        for (int ix = 0; ix < NX; ix++) {
            f << dose[phantom_idx(ix, iy, iz)];
            if (ix < NX - 1) f << ",";
        }
        f << "\n";
    }
    f.close();
    printf("  Salvato: %s\n", filename);
}

// SALVA DOSE 3D COMPLETA
inline void save_dose_binary(const double *dose, const char *filename) {
    FILE *f = fopen(filename, "wb");
    if (!f) { printf("ERRORE: impossibile aprire %s\n", filename); return; }
    fwrite(dose, sizeof(double), NX * NY * NZ, f);
    fclose(f);
    printf("Salvato: %s  (%d float64)\n", filename, NX * NY * NZ);
}

// STAMPA TABELLA PDD AI PUNTI DI RIFERIMENTO CLINICI
inline void print_pdd_table(const double *profondita, const double *pdd, const char *label) {
    printf("\n  PDD — %s\n", label);
    printf("  %-20s  %10s  %s\n", "Profondità [cm]", "Dose [%]", "Riferimento");
    printf("  %s\n", "─────────────────────────────────────────");

    double refs[]      = {1.5, 3.0, 5.0, 10.0, 15.0, 20.0, 25.0};
    const char *notes[] = {"build-up", "d_max 6MV", "", "D10", "", "D20", ""};

    for (int r = 0; r < 7; r++) {
        int k = (int)(refs[r] / VOXEL_CM);
        if (k >= 0 && k < NZ)
            printf("  %-20.1f  %10.2f  %s\n", profondita[k], pdd[k], notes[r]);
    }
}

// STATISTICHE GENERALI SULLA DOSE
inline void print_dose_stats(const double *dose, long long numero_particelle_totali, double tempo_esecuzione_secondi) {
    double dose_massima_assoluta = 0.0;
    double energia_totale_depositata = 0.0;
    int conteggio_voxel_colpiti = 0;

    for (int i = 0; i < NX * NY * NZ; i++) {
        if (dose[i] > 0.0) {
            conteggio_voxel_colpiti++;
            energia_totale_depositata += dose[i];
            if (dose[i] > dose_massima_assoluta){
                dose_massima_assoluta = dose[i];
            }
        }
    }

    printf("\n Statistiche simulazione: \n");
    printf("  Particelle simulate : %lld\n",  numero_particelle_totali);
    printf("  Tempo totale        : %.2f s\n", tempo_esecuzione_secondi);
    printf("  Throughput          : %.3f MP/s\n", numero_particelle_totali / tempo_esecuzione_secondi / 1.0e6);
    printf("  Tempo/particella    : %.1f μs\n", tempo_esecuzione_secondi / numero_particelle_totali * 1.0e6);
    printf("  Voxel con dose>0    : %d / %d (%.1f%%)\n", conteggio_voxel_colpiti, NX*NY*NZ, 100.0*conteggio_voxel_colpiti/(NX*NY*NZ));
    printf("  Energia totale dep. : %.4e MeV\n", energia_totale_depositata);
    printf("  Energia/particella  : %.4e MeV\n", numero_particelle_totali > 0 ? energia_totale_depositata / numero_particelle_totali : 0.0);
    printf("  Dose massima (u.a.) : %.6e\n", dose_massima_assoluta);
}


Writing ./CPU_Sequenziale/output.h


### File main.cpp

Programma principale che unisce tutti i moduli ed esegue la simulazione Monte Carlo per radioterapia. Effettua:
* Campionamento della sorgente;
* Applica algoritmo di Traversal;
* Gestisce la storia di ogni fotone

Implementa il trasporto di fotoni in un phantom voxelizzato con:
 * Spettro 6MV (Sheikh-Bagheri & Rogers 2002)
 * Legge di Beer-Lambert + voxel traversal (Amanatides & Woo 1987)
 * Effetto fotoelettrico, Compton (metodo di Kahn), produzione coppie
 * Sezioni d'urto da NIST XCOM (Hubbell & Seltzer 1996)
 * Approssimazione KERMA ≈ dose (deposito locale elettroni)
 * PRNG: xoshiro256** (Blackman & Vigna 2018)

In [70]:
%%writefile ./CPU_Sequenziale/main.cpp

#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>
#include <cstring>
#include <vector>

#include "physics.h"
#include "compton.h"
#include "random.h"
#include "phantom.h"
#include "output.h"

// stato di una particella sullo stack
struct Particle {
    double x, y, z;    // posizione [cm]
    double ux, uy, uz; // versore direzione (normalizzato)
    double energia;     // energia [MeV]
};

// CAMPIONAMENTO SORGENTE
inline Particle sample_source(const Spectrum &spettro, Xoshiro256 &rng) {
    static const double FIELD_HALF = 5.0;
    double cx = PHANTOM_CM / 2.0;
    double cy = PHANTOM_CM / 2.0;

    Particle p;
    p.x = cx + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.y = cy + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.z = 1.0e-7;
    p.ux = 0.0;
    p.uy = 0.0;
    p.uz = 1.0;
    p.energia = spettro.sample(rng);
    return p;
}

// DISTANZA AL PROSSIMO CONFINE DI VOXEL
inline double boundary_distance(double x, double y, double z, double ux, double uy, double uz, int ix, int iy, int iz) {
    double distanza_minima_confine = 1.0e30; // inizializzata a infinito
    if (std::fabs(ux) > 1.0e-12) {
        double confine_voxel_X;
        if (ux > 0){
            confine_voxel_X = (ix + 1) * VOXEL_CM;
        } else{
            confine_voxel_X = ix * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_X - x) / ux;
        if (distanza_lineare > 1.0e-10){
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    if (std::fabs(uy) > 1.0e-12) {
        double confine_voxel_Y;
        if (uy > 0){
            confine_voxel_Y = (iy + 1) * VOXEL_CM;
        } else{
            confine_voxel_Y = iy * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_Y - y) / uy;
        if (distanza_lineare > 1.0e-10){
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    if (std::fabs(uz) > 1.0e-12) {
        double confine_voxel_Z;
        if (uz > 0){
            confine_voxel_Z = (iz + 1) * VOXEL_CM;
        } else{
            confine_voxel_Z = iz * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_Z - z) / uz;
        if (distanza_lineare > 1.0e-10) {
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    return distanza_minima_confine;
}

// TRASPORTO FOTONE CICLO COMPLETO
void transport_photon(Particle fotone_iniziale, const int *phantom, double *dose, Xoshiro256 &rng) {

    Particle stack[64];
    int num_particelle_stack = 0;
    stack[num_particelle_stack++] = fotone_iniziale;

    while (num_particelle_stack > 0) {
        Particle particella_corrente = stack[--num_particelle_stack];
        for (int step = 0; step < 100000; step++) {
            // Cutoff energetico
            if (particella_corrente.energia < ECUT) {
                // Deposita energia residua nel voxel corrente
                if (inside(particella_corrente.x, particella_corrente.y, particella_corrente.z)) {
                    int ix = vox(particella_corrente.x);
                    int iy = vox(particella_corrente.y);
                    int iz = vox(particella_corrente.z);
                    dose[phantom_idx(ix, iy, iz)] += particella_corrente.energia;
                }
                break;
            }
            // Verifica bounds
            if (!inside(particella_corrente.x, particella_corrente.y, particella_corrente.z))
                break;
            int ix = vox(particella_corrente.x);
            int iy = vox(particella_corrente.y);
            int iz = vox(particella_corrente.z);
            int materiale = phantom[phantom_idx(ix, iy, iz)];
            double mu = get_mu_total(particella_corrente.energia, materiale); // coefficiente di attenuazione totale

            if (mu <= 0.0)
                break;
            // Campiona cammino libero medio
            double xi = rng();
            double distanza_teorica = -std::log(xi) / mu;
            double distanza_fisica = boundary_distance(particella_corrente.x, particella_corrente.y, particella_corrente.z, particella_corrente.ux, particella_corrente.uy, particella_corrente.uz, ix, iy, iz);

            if (distanza_teorica <= distanza_fisica) {
                // Sposta la particella al punto di interazione
                particella_corrente.x += particella_corrente.ux * distanza_teorica;
                particella_corrente.y += particella_corrente.uy * distanza_teorica;
                particella_corrente.z += particella_corrente.uz * distanza_teorica;

                if (!inside(particella_corrente.x, particella_corrente.y, particella_corrente.z))
                    break;

                // Ricalcola voxel
                ix = vox(particella_corrente.x);
                iy = vox(particella_corrente.y);
                iz = vox(particella_corrente.z);
                materiale = phantom[phantom_idx(ix, iy, iz)];
                int id_voxel = phantom_idx(ix, iy, iz);
                int tipo_interazione = select_interaction(particella_corrente.energia, materiale, rng());

                // FOTOELETTRICO: assorbimento totale
                if (tipo_interazione == 0) {
                    dose[id_voxel] += particella_corrente.energia;
                    break;
                }
                // COMPTON: metodo di Kahn
                else if (tipo_interazione == 1) {
                    double cos_theta;
                    double energia_scatter;
                    sample_compton(particella_corrente.energia, rng, cos_theta, energia_scatter);

                    // Deposita energia ceduta all'elettrone (KERMA locale)
                    double energia_ceduta = particella_corrente.energia - energia_scatter;
                    if (energia_ceduta > 0.0) {
                        dose[id_voxel] += energia_ceduta;
                    }

                    // Aggiorna energia e direzione del fotone
                    particella_corrente.energia = energia_scatter;
                    double phi = 2.0 * PI * rng();
                    rotate_direction(particella_corrente.ux, particella_corrente.uy, particella_corrente.uz, cos_theta, phi);

                    if (particella_corrente.energia < ECUT) {
                        dose[id_voxel] += particella_corrente.energia;
                        break;
                    }
                }

                // PRODUZIONE DI COPPIE
                else {
                    // Energia cinetica disponibile per elettrone e positrone
                    double energia_cinetica_residua = particella_corrente.energia - 2.0 * ME_C2;
                    if (energia_cinetica_residua > 0.0) {
                        dose[id_voxel] += energia_cinetica_residua;
                    }

                    if (ME_C2 > ECUT && num_particelle_stack + 2 <= 62) {
                        double cos_theta = 2.0 * rng() - 1.0;
                        double phi_a = 2.0 * PI * rng();
                        double sen_theta = std::sqrt(std::max(0.0, 1.0 - cos_theta * cos_theta));

                        Particle fotone_secondario_1, fotone_secondario_2;
                        fotone_secondario_1.x = fotone_secondario_2.x = particella_corrente.x;
                        fotone_secondario_1.y = fotone_secondario_2.y = particella_corrente.y;
                        fotone_secondario_1.z = fotone_secondario_2.z = particella_corrente.z;
                        fotone_secondario_1.ux =  sen_theta * std::cos(phi_a);
                        fotone_secondario_1.uy =  sen_theta * std::sin(phi_a);
                        fotone_secondario_1.uz =  cos_theta;
                        fotone_secondario_2.ux = -fotone_secondario_1.ux;
                        fotone_secondario_2.uy = -fotone_secondario_1.uy;
                        fotone_secondario_2.uz = -fotone_secondario_1.uz;
                        fotone_secondario_1.energia = fotone_secondario_2.energia = ME_C2;

                        stack[num_particelle_stack++] = fotone_secondario_1;
                        stack[num_particelle_stack++] = fotone_secondario_2;
                    }
                    break;
                }

            } else {
                double eps = 1.0e-7;
                particella_corrente.x += particella_corrente.ux * (distanza_fisica + eps);
                particella_corrente.y += particella_corrente.uy * (distanza_fisica + eps);
                particella_corrente.z += particella_corrente.uz * (distanza_fisica + eps);
            }

        }
    }
}

int main(int argc, char *argv[]) {

    // valori default
    long long num_fotoni = 1000000;   // default: 1M fotoni
    int tipo_phantom = 0;         // 0=acqua, 1=eterogeneo
    uint64_t seed   = 42ULL;

    if (argc > 1) num_fotoni = std::atoll(argv[1]);
    if (argc > 2) tipo_phantom = std::atoi(argv[2]);
    if (argc > 3) seed = (uint64_t)std::atoll(argv[3]);

    const char *phantom_label;
    if (tipo_phantom == 0){
      phantom_label = "Acqua omogenea";
    } else{
      phantom_label = "Acqua + Osso";
    }

    printf("  Monte Carlo per Radioterapia — CPU Sequenziale\n");
    printf("\n  Parametri:\n");
    printf("  Phantom    : %dx%dx%d voxel  |  voxel %.0fmm  |  %.0f³ cm³\n", NX, NY, NZ, VOXEL_CM * 10.0, PHANTOM_CM);
    printf("  Materiale  : %s\n", phantom_label);
    printf("  N fotoni   : %lld\n", num_fotoni);
    printf("  Seed       : %llu\n", (unsigned long long)seed);
    printf("  ECUT       : %.0f keV\n\n", ECUT * 1000.0);

    int *phantom = new int [NX * NY * NZ];
    double *dose = new double[NX * NY * NZ]();
    double *pdd = new double[NZ];
    double *coordinate_cm = new double[NZ];
    double *profilo_dose = new double[NX];
    double *coordinate_cm_laterali = new double[NX];

    if (tipo_phantom == 0){
        printf("Costruzione phantom con acqua \n");
        build_phantom_water(phantom);
    }else {
        printf("Costruzione phantom eterogeneo \n");
        build_phantom_hetero(phantom);
    }

    Spectrum spettro;  // spettro 6MV con CDF
    Xoshiro256 rng(seed); // PRNG xoshiro256**

    printf(" Avvio simulazione \n");
    auto tempo_inizio_esatto = std::chrono::high_resolution_clock::now();

    long long report_step = std::max(1LL, num_fotoni / 20);   // report ogni 5%

    for (long long i = 0; i < num_fotoni; i++) {
        if ((i + 1) % report_step == 0) {
            auto tempo_attuale     = std::chrono::high_resolution_clock::now();
            double tempo_esecuzione = std::chrono::duration<double>(tempo_attuale - tempo_inizio_esatto).count();
            double rate  = (i + 1) / tempo_esecuzione;
            double tempo_necessrio_fotoni_rimanenti   = (num_fotoni - i - 1) / rate;
            printf(" [%5.1f%%]  %.0f fotoni/s  Estimated Time of Arrival %.0fs\n", 100.0 * (i + 1) / num_fotoni, rate, tempo_necessrio_fotoni_rimanenti);
        }

        Particle p = sample_source(spettro, rng);
        transport_photon(p, phantom, dose, rng);
    }

    auto tempo_fine_esatto = std::chrono::high_resolution_clock::now();
    double tempo_esecuzione = std::chrono::duration<double>(tempo_fine_esatto - tempo_inizio_esatto).count();

    print_dose_stats(dose, num_fotoni, tempo_esecuzione);

    compute_pdd(dose, pdd, coordinate_cm);
    compute_lateral_profile(dose, profilo_dose, coordinate_cm_laterali, 10.0);
    print_pdd_table(coordinate_cm, pdd, phantom_label);

    const char *pdd_file;
    const char *profilo_file;
    const char *slice_file;
    const char *bin_file;

    if (tipo_phantom == 0){
      pdd_file = "./CPU_Sequenziale/pdd_water.csv";
      profilo_file = "./CPU_Sequenziale/profile_water.csv";
      slice_file = "./CPU_Sequenziale/dose_slice_water.csv";
      bin_file = "./CPU_Sequenziale/dose_water.bin";
    } else{
      pdd_file = "./CPU_Sequenziale/pdd_hetero.csv";
      profilo_file = "./CPU_Sequenziale/profile_hetero.csv";
      slice_file = "./CPU_Sequenziale/dose_slice_hetero.csv";
      bin_file = "./CPU_Sequenziale/dose_hetero.bin";
    }
    save_pdd_csv(coordinate_cm, pdd, pdd_file);
    save_profile_csv(coordinate_cm_laterali, profilo_dose, profilo_file);
    save_dose_slice_csv(dose, slice_file);
    save_dose_binary(dose, bin_file);

    delete[] phantom;
    delete[] dose;
    delete[] pdd;
    delete[] coordinate_cm;
    delete[] profilo_dose;
    delete[] coordinate_cm_laterali;

    printf("  Simulazione completata.\n");

    return 0;
}


Writing ./CPU_Sequenziale/main.cpp


### File BeerLambert.cpp

E' stata implementata una versione semplificata di un simulatore Monte Carlo per radioterapia per validare la Legge di Beer-Lambert. Viene simulato il trasporto di fotoni (con uno spettro energetico da 6MV) attraverso un mezzo materiale per dimostrare che l'attenuazione della radiazione segua un decadimento esponenziale perfetto quando viene considerato solo l'assorbimento primario.

Per fare questo non appena il fotone interagisce con un voxel del phantom tutta la sua energia viene depositata immediatamente. Vengono ignorati i processi di scattering, come l'effetto Compton, dove il fotone cambierebbe solo direzione ed energia continuando il suo viaggio. In questo modo, il numero di fotoni che raggiungono una certa profondità dipende esclusivamente dalla probabilità di non aver interagito prima.

In [71]:
%%writefile ./CPU_Sequenziale/BeerLambert.cpp

#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>
#include <cstring>
#include <vector>

#include "physics.h"
#include "compton.h"
#include "random.h"
#include "phantom.h"
#include "output.h"

// stato di una particella sullo stack
struct Particle {
    double x, y, z;    // posizione [cm]
    double ux, uy, uz; // versore direzione (normalizzato)
    double energia;     // energia [MeV]
};

// CAMPIONAMENTO SORGENTE
inline Particle sample_source(const Spectrum &spettro, Xoshiro256 &rng) {
    static const double FIELD_HALF = 5.0;   // +-5 cm -> campo 10×10 cm^2
    double cx = PHANTOM_CM / 2.0;
    double cy = PHANTOM_CM / 2.0;

    Particle p;
    p.x = cx + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.y = cy + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.z = 1.0e-7;
    p.ux = 0.0;
    p.uy = 0.0;
    p.uz = 1.0;
    p.energia = spettro.sample(rng);
    return p;
}

// DISTANZA AL PROSSIMO CONFINE DI VOXEL
inline double boundary_distance(double x, double y, double z, double ux, double uy, double uz, int ix, int iy, int iz) {
    double distanza_minima_confine = 1.0e30; // inizializzata a infinito
    if (std::fabs(ux) > 1.0e-12) {
        double confine_voxel_X;
        if (ux > 0){
            confine_voxel_X = (ix + 1) * VOXEL_CM;
        } else{
            confine_voxel_X = ix * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_X - x) / ux;
        if (distanza_lineare > 1.0e-10){
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    if (std::fabs(uy) > 1.0e-12) {
        double confine_voxel_Y;
        if (uy > 0){
            confine_voxel_Y = (iy + 1) * VOXEL_CM;
        } else{
            confine_voxel_Y = iy * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_Y - y) / uy;
        if (distanza_lineare > 1.0e-10){
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    if (std::fabs(uz) > 1.0e-12) {
        double confine_voxel_Z;
        if (uz > 0){
            confine_voxel_Z = (iz + 1) * VOXEL_CM;
        } else{
            confine_voxel_Z = iz * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_Z - z) / uz;
        if (distanza_lineare > 1.0e-10) {
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    return distanza_minima_confine;
}

// TRASPORTO FOTONE SEMPLIFICATO PER BEER LAMBERT
void transport_photon(Particle p, const int *phantom, double *dose, Xoshiro256 &rng) {
    while (p.energia > ECUT && inside(p.x, p.y, p.z)) {
        int mat = phantom[phantom_idx((int)(p.x/VOXEL_CM), (int)(p.y/VOXEL_CM), (int)(p.z/VOXEL_CM))];
        double mu = get_mu_total(p.energia, mat);
        double d = -std::log(rng()) / mu;

        p.x += p.ux * d;
        p.y += p.uy * d;
        p.z += p.uz * d;

        if (inside(p.x, p.y, p.z)) {
            int ix = (int)(p.x / VOXEL_CM);
            int iy = (int)(p.y / VOXEL_CM);
            int iz = (int)(p.z / VOXEL_CM);

            dose[phantom_idx(ix, iy, iz)] += p.energia;

            break;
        }
    }
}

int main(int argc, char *argv[]) {

    // valori default
    long long num_fotoni = 1000000;   // default: 1M fotoni
    int tipo_phantom = 0;         // 0=acqua, 1=eterogeneo
    uint64_t seed   = 42ULL;

    if (argc > 1) num_fotoni = std::atoll(argv[1]);
    if (argc > 2) tipo_phantom = std::atoi(argv[2]);
    if (argc > 3) seed = (uint64_t)std::atoll(argv[3]);

    const char *phantom_label;
    if (tipo_phantom == 0){
      phantom_label = "Acqua omogenea";
    } else{
      phantom_label = "Acqua + Osso";
    }

    printf("  Monte Carlo per Radioterapia — CPU Sequenziale\n");
    printf("\n  Parametri:\n");
    printf("  Phantom    : %dx%dx%d voxel  |  voxel %.0fmm  |  %.0f³ cm³\n", NX, NY, NZ, VOXEL_CM * 10.0, PHANTOM_CM);
    printf("  Materiale  : %s\n", phantom_label);
    printf("  N fotoni   : %lld\n", num_fotoni);
    printf("  Seed       : %llu\n", (unsigned long long)seed);
    printf("  ECUT       : %.0f keV\n\n", ECUT * 1000.0);

    int *phantom = new int [NX * NY * NZ];
    double *dose = new double[NX * NY * NZ]();
    double *pdd = new double[NZ];
    double *coordinate_cm = new double[NZ];
    double *profilo_dose = new double[NX];
    double *coordinate_cm_laterali = new double[NX];

    if (tipo_phantom == 0){
        printf("Costruzione phantom con acqua \n");
        build_phantom_water(phantom);
    }else {
        printf("Costruzione phantom eterogeneo \n");
        build_phantom_hetero(phantom);
    }

    Spectrum spettro;  // spettro 6MV con CDF
    Xoshiro256 rng(seed); // PRNG xoshiro256**

    printf(" Avvio simulazione \n");
    auto tempo_inizio_esatto = std::chrono::high_resolution_clock::now();

    long long report_step = std::max(1LL, num_fotoni / 20);   // report ogni 5%

    for (long long i = 0; i < num_fotoni; i++) {
        if ((i + 1) % report_step == 0) {
            auto tempo_attuale     = std::chrono::high_resolution_clock::now();
            double tempo_esecuzione = std::chrono::duration<double>(tempo_attuale - tempo_inizio_esatto).count();
            double rate  = (i + 1) / tempo_esecuzione;
            double tempo_necessrio_fotoni_rimanenti   = (num_fotoni - i - 1) / rate;
            printf(" [%5.1f%%]  %.0f fotoni/s  Estimated Time of Arrival %.0fs\n", 100.0 * (i + 1) / num_fotoni, rate, tempo_necessrio_fotoni_rimanenti);
        }

        Particle p = sample_source(spettro, rng);
        transport_photon(p, phantom, dose, rng);
    }

    auto tempo_fine_esatto = std::chrono::high_resolution_clock::now();
    double tempo_esecuzione = std::chrono::duration<double>(tempo_fine_esatto - tempo_inizio_esatto).count();

    print_dose_stats(dose, num_fotoni, tempo_esecuzione);

    compute_pdd(dose, pdd, coordinate_cm);
    compute_lateral_profile(dose, profilo_dose, coordinate_cm_laterali, 10.0);
    print_pdd_table(coordinate_cm, pdd, phantom_label);

    const char *pdd_file;
    const char *profilo_file;
    const char *slice_file;
    const char *bin_file;

    if (tipo_phantom == 0){
      pdd_file = "./CPU_Sequenziale/pdd_water_BL.csv";
      profilo_file = "./CPU_Sequenziale/profile_water_BL.csv";
      slice_file = "./CPU_Sequenziale/dose_slice_water_BL.csv";
      bin_file = "./CPU_Sequenziale/dose_water_BL.bin";
    } else{
      pdd_file = "./CPU_Sequenziale/pdd_hetero_BL.csv";
      profilo_file = "./CPU_Sequenziale/profile_hetero_BL.csv";
      slice_file = "./CPU_Sequenziale/dose_slice_hetero_BL.csv";
      bin_file = "./CPU_Sequenziale/dose_hetero_BL.bin";
    }
    save_pdd_csv(coordinate_cm, pdd, pdd_file);
    save_profile_csv(coordinate_cm_laterali, profilo_dose, profilo_file);
    save_dose_slice_csv(dose, slice_file);
    save_dose_binary(dose, bin_file);

    delete[] phantom;
    delete[] dose;
    delete[] pdd;
    delete[] coordinate_cm;
    delete[] profilo_dose;
    delete[] coordinate_cm_laterali;

    printf("  Simulazione completata.\n");

    return 0;
}


Writing ./CPU_Sequenziale/BeerLambert.cpp


## Compilazione

Compilazione main completo

In [72]:
!g++ -O2 -std=c++17 -o ./CPU_Sequenziale/mc_rt_cpu_sequenziale ./CPU_Sequenziale/main.cpp -lm

Compilazione main per validazione Beer Lambert

In [73]:
!g++ -O2 -std=c++17 -o ./CPU_Sequenziale/test_beer_lambert_sequenziale ./CPU_Sequenziale/BeerLambert.cpp -lm

## Esecuzione

In [91]:
numero_fotoni = 10000000

In [92]:
print(numero_fotoni)

10000000


In [93]:
!./CPU_Sequenziale/mc_rt_cpu_sequenziale $numero_fotoni 0 42

  Monte Carlo per Radioterapia — CPU Sequenziale

  Parametri:
  Phantom    : 100x100x100 voxel  |  voxel 3mm  |  30³ cm³
  Materiale  : Acqua omogenea
  N fotoni   : 10000000
  Seed       : 42
  ECUT       : 10 keV

Costruzione phantom con acqua 
 Avvio simulazione 
 [  5.0%]  86395 fotoni/s  Estimated Time of Arrival 110s
 [ 10.0%]  94044 fotoni/s  Estimated Time of Arrival 96s
 [ 15.0%]  91465 fotoni/s  Estimated Time of Arrival 93s
 [ 20.0%]  93986 fotoni/s  Estimated Time of Arrival 85s
 [ 25.0%]  92259 fotoni/s  Estimated Time of Arrival 81s
 [ 30.0%]  92092 fotoni/s  Estimated Time of Arrival 76s
 [ 35.0%]  93578 fotoni/s  Estimated Time of Arrival 69s
 [ 40.0%]  92717 fotoni/s  Estimated Time of Arrival 65s
 [ 45.0%]  93881 fotoni/s  Estimated Time of Arrival 59s
 [ 50.0%]  93442 fotoni/s  Estimated Time of Arrival 54s
 [ 55.0%]  93959 fotoni/s  Estimated Time of Arrival 48s
 [ 60.0%]  94739 fotoni/s  Estimated Time of Arrival 42s
 [ 65.0%]  94060 fotoni/s  Estimated Time of Ar

In [ ]:
!./CPU_Sequenziale/mc_rt_cpu_sequenziale $numero_fotoni 1 42

In [ ]:
!./CPU_Sequenziale/test_beer_lambert_sequenziale $numero_fotoni 0 42

# Codice CPU Parallelo

## Implementazione

### File compton.h

Effetto Compton comprende:
1.   Metodo di Kahn che campiona l'angolo di scattering Compton dalla distribuzione di Klein-Nishina attraverso il rejection sampling.
      * Input
        * energy_mev: energia fotone incidente MeV
        * xi_1, xi_2, xi_3: numeri casuali
      * Output
        * cos_theta: cosenzo angolo di scattering
        * energia_scatter: energia fotone diffuso

      Si hanno due fasi:
        * decomposizione in cui viene divisa la sezione d'urto in due rami probabilistici e si sceglie un ramo in base al numero casuale e si genera un valore di tau;
        * criterio di accettazione, in cui questo valore tau viene accettato solo se supera un test statistico basato sulla conservazione del momento e dell'energia. Se il test non viene superato si ricomincia.

2.   Wrapper con loop di rejection che garantisce che la funzione non restituisca mai un valore fisicamente impossivile.
3.   Rotazione della direzione del fotone dopo lo scattering. In input prende il versore attuale, viene calcolato un nuovo angolo e applicata una matrice di rotazione così da ottnere un nuovo versore di direzione che il fotone seguirà nel voxel successivo.

        * Input
            * ux, uy, uz che sono i versori e rappresentano la direzione del fotone prima dell'urto.



In [ ]:
%%writefile ./CPU_Parallelo/compton.h
#pragma once

#include <cmath>
#include "physics.h"

// Kahn
inline void kahn_compton(double energia_mev, double xi_1, double xi_2, double xi_3, double &cos_theta, double &energia_scatter) {
    double alpha = energia_mev / ME_C2;
    double tau_min = 1.0 / (1.0 + 2.0 * alpha);

    double area_ramo_1  = std::log(1.0 / tau_min);
    double area_ramo_2  = (1.0 - tau_min * tau_min) * 0.5;
    double area_totale = area_ramo_1 + area_ramo_2;
    double tau;

    if (xi_1 * area_totale < area_ramo_1) {
        // si campiona ramo logaritmico
        tau = std::pow(tau_min, 1.0 - xi_2);
    } else {
        // si fa interpolazione lineare e si campiona ramo lineare
        double tau_min_quadro = tau_min * tau_min;
        double tau_quadro = tau_min_quadro + xi_2*(1.0 - tau_min_quadro);
        tau = std::sqrt(std::max(tau_quadro, 1e-30));
    }

    tau = std::min(std::max(tau, tau_min), 1.0);

    cos_theta = 1.0 - (1.0 - tau) / (alpha * tau);
    cos_theta = std::min(std::max(cos_theta, -1.0), 1.0);
    energia_scatter = tau * energia_mev;

    // calcolo della probabilità di accettazione
    double sin2_theta = std::max(0.0, 1.0 - cos_theta * cos_theta);
    double termine_correttivo = (tau * sin2_theta) / (1.0 + tau * tau);
    double probabilita_accettazione = 1.0 - termine_correttivo;
    probabilita_accettazione = std::max(0.0, std::min(probabilita_accettazione, 1.0));

    if (xi_3 > probabilita_accettazione) {
        cos_theta = 2.0;
    }
}

// Wrapper con loop rejection
template<typename RNG>
inline void sample_compton(double energia_mev, RNG &rng, double &cos_theta, double &energia_scatter) {
    while(true){ // loop rejection perchè prima o poi un valore valido viene estratto
        double xi_1 = rng();
        double xi_2 = rng();
        double xi_3 = rng();
        kahn_compton(energia_mev, xi_1, xi_2, xi_3, cos_theta, energia_scatter);
        if (cos_theta <= 1.0)
            return;
    }
}

// Rotazione della direzione del fotone dopo lo scattering
inline void rotate_direction(double &ux, double &uy, double &uz, double cos_theta, double phi) {
    double sin_theta = std::sqrt(std::max(0.0, 1.0 - cos_theta * cos_theta));
    double cos_phi = std::cos(phi);
    double sen_phi = std::sin(phi);

    double ux_new, uy_new, uz_new;

    if (std::fabs(uz) > 0.99999) { // se viaggia quasi parallelo all'asse Z
        double segno = 1.0;
        if(uz > 0.0){
          segno = 1.0;
        }else{
          segno = -1.0;
        }
        ux_new = sin_theta * cos_phi;
        uy_new = sin_theta * sen_phi * segno;
        uz_new = cos_theta * segno;
    } else {
        double proiezione_xy = std::sqrt(1.0 - uz * uz);
        ux_new = sin_theta * (ux * uz * cos_phi - uy * sen_phi) / proiezione_xy + ux * cos_theta;
        uy_new = sin_theta * (uy * uz * cos_phi + ux * sen_phi) / proiezione_xy + uy * cos_theta;
        uz_new = -sin_theta * cos_phi * proiezione_xy + uz * cos_theta;
    }

    // Normalizzazione
    double norm = std::sqrt(ux_new * ux_new + uy_new * uy_new + uz_new * uz_new);
    if (norm > 0.0){
       ux = ux_new/norm;
       uy = uy_new/norm;
       uz = uz_new/norm;
    }
}

Writing ./CPU_Parallelo/compton.h


### File phantom.h

Si occupa della costruzione del phantom voxelizzato. In particolare si hanno:
* Phantom Omogeneo
  * Si considera composto da acqua pura di 30x30x30 cm^3, quindi 100^3 voxel con 3mm a lato
* Phantom acqua + inserto osseo
  * si considera composto oltre che da acqua da un inserto osseo di 5x5x5 cm^3 al centro

Inizializza un volume cubico stadard dove ogni singolo voxel è marcato con acqua e a quello con inserto osseo viene inserito al centro della griglia un cubo di materiale osseo. *Viene, inoltre, considerato che metà lato dell'inserto è pari a 2.5cm / 0.3cm/voxel che è circa uguale e 8 voxel*

In [ ]:
%%writefile ./CPU_Parallelo/phantom.h

#pragma once

#include <cstring>
#include <cstdio>
#include "physics.h"

// Phantom Omogeneo (solo acqua)
inline void build_phantom_water(int *phantom) {
    int total = NX * NY * NZ;
    for (int i = 0; i < total; i++)
        phantom[i] = MAT_WATER;
}

// Phantom acqua + inserto osseo
inline void build_phantom_hetero(int *phantom) {
    build_phantom_water(phantom);

    // Centro del phantom in indici voxel
    int cx = NX / 2;
    int cy = NY / 2;
    int cz = NZ / 2;

    int meta_lato_inserto = (int)(2.5 / VOXEL_CM + 0.5);

    int count = 0;
    for (int iz = cz - meta_lato_inserto; iz < cz + meta_lato_inserto; iz++)
    for (int iy = cy - meta_lato_inserto; iy < cy + meta_lato_inserto; iy++)
    for (int ix = cx - meta_lato_inserto; ix < cx + meta_lato_inserto; ix++) {
        if (ix >= 0 && ix < NX && iy >= 0 && iy < NY && iz >= 0 && iz < NZ) { // controlli per non scrivere fuori dalla memoria del phantom
            phantom[phantom_idx(ix, iy, iz)] = MAT_BONE;
            count++;
        }
    }

    double volume_fisico_totale = count * VOXEL_CM * VOXEL_CM * VOXEL_CM;
    printf("Inserto osseo: %d voxel = %.1f cm³  ( volume teorico 125 cm³)\n", count, volume_fisico_totale);
}

// Inizializzazione dose grid a zero
inline void init_dose(double *dose) {
    memset(dose, 0, NX * NY * NZ * sizeof(double));
}


Writing ./CPU_Parallelo/phantom.h


### File physics.h

Contiene le costanti fondamentali al calcolo e le tabelle dei coefficienti di interazione derivate dal database ufficiale NIST XCOM (https://physics.nist.gov/PhysRefData/XrayMassCoef/tab4.html), oltre alle funzioni matematiche per l'estrazione delle probabilità fisiche. Vengono definite le soglie ECUT e PCUT, la risoluzione spaziale della griglia voxelizzata e la cnversione fisica tra idnici dei voxel.

La griglia energetica viene considerata come l'insieme dei punti discreti su cui sono stati misurati e tabulati i dati fisici e secondo il db NIST XCOM si considera da 28 punti, quindi da 0.01 MeV a 20 MeV. In questo modo se il fotone ha un'energia che cade su uno di questi punti viene letto direttamente il valore dalla tabella altrimenti si usa la griglia per sapere tra quali punti effettuare l'interpolazione.

In [ ]:
%%writefile ./CPU_Parallelo/physics.h

#pragma once

#include <cmath>
#include <cassert>

// COSTANTI FISIHCHE
static const double ME_C2    = 0.51099895;  // MeV  massa a riposo elettrone
static const double PI       = 3.14159265358979323846;
static const double ECUT     = 0.010;       // MeV  cutoff fotoni  (10 keV)
static const double PCUT     = 0.100;       // MeV  cutoff elettroni

// GEOMETRIA PHANTOM
static const int    NX = 100, NY = 100, NZ = 100;   // voxel per asse
static const double VOXEL_CM = 0.30;                // lato voxel [cm] = 3 mm
static const double PHANTOM_CM = NX * VOXEL_CM;     // 30.0 cm per asse

// INDICI MATERIALI
#define MAT_WATER 0   // acqua  ρ = 1.000 g/cm^3
#define MAT_BONE  1   // osso (ICRU)  ρ = 1.850 g/cm^3
#define MAT_LUNG  2   // polmone (ICRU)  ρ = 0.260 g/cm^3
#define MAT_AIR   3   // aria  ρ = 0.001205 g/cm^3
#define N_MAT     4   // numero materiali disponibili

// DENSITÀ [g/cm^3]
static const double DENSITY[N_MAT] = { 1.000, 1.850, 0.260, 0.001205 };

// GRIGLIA ENERGETICA [MeV]  (28 punti, da 0.01 a 20 MeV)
static const int N_ENERGY = 28;
static const double ENERGY_GRID[N_ENERGY] = {
    0.010, 0.015, 0.020, 0.030, 0.040, 0.050, 0.060, 0.080, 0.100,
    0.150, 0.200, 0.300, 0.400, 0.500, 0.600, 0.800, 1.000, 1.250,
    1.500, 2.000, 3.000, 4.000, 5.000, 6.000, 8.000, 10.000,
    15.000, 20.000
};

// COEFFICIENTI μ/ρ [cm^2/g]  per ogni materiale e processo -> [materiale][bin_energia]
static const double MU_TOTAL[N_MAT][N_ENERGY] = {
    // ACQUA
    { 5.329, 1.673, 0.8096, 0.3756, 0.2683, 0.2269, 0.2059, 0.1837, 0.1707,
      0.1505, 0.1370, 0.1186, 0.1061, 0.09687, 0.09007, 0.07865, 0.07072, 0.06323,
      0.05754, 0.04942, 0.03969, 0.03403, 0.03031, 0.02770, 0.02429, 0.02219,
      0.01941, 0.01813 },
    // OSSO
    { 19.89, 7.131, 3.085, 1.012, 0.5475, 0.3941, 0.3178, 0.2595, 0.2368,
      0.1958, 0.1698, 0.1393, 0.1222, 0.1107, 0.1018, 0.08795, 0.07838, 0.06945,
      0.06283, 0.05351, 0.04257, 0.03624, 0.03209, 0.02913, 0.02536, 0.02296,
      0.01978, 0.01832 },
    // POLMONE
    { 5.208, 1.638, 0.7933, 0.3681, 0.2629, 0.2224, 0.2018, 0.1800, 0.1673,
      0.1475, 0.1342, 0.1162, 0.1040, 0.09493, 0.08827, 0.07708, 0.06930, 0.06196,
      0.05639, 0.04843, 0.03889, 0.03335, 0.02970, 0.02715, 0.02380, 0.02175,
      0.01902, 0.01776 },
    // ARIA
    { 36.01, 12.28, 5.279, 1.625, 0.6918, 0.3954, 0.2885, 0.2085, 0.1875,
      0.1504, 0.1340, 0.1123, 0.09921, 0.09076, 0.08414, 0.07285, 0.06529, 0.05817,
      0.05298, 0.04534, 0.03597, 0.03054, 0.02699, 0.02453, 0.02131, 0.01931,
      0.01673, 0.01551 }
};

// EFFETTO FOTOELETTRICO
static const double MU_PE[N_MAT][N_ENERGY] = {
    // ACQUA
    { 4.944, 1.374, 0.5195, 0.1036, 0.02407, 0.005800, 0.001334, 5.510e-5, 3.998e-5,
      2.799e-6, 2.200e-7, 1.400e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    // OSSO
    { 19.35, 6.833, 2.818, 0.7469, 0.2837, 0.1152, 0.04660, 0.008680, 0.001900,
      1.800e-4, 2.000e-5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    // POLMONE
    { 4.845, 1.346, 0.5091, 0.1015, 0.02359, 0.005684, 0.001307, 5.400e-5, 3.918e-5,
      2.743e-6, 2.156e-7, 1.372e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    // ARIA
    { 35.52, 11.99, 5.012, 1.379, 0.4529, 0.1581, 0.05757, 0.008251, 0.001581,
      8.208e-5, 7.636e-6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 }
};

// SCATTERING COMPTON
static const double MU_COMPTON[N_MAT][N_ENERGY] = {
    // ACQUA
    { 0.3854, 0.2988, 0.2672, 0.2651, 0.2595, 0.2476, 0.2329, 0.1984, 0.1661,
      0.1505, 0.1370, 0.1186, 0.1061, 0.09687, 0.09007, 0.07865, 0.07072, 0.06323,
      0.05754, 0.04942, 0.03969, 0.03403, 0.03031, 0.02770, 0.02429, 0.02219,
      0.01878, 0.01719 },
    // OSSO
    { 0.4869, 0.2684, 0.2503, 0.2465, 0.2429, 0.2310, 0.2172, 0.1848, 0.1548,
      0.1400, 0.1275, 0.1103, 0.09870, 0.09010, 0.08377, 0.07313, 0.06575, 0.05862,
      0.05338, 0.04579, 0.03667, 0.03133, 0.02784, 0.02539, 0.02217, 0.02016,
      0.01702, 0.01552 },
    // POLMONE
    { 0.3777, 0.2928, 0.2619, 0.2598, 0.2543, 0.2426, 0.2282, 0.1944, 0.1628,
      0.1475, 0.1342, 0.1162, 0.1040, 0.09493, 0.08827, 0.07708, 0.06930, 0.06196,
      0.05639, 0.04843, 0.03889, 0.03335, 0.02970, 0.02715, 0.02380, 0.02175,
      0.01840, 0.01684 },
    // ARIA
    { 0.3779, 0.2933, 0.2624, 0.2602, 0.2547, 0.2430, 0.2285, 0.1946, 0.1630,
      0.1477, 0.1344, 0.1163, 0.1041, 0.09516, 0.08844, 0.07723, 0.06942, 0.06207,
      0.05649, 0.04852, 0.03894, 0.03339, 0.02973, 0.02718, 0.02382, 0.02177,
      0.01843, 0.01686 }
};

// PRODUZIONE DI COPPIE
static const double MU_PAIR[N_MAT][N_ENERGY] = {
    // ACQUA
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.000630, 0.000940 },
    // OSSO
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.002760, 0.002800 },
    // POLMONE
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.000617, 0.000921 },
    // ARIA
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.000589, 0.000879 }
};

// INTERPOLAZIONE LINEARE SU GRIGLIA ENERGETICA
inline double interp_mu(double energia_mev, const double tabella[N_ENERGY]) {
    if (energia_mev <= ENERGY_GRID[0])
        return tabella[0];
    if (energia_mev >= ENERGY_GRID[N_ENERGY-1])
        return tabella[N_ENERGY-1];

    int indice_limite_inferiore = 0;
    int indice_imite_superiore = N_ENERGY - 1;

    while (indice_imite_superiore - indice_limite_inferiore > 1) {
        int punto_centrale = (indice_limite_inferiore + indice_imite_superiore) / 2;
        if (ENERGY_GRID[punto_centrale] <= energia_mev){
            indice_limite_inferiore = punto_centrale;
        }else{
            indice_imite_superiore = punto_centrale;
        }
    }

    double fattore_interpolazione = (energia_mev - ENERGY_GRID[indice_limite_inferiore]) / (ENERGY_GRID[indice_imite_superiore] - ENERGY_GRID[indice_limite_inferiore]);
    return tabella[indice_limite_inferiore] * (1.0 - fattore_interpolazione) + tabella[indice_imite_superiore] * fattore_interpolazione;
}

// CALCOLO MU TOTALE
inline double get_mu_total(double energia, int materiale) {   // probabilita di avere un urto
    return interp_mu(energia, MU_TOTAL[materiale]) * DENSITY[materiale];
}
inline double get_mu_pe(double energia, int materiale) {      // probabilita effetto fotoelettrico
    return interp_mu(energia, MU_PE[materiale]) * DENSITY[materiale];
}
inline double get_mu_compton(double energia, int materiale) {   // probabilità di compton
    return interp_mu(energia, MU_COMPTON[materiale]) * DENSITY[materiale];
}
inline double get_mu_pair(double energia, int materiale) {    // probabilita produzione coppie
    return interp_mu(energia, MU_PAIR[materiale]) * DENSITY[materiale];
}

// SELEZIONE TIPO DI INTERAZIONE
// Restituisce: 0=fotoelettrico, 1=Compton, 2=produzione coppie
// xi: numero casuale uniforme in [0,1)
inline int select_interaction(double energia, int materiale, double xi) {
    double mu_totale = get_mu_total(energia, materiale);

    if (mu_totale <= 0.0)
        return 1;

    double probabilita_fotoelettrico = get_mu_pe(energia, materiale) / mu_totale;
    double probabilita_compton = get_mu_compton(energia, materiale) / mu_totale;

    if (xi < probabilita_fotoelettrico)
        return 0;   // fotoelettrico
    if (xi < probabilita_fotoelettrico + probabilita_compton)
        return 1; // compton
    return 2;  // produzione di coppie
}

// INDICE LINEARE PHANTOM CON ROW MAJOR ORDER
inline int phantom_idx(int ix, int iy, int iz) {
    return ix + NX * (iy + NY * iz);
}

// CONTROLLO COORDINATE CONTORNO CUBO
inline bool inside(double x, double y, double z) {
    return x >= 0.0 && x < PHANTOM_CM &&
           y >= 0.0 && y < PHANTOM_CM &&
           z >= 0.0 && z < PHANTOM_CM;
}

// CONTROLLO PSIZIONE IN VOXEL
inline int vox(double coord) {
    int indice_voxel = (int)(coord / VOXEL_CM);
    if (indice_voxel < 0)
        indice_voxel = 0;
    if (indice_voxel >= NX)
        indice_voxel = NX - 1;
    return indice_voxel;
}


Writing ./CPU_Parallelo/physics.h


### File random.h

Si occupa di generare numeri casuali ad alte prestazioni e campionare l'energia dei fotoni incidenti seguendo lo spettro reale di un acceleratore lineare LINAC. In particolare:
* è stato implementato l'algoritmo xoshiro256
  * viene utilizzato l'algoritmo splotmix64 per garantire che partendo da un singolo seed tutti i bit sono inizializzati in modo correlato
* è stato modellato un fascio reale di radioterapia (basato sui dati della lettetatura) e questo spettro è stato suddiviso in 24 intervalli energetici (bin) da 0.25 MeV fino a 6 MeV.
  * Si fa riferimento allo spettro 6MV del fascio standard Varian 2100C a 6MV, campo 10x10cm^2 a SAD 100cm

In [ ]:
%%writefile ./CPU_Parallelo/random.h

#pragma once

#include <cstdint>
#include <cmath>
#include <cstring>

// XOSHIRO256
struct Xoshiro256 {
    uint64_t s[4];

    // Inizializzazione con un seed a 64 bit usando splitmix64
    explicit Xoshiro256(uint64_t seed) { // con explicit si evitano conversioni automatiche
        auto splitmix = [](uint64_t &x) -> uint64_t {
            x += 0x9e3779b97f4a7c15ULL; // rappresentazione a 64 bit della sezione aurea (2^64/phi)
            uint64_t z = x;
            z = (z ^ (z >> 30)) * 0xbf58476d1ce4e5b9ULL;
            z = (z ^ (z >> 27)) * 0x94d049bb133111ebULL;
            return z ^ (z >> 31);
        };
        s[0] = splitmix(seed);
        s[1] = splitmix(seed);
        s[2] = splitmix(seed);
        s[3] = splitmix(seed);
    }

    // Genera un uint64_t casuale
    uint64_t next() {
        const uint64_t result = rotate_left(s[1] * 5, 7) * 9;
        const uint64_t t = s[1] << 17;
        s[2] ^= s[0]; s[3] ^= s[1]; // ^ indica opertaroe XOR bit a bit
        s[1] ^= s[2]; s[0] ^= s[3];
        s[2] ^= t;
        s[3] = rotate_left(s[3], 45);
        return result;
    }

    // Genera double uniforme in (0, 1) escludendo 0 per evitare log(0)
    double operator()() {
        double r;
        do {
            // usa i 53 bit superiori
            r = (double)(next() >> 11) * (1.0 / (double)(1ULL << 53)); // ULL intero a 64 bit senza segno
        } while (r <= 0.0);
        return r;
    }

private:
    static uint64_t rotate_left(const uint64_t x, int k) {
        return (x << k) | (x >> (64 - k));
    }
};

// SPETTRO 6MV
static const int SPECTRUM_BINS = 24;
static const double SPECTRUM_E[SPECTRUM_BINS] = {
    0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 1.75, 2.00,
    2.25, 2.50, 2.75, 3.00, 3.25, 3.50, 3.75, 4.00,
    4.25, 4.50, 4.75, 5.00, 5.25, 5.50, 5.75, 6.00
};

// Fluenza relativa normalizzata  (somma = 1.0)
static const double SPECTRUM_FLUENCE[SPECTRUM_BINS] = {
    0.0243, 0.0676, 0.0862, 0.0929, 0.0919, 0.0868, 0.0794, 0.0712,
    0.0628, 0.0548, 0.0471, 0.0399, 0.0334, 0.0276, 0.0224, 0.0178,
    0.0138, 0.0104, 0.0075, 0.0052, 0.0034, 0.0020, 0.0010, 0.0004
};

// CDF precalcolata all'inizializzazione
struct Spectrum {
    double cdf[SPECTRUM_BINS];
    double energie[SPECTRUM_BINS];
    double larghezza_intervallo_energetico_bin;

    Spectrum() {
        double somma_fluence = 0.0;
        for (int i = 0; i < SPECTRUM_BINS; i++) {
          somma_fluence += SPECTRUM_FLUENCE[i];
        }

        cdf[0] = SPECTRUM_FLUENCE[0] / somma_fluence;
        for (int i = 1; i < SPECTRUM_BINS; i++)
            cdf[i] = cdf[i-1] + SPECTRUM_FLUENCE[i] / somma_fluence;
        cdf[SPECTRUM_BINS-1] = 1.0;

        for (int i = 0; i < SPECTRUM_BINS; i++)
            energie[i] = SPECTRUM_E[i];

        larghezza_intervallo_energetico_bin = 0.25;
    }

    // Campiona energia con binary search sulla CDF
    double sample(Xoshiro256 &rng) const {
        double xi = rng();
        // Ricerca binaria sulla CDF
        int indice_limite_inferiore = 0;
        int indice_limite_superiore = SPECTRUM_BINS - 1;
        while (indice_limite_inferiore < indice_limite_superiore) {
            int punto_centrale = (indice_limite_inferiore + indice_limite_superiore) / 2;
            if (cdf[punto_centrale] < xi){
                indice_limite_inferiore = punto_centrale + 1;
            }
            else{
                indice_limite_superiore = punto_centrale;
            }
        }

        double energia_centrale = energie[indice_limite_inferiore];
        double offset = (rng() - 0.5) * larghezza_intervallo_energetico_bin;
        double energia = energia_centrale + offset;

        if (energia < 0.01){
           energia = 0.01;
        }
        if (energia > 6.00){
          energia = 6.00;
        }
        return energia;
    }
};


Writing ./CPU_Parallelo/random.h


### File output.h

Contiene le routine necessarie per elaborare la matrice tridimensionale della dose e generare grafici e statistiche. In particolare:
* PDD o Percent Depht Dose è la misura di come la dose varia man mano che il fascio penetra in profondità nel paziente. Considera:
  * Media spaziale che calcola la media su una finestra di +- 8 voxel per simulare la risposta di una camera a ionizzazione reale;
  * normalizzazione che imposta la dose massima lungo l'asse mentre le altre vengono scalate
  * costruzione di profondità che calcola la coordinata fisica per ogni punto di misura.

* Calcola la funzione che analizza come la dose si distribuisce trasversalmente rispetto alla direzione del fascio
* Esportazione dei dati in csv
* Monitoraggio efficienza
  * numero di particelle al secondo;
  * energia depositata
  * voxel occupati

In [ ]:
%%writefile ./CPU_Parallelo/output.h

#pragma once

#include <cstdio>
#include <cmath>
#include <fstream>
#include <algorithm>
#include "physics.h"

// PDD
inline void compute_pdd(const double *dose, double *pdd, double *profondita, int semiampiezza_della_media = 8) {
    int cx = NX / 2; // centro del fascio in X
    int cy = NY / 2; // centro del fascio in Y

    double max_dose = 0.0;
    for (int iz = 0; iz < NZ; iz++) {
        double valore_dose = 0.0;
        int    num_voxel_sommati = 0;
        for (int ix = cx - semiampiezza_della_media; ix <= cx + semiampiezza_della_media; ix++){
          for (int iy = cy - semiampiezza_della_media; iy <= cy + semiampiezza_della_media; iy++) {
              if (ix >= 0 && ix < NX && iy >= 0 && iy < NY) {
                  valore_dose += dose[phantom_idx(ix, iy, iz)];
                  num_voxel_sommati++;
              }
          }
        }
        if (num_voxel_sommati > 0) {
            pdd[iz] = valore_dose / num_voxel_sommati;
        } else{
            0.0;
        }
        profondita[iz] = (iz + 0.5) * VOXEL_CM;
        if (pdd[iz] > max_dose){
            max_dose = pdd[iz];
        }
    }

    if (max_dose > 0.0)
        for (int iz = 0; iz < NZ; iz++)
            pdd[iz] = pdd[iz] / max_dose * 100.0;
}

// PROFILO LATERALE a profondità fissa (lungo X, centrato su Y)
inline void compute_lateral_profile(const double *dose, double *profilo_dose_relativa, double *coordinate_cm, double profondita_scelta = 10.0, int semiampiezza_media = 2) {
    int iz = std::min((int)(profondita_scelta / VOXEL_CM), NZ - 1);
    int centro_asse_y = NY / 2;

    double dose_massima_profilo = 0.0;
    for (int ix = 0; ix < NX; ix++) {
        double somma_dose_accumulata = 0.0;
        int conteggio_voxel_validi = 0;
        for (int iy = centro_asse_y - semiampiezza_media; iy <= centro_asse_y + semiampiezza_media; iy++) {
            if (iy >= 0 && iy < NY) {
                somma_dose_accumulata += dose[phantom_idx(ix, iy, iz)];
                conteggio_voxel_validi++;
            }
        }
        if (conteggio_voxel_validi > 0){
            profilo_dose_relativa[ix] = somma_dose_accumulata / conteggio_voxel_validi;
        } else{
            profilo_dose_relativa[ix] = 0.0;
        }
        coordinate_cm[ix] = (ix + 0.5) * VOXEL_CM - PHANTOM_CM / 2.0;
        if (profilo_dose_relativa[ix] > dose_massima_profilo) {
            dose_massima_profilo = profilo_dose_relativa[ix];
        }
    }

    if (dose_massima_profilo > 0.0)
        for (int ix = 0; ix < NX; ix++)
            profilo_dose_relativa[ix] = profilo_dose_relativa[ix] / dose_massima_profilo * 100.0;
}

// SALVA PDD SU CSV
inline void save_pdd_csv(const double *vettore_profondita_cm, const double *pdd, const char *filename) {
    std::ofstream f(filename);
    f << "depth_cm,dose_percent\n";
    for (int iz = 0; iz < NZ; iz++)
        f << vettore_profondita_cm[iz] << "," << pdd[iz] << "\n";
    f.close();
    printf("Salvato: %s\n", filename);
}

// SALVA PROFILO LATERALE SU CSV
inline void save_profile_csv(const double *coordinate_cm, const double *profilo_dose_relativa, const char *filename) {
    std::ofstream f(filename);
    f << "position_cm,dose_percent\n";
    for (int ix = 0; ix < NX; ix++)
        f << coordinate_cm[ix] << "," << profilo_dose_relativa[ix] << "\n";
    f.close();
    printf("  Salvato: %s\n", filename);
}

// SALVA SLICE 2D CENTRALE SU CSV (per heatmap)
inline void save_dose_slice_csv(const double *dose, const char *filename) {
    std::ofstream f(filename);
    int iy = NY / 2;
    for (int iz = 0; iz < NZ; iz++) {
        for (int ix = 0; ix < NX; ix++) {
            f << dose[phantom_idx(ix, iy, iz)];
            if (ix < NX - 1) f << ",";
        }
        f << "\n";
    }
    f.close();
    printf("  Salvato: %s\n", filename);
}

// SALVA DOSE 3D COMPLETA
inline void save_dose_binary(const double *dose, const char *filename) {
    FILE *f = fopen(filename, "wb");
    if (!f) { printf("ERRORE: impossibile aprire %s\n", filename); return; }
    fwrite(dose, sizeof(double), NX * NY * NZ, f);
    fclose(f);
    printf("Salvato: %s  (%d float64)\n", filename, NX * NY * NZ);
}

// STAMPA TABELLA PDD AI PUNTI DI RIFERIMENTO CLINICI
inline void print_pdd_table(const double *profondita, const double *pdd, const char *label) {
    printf("\n  PDD — %s\n", label);
    printf("  %-20s  %10s  %s\n", "Profondità [cm]", "Dose [%]", "Riferimento");
    printf("  %s\n", "─────────────────────────────────────────");

    double refs[]      = {1.5, 3.0, 5.0, 10.0, 15.0, 20.0, 25.0};
    const char *notes[] = {"build-up", "d_max 6MV", "", "D10", "", "D20", ""};

    for (int r = 0; r < 7; r++) {
        int k = (int)(refs[r] / VOXEL_CM);
        if (k >= 0 && k < NZ)
            printf("  %-20.1f  %10.2f  %s\n", profondita[k], pdd[k], notes[r]);
    }
}

// STATISTICHE GENERALI SULLA DOSE
inline void print_dose_stats(const double *dose, long long numero_particelle_totali, double tempo_esecuzione_secondi) {
    double dose_massima_assoluta = 0.0;
    double energia_totale_depositata = 0.0;
    int conteggio_voxel_colpiti = 0;

    for (int i = 0; i < NX * NY * NZ; i++) {
        if (dose[i] > 0.0) {
            conteggio_voxel_colpiti++;
            energia_totale_depositata += dose[i];
            if (dose[i] > dose_massima_assoluta){
                dose_massima_assoluta = dose[i];
            }
        }
    }

    printf("\n Statistiche simulazione: \n");
    printf("  Particelle simulate : %lld\n",  numero_particelle_totali);
    printf("  Tempo totale        : %.2f s\n", tempo_esecuzione_secondi);
    printf("  Throughput          : %.3f MP/s\n", numero_particelle_totali / tempo_esecuzione_secondi / 1.0e6);
    printf("  Tempo/particella    : %.1f μs\n", tempo_esecuzione_secondi / numero_particelle_totali * 1.0e6);
    printf("  Voxel con dose>0    : %d / %d (%.1f%%)\n", conteggio_voxel_colpiti, NX*NY*NZ, 100.0*conteggio_voxel_colpiti/(NX*NY*NZ));
    printf("  Energia totale dep. : %.4e MeV\n", energia_totale_depositata);
    printf("  Energia/particella  : %.4e MeV\n", numero_particelle_totali > 0 ? energia_totale_depositata / numero_particelle_totali : 0.0);
    printf("  Dose massima (u.a.) : %.6e\n", dose_massima_assoluta);
}


Writing ./CPU_Parallelo/output.h


### File main.cpp

Programma principale che unisce tutti i moduli ed esegue la simulazione Monte Carlo per radioterapia. Effettua:
* Campionamento della sorgente;
* Applica algoritmo di Traversal;
* Gestisce la storia di ogni fotone

Implementa il trasporto di fotoni in un phantom voxelizzato con:
 * Spettro 6MV (Sheikh-Bagheri & Rogers 2002)
 * Legge di Beer-Lambert + voxel traversal (Amanatides & Woo 1987)
 * Effetto fotoelettrico, Compton (metodo di Kahn), produzione coppie
 * Sezioni d'urto da NIST XCOM (Hubbell & Seltzer 1996)
 * Approssimazione KERMA ≈ dose (deposito locale elettroni)
 * PRNG: xoshiro256** (Blackman & Vigna 2018)

In [ ]:
%%writefile ./CPU_Parallelo/main.cpp

#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>
#include <cstring>
#include <vector>
#include <thread>
#include <atomic>
#include <mutex>

#include "physics.h"
#include "compton.h"
#include "random.h"
#include "phantom.h"
#include "output.h"

// ─────────────────────────────────────────────────────────────────────────────
// STRUTTURA PARTICELLA
// ─────────────────────────────────────────────────────────────────────────────
struct Particle {
    double x, y, z;
    double ux, uy, uz;
    double energia;
};

// ─────────────────────────────────────────────────────────────────────────────
// CAMPIONAMENTO SORGENTE
// ─────────────────────────────────────────────────────────────────────────────
inline Particle sample_source(const Spectrum &spettro, Xoshiro256 &rng) {
    static const double FIELD_HALF = 5.0;
    double cx = PHANTOM_CM / 2.0;
    double cy = PHANTOM_CM / 2.0;
    Particle p;
    p.x = cx + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.y = cy + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.z = 1.0e-7;
    p.ux = 0.0; p.uy = 0.0; p.uz = 1.0;
    p.energia = spettro.sample(rng);
    return p;
}

// ─────────────────────────────────────────────────────────────────────────────
// DISTANZA AL PROSSIMO CONFINE DI VOXEL
// ─────────────────────────────────────────────────────────────────────────────
inline double boundary_distance(double x, double y, double z,
                                 double ux, double uy, double uz,
                                 int ix, int iy, int iz) {
    double d = 1.0e30;
    if (std::fabs(ux) > 1.0e-12) {
        double cx = (ux > 0) ? (ix + 1) * VOXEL_CM : ix * VOXEL_CM;
        double t  = (cx - x) / ux;
        if (t > 1.0e-10) d = std::min(d, t);
    }
    if (std::fabs(uy) > 1.0e-12) {
        double cy = (uy > 0) ? (iy + 1) * VOXEL_CM : iy * VOXEL_CM;
        double t  = (cy - y) / uy;
        if (t > 1.0e-10) d = std::min(d, t);
    }
    if (std::fabs(uz) > 1.0e-12) {
        double cz = (uz > 0) ? (iz + 1) * VOXEL_CM : iz * VOXEL_CM;
        double t  = (cz - z) / uz;
        if (t > 1.0e-10) d = std::min(d, t);
    }
    return d;
}

// ─────────────────────────────────────────────────────────────────────────────
// TRASPORTO FOTONE — scrive sulla dose_locale del thread chiamante
// ─────────────────────────────────────────────────────────────────────────────
void transport_photon(Particle fotone_iniziale, const int *phantom,
                      double *dose_locale, Xoshiro256 &rng) {
    Particle stack[64];
    int top = 0;
    stack[top++] = fotone_iniziale;

    while (top > 0) {
        Particle p = stack[--top];
        for (int step = 0; step < 100000; step++) {

            if (p.energia < ECUT) {
                if (inside(p.x, p.y, p.z)) {
                    int ix = vox(p.x), iy = vox(p.y), iz = vox(p.z);
                    dose_locale[phantom_idx(ix, iy, iz)] += p.energia;
                }
                break;
            }
            if (!inside(p.x, p.y, p.z)) break;

            int ix = vox(p.x), iy = vox(p.y), iz = vox(p.z);
            int mat = phantom[phantom_idx(ix, iy, iz)];
            double mu = get_mu_total(p.energia, mat);
            if (mu <= 0.0) break;

            double distanza_teorica = -std::log(rng()) / mu;
            double distanza_fisica  = boundary_distance(p.x, p.y, p.z,
                                                        p.ux, p.uy, p.uz,
                                                        ix, iy, iz);
            if (distanza_teorica <= distanza_fisica) {
                p.x += p.ux * distanza_teorica;
                p.y += p.uy * distanza_teorica;
                p.z += p.uz * distanza_teorica;
                if (!inside(p.x, p.y, p.z)) break;

                ix = vox(p.x); iy = vox(p.y); iz = vox(p.z);
                mat = phantom[phantom_idx(ix, iy, iz)];
                int id = phantom_idx(ix, iy, iz);
                int tipo = select_interaction(p.energia, mat, rng());

                if (tipo == 0) {                         // fotoelettrico
                    dose_locale[id] += p.energia;
                    break;
                } else if (tipo == 1) {                  // Compton
                    double cos_theta, e_scatter;
                    sample_compton(p.energia, rng, cos_theta, e_scatter);
                    double ceduta = p.energia - e_scatter;
                    if (ceduta > 0.0) dose_locale[id] += ceduta;
                    p.energia = e_scatter;
                    rotate_direction(p.ux, p.uy, p.uz, cos_theta, 2.0 * PI * rng());
                    if (p.energia < ECUT) { dose_locale[id] += p.energia; break; }
                } else {                                  // produzione coppie
                    double e_cin = p.energia - 2.0 * ME_C2;
                    if (e_cin > 0.0) dose_locale[id] += e_cin;
                    if (ME_C2 > ECUT && top + 2 <= 62) {
                        double ct = 2.0 * rng() - 1.0;
                        double phi = 2.0 * PI * rng();
                        double st  = std::sqrt(std::max(0.0, 1.0 - ct * ct));
                        Particle f1, f2;
                        f1.x = f2.x = p.x; f1.y = f2.y = p.y; f1.z = f2.z = p.z;
                        f1.ux =  st * std::cos(phi); f1.uy =  st * std::sin(phi); f1.uz =  ct;
                        f2.ux = -f1.ux;              f2.uy = -f1.uy;              f2.uz = -ct;
                        f1.energia = f2.energia = ME_C2;
                        stack[top++] = f1;
                        stack[top++] = f2;
                    }
                    break;
                }
            } else {
                p.x += p.ux * (distanza_fisica + 1.0e-7);
                p.y += p.uy * (distanza_fisica + 1.0e-7);
                p.z += p.uz * (distanza_fisica + 1.0e-7);
            }
        }
    }
}

// ─────────────────────────────────────────────────────────────────────────────
// FUNZIONE WORKER — eseguita da ogni std::thread
//
// Ogni thread riceve:
//   tid          : indice thread [0, num_thread)
//   num_thread   : numero totale di thread
//   num_fotoni   : fotoni totali da simulare
//   seed         : seed globale (il thread deriva il proprio via golden ratio)
//   phantom      : phantom condiviso in sola lettura
//   dose_globale : array globale; il thread vi scrive SOLO nella riduzione finale
//   contatore    : contatore atomico condiviso per il progress report
//   tempo_inizio : per calcolare il rate
//
// Strategia: ogni thread simula i fotoni con indici i = tid, tid+num_thread,
// tid+2*num_thread, ... (partizionamento ciclico per bilanciare il carico).
// La dose viene accumulata in dose_locale (privata), poi sommata alla
// dose_globale in una sezione protetta da mutex alla fine.
// ─────────────────────────────────────────────────────────────────────────────
void worker(int tid, int num_thread, long long num_fotoni,
            uint64_t seed, const int *phantom, double *dose_globale,
            std::atomic<long long> &contatore,
            std::chrono::time_point<std::chrono::high_resolution_clock> tempo_inizio) {

    // RNG indipendente per thread: seed derivato con costante golden ratio
    // Garantisce stream statisticamente scorrelati senza jump-ahead
    uint64_t seed_thread = seed + (uint64_t)tid * 2654435761ULL;
    Xoshiro256 rng(seed_thread);

    Spectrum spettro;

    // Dose locale: privata al thread, nessuna race condition
    std::vector<double> dose_locale(NX * NY * NZ, 0.0);

    // Partizionamento ciclico: thread 0 → fotoni 0, N, 2N, ...
    //                          thread 1 → fotoni 1, N+1, 2N+1, ...
    for (long long i = tid; i < num_fotoni; i += num_thread) {
        Particle p = sample_source(spettro, rng);
        transport_photon(p, phantom, dose_locale.data(), rng);

        // Progress report: solo thread 0, ogni 5% dei fotoni di sua competenza
        if (tid == 0) {
            long long fatti = contatore.fetch_add(num_thread,
                                std::memory_order_relaxed) + num_thread;
            long long step  = std::max(1LL, num_fotoni / 20);
            if (fatti % step < num_thread) {
                auto ora = std::chrono::high_resolution_clock::now();
                double dt   = std::chrono::duration<double>(ora - tempo_inizio).count();
                double rate  = fatti / dt;
                double eta   = (num_fotoni - fatti) / rate;
                printf(" [%5.1f%%]  %.0f fotoni/s  ETA %.0fs\n",
                       100.0 * fatti / num_fotoni, rate, eta);
            }
        }
    }

    // Riduzione: somma dose_locale nella dose_globale
    // Non servono mutex perché ogni thread scrive su un intervallo diverso?
    // No: i voxel irradiati si sovrappongono tra thread → serve protezione.
    // Usiamo una riduzione sequenziale thread per thread (non un mutex per voxel):
    // questo blocca un solo thread alla volta per NX*NY*NZ addizioni, ma avviene
    // UNA sola volta per thread → costo O(thread * voxel), trascurabile rispetto
    // al trasporto O(fotoni * interazioni).
    //
    // Alternativa più scalabile per N_thread grande (> 32):
    //   dividere il vettore dose in N_thread blocchi e fare la riduzione in parallelo.
    //   Non necessario qui: Colab ha al massimo 2-4 core fisici.
    static std::mutex mutex_riduzione;
    {
        std::lock_guard<std::mutex> lock(mutex_riduzione);
        for (int k = 0; k < NX * NY * NZ; k++)
            dose_globale[k] += dose_locale[k];
    }
}

// ─────────────────────────────────────────────────────────────────────────────
// MAIN
// ─────────────────────────────────────────────────────────────────────────────
int main(int argc, char *argv[]) {

    long long num_fotoni  = 1000000;
    int tipo_phantom      = 0;
    uint64_t seed         = 42ULL;
    int num_thread        = (int)std::thread::hardware_concurrency();
    if (num_thread < 1) num_thread = 1;

    if (argc > 1) num_fotoni   = std::atoll(argv[1]);
    if (argc > 2) tipo_phantom = std::atoi(argv[2]);
    if (argc > 3) seed         = (uint64_t)std::atoll(argv[3]);
    if (argc > 4) num_thread   = std::atoi(argv[4]);   // opzionale: forza N thread

    const char *phantom_label = (tipo_phantom == 0) ? "Acqua omogenea" : "Acqua + Osso";

    printf("  Monte Carlo per Radioterapia — CPU Parallelo (std::thread)\n");
    printf("\n  Parametri:\n");
    printf("  Phantom    : %dx%dx%d voxel  |  voxel %.0fmm  |  %.0f^3 cm^3\n",
           NX, NY, NZ, VOXEL_CM * 10.0, PHANTOM_CM);
    printf("  Materiale  : %s\n", phantom_label);
    printf("  N fotoni   : %lld\n", num_fotoni);
    printf("  Seed       : %llu\n", (unsigned long long)seed);
    printf("  ECUT       : %.0f keV\n", ECUT * 1000.0);
    printf("  Thread     : %d\n\n", num_thread);

    int *phantom = new int[NX * NY * NZ];
    double *dose = new double[NX * NY * NZ]();

    if (tipo_phantom == 0) {
        printf("Costruzione phantom con acqua\n");
        build_phantom_water(phantom);
    } else {
        printf("Costruzione phantom eterogeneo\n");
        build_phantom_hetero(phantom);
    }

    auto tempo_inizio = std::chrono::high_resolution_clock::now();
    std::atomic<long long> contatore{0};

    // Lancio dei thread
    std::vector<std::thread> threads;
    threads.reserve(num_thread);
    for (int t = 0; t < num_thread; t++) {
        threads.emplace_back(worker, t, num_thread, num_fotoni,
                             seed, phantom, dose,
                             std::ref(contatore), tempo_inizio);
    }

    // Attesa terminazione
    for (auto &th : threads) th.join();

    auto tempo_fine = std::chrono::high_resolution_clock::now();
    double tempo_esecuzione = std::chrono::duration<double>(
        tempo_fine - tempo_inizio).count();

    double *pdd                    = new double[NZ];
    double *coordinate_cm          = new double[NZ];
    double *profilo_dose           = new double[NX];
    double *coordinate_cm_laterali = new double[NX];

    print_dose_stats(dose, num_fotoni, tempo_esecuzione);
    compute_pdd(dose, pdd, coordinate_cm);
    compute_lateral_profile(dose, profilo_dose, coordinate_cm_laterali, 10.0);
    print_pdd_table(coordinate_cm, pdd, phantom_label);

    const char *pdd_file, *profilo_file, *slice_file, *bin_file;
    if (tipo_phantom == 0) {
        pdd_file     = "./CPU_Parallelo/pdd_water.csv";
        profilo_file = "./CPU_Parallelo/profile_water.csv";
        slice_file   = "./CPU_Parallelo/dose_slice_water.csv";
        bin_file     = "./CPU_Parallelo/dose_water.bin";
    } else {
        pdd_file     = "./CPU_Parallelo/pdd_hetero.csv";
        profilo_file = "./CPU_Parallelo/profile_hetero.csv";
        slice_file   = "./CPU_Parallelo/dose_slice_hetero.csv";
        bin_file     = "./CPU_Parallelo/dose_hetero.bin";
    }

    save_pdd_csv(coordinate_cm, pdd, pdd_file);
    save_profile_csv(coordinate_cm_laterali, profilo_dose, profilo_file);
    save_dose_slice_csv(dose, slice_file);
    save_dose_binary(dose, bin_file);

    delete[] phantom; delete[] dose;
    delete[] pdd; delete[] coordinate_cm;
    delete[] profilo_dose; delete[] coordinate_cm_laterali;

    printf("  Simulazione completata.\n");
    return 0;
}

Overwriting ./CPU_Parallelo/main.cpp


### File BeerLambert.cpp

E' stata implementata una versione semplificata di un simulatore Monte Carlo per radioterapia per validare la Legge di Beer-Lambert. Viene simulato il trasporto di fotoni (con uno spettro energetico da 6MV) attraverso un mezzo materiale per dimostrare che l'attenuazione della radiazione segua un decadimento esponenziale perfetto quando viene considerato solo l'assorbimento primario.

Per fare questo non appena il fotone interagisce con un voxel del phantom tutta la sua energia viene depositata immediatamente. Vengono ignorati i processi di scattering, come l'effetto Compton, dove il fotone cambierebbe solo direzione ed energia continuando il suo viaggio. In questo modo, il numero di fotoni che raggiungono una certa profondità dipende esclusivamente dalla probabilità di non aver interagito prima.

In [ ]:
%%writefile ./CPU_Parallelo/BeerLambert.cpp

#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>
#include <cstring>
#include <vector>

#include "physics.h"
#include "compton.h"
#include "random.h"
#include "phantom.h"
#include "output.h"

// stato di una particella sullo stack
struct Particle {
    double x, y, z;    // posizione [cm]
    double ux, uy, uz; // versore direzione (normalizzato)
    double energia;     // energia [MeV]
};

// CAMPIONAMENTO SORGENTE
inline Particle sample_source(const Spectrum &spettro, Xoshiro256 &rng) {
    static const double FIELD_HALF = 5.0;   // +-5 cm -> campo 10×10 cm^2
    double cx = PHANTOM_CM / 2.0;
    double cy = PHANTOM_CM / 2.0;

    Particle p;
    p.x = cx + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.y = cy + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.z = 1.0e-7;
    p.ux = 0.0;
    p.uy = 0.0;
    p.uz = 1.0;
    p.energia = spettro.sample(rng);
    return p;
}

// DISTANZA AL PROSSIMO CONFINE DI VOXEL
inline double boundary_distance(double x, double y, double z, double ux, double uy, double uz, int ix, int iy, int iz) {
    double distanza_minima_confine = 1.0e30; // inizializzata a infinito
    if (std::fabs(ux) > 1.0e-12) {
        double confine_voxel_X;
        if (ux > 0){
            confine_voxel_X = (ix + 1) * VOXEL_CM;
        } else{
            confine_voxel_X = ix * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_X - x) / ux;
        if (distanza_lineare > 1.0e-10){
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    if (std::fabs(uy) > 1.0e-12) {
        double confine_voxel_Y;
        if (uy > 0){
            confine_voxel_Y = (iy + 1) * VOXEL_CM;
        } else{
            confine_voxel_Y = iy * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_Y - y) / uy;
        if (distanza_lineare > 1.0e-10){
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    if (std::fabs(uz) > 1.0e-12) {
        double confine_voxel_Z;
        if (uz > 0){
            confine_voxel_Z = (iz + 1) * VOXEL_CM;
        } else{
            confine_voxel_Z = iz * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_Z - z) / uz;
        if (distanza_lineare > 1.0e-10) {
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    return distanza_minima_confine;
}

// TRASPORTO FOTONE SEMPLIFICATO PER BEER LAMBERT
void transport_photon(Particle p, const int *phantom, double *dose, Xoshiro256 &rng) {
    while (p.energia > ECUT && inside(p.x, p.y, p.z)) {
        int mat = phantom[phantom_idx((int)(p.x/VOXEL_CM), (int)(p.y/VOXEL_CM), (int)(p.z/VOXEL_CM))];
        double mu = get_mu_total(p.energia, mat);
        double d = -std::log(rng()) / mu;

        p.x += p.ux * d;
        p.y += p.uy * d;
        p.z += p.uz * d;

        if (inside(p.x, p.y, p.z)) {
            int ix = (int)(p.x / VOXEL_CM);
            int iy = (int)(p.y / VOXEL_CM);
            int iz = (int)(p.z / VOXEL_CM);

            dose[phantom_idx(ix, iy, iz)] += p.energia;

            break;
        }
    }
}

int main(int argc, char *argv[]) {

    // valori default
    long long num_fotoni = 1000000;   // default: 1M fotoni
    int tipo_phantom = 0;         // 0=acqua, 1=eterogeneo
    uint64_t seed   = 42ULL;

    if (argc > 1) num_fotoni = std::atoll(argv[1]);
    if (argc > 2) tipo_phantom = std::atoi(argv[2]);
    if (argc > 3) seed = (uint64_t)std::atoll(argv[3]);

    const char *phantom_label;
    if (tipo_phantom == 0){
      phantom_label = "Acqua omogenea";
    } else{
      phantom_label = "Acqua + Osso";
    }

    printf("  Monte Carlo per Radioterapia — CPU Sequenziale\n");
    printf("\n  Parametri:\n");
    printf("  Phantom    : %dx%dx%d voxel  |  voxel %.0fmm  |  %.0f³ cm³\n", NX, NY, NZ, VOXEL_CM * 10.0, PHANTOM_CM);
    printf("  Materiale  : %s\n", phantom_label);
    printf("  N fotoni   : %lld\n", num_fotoni);
    printf("  Seed       : %llu\n", (unsigned long long)seed);
    printf("  ECUT       : %.0f keV\n\n", ECUT * 1000.0);

    int *phantom = new int [NX * NY * NZ];
    double *dose = new double[NX * NY * NZ]();
    double *pdd = new double[NZ];
    double *coordinate_cm = new double[NZ];
    double *profilo_dose = new double[NX];
    double *coordinate_cm_laterali = new double[NX];

    if (tipo_phantom == 0){
        printf("Costruzione phantom con acqua \n");
        build_phantom_water(phantom);
    }else {
        printf("Costruzione phantom eterogeneo \n");
        build_phantom_hetero(phantom);
    }

    Spectrum spettro;  // spettro 6MV con CDF
    Xoshiro256 rng(seed); // PRNG xoshiro256**

    printf(" Avvio simulazione \n");
    auto tempo_inizio_esatto = std::chrono::high_resolution_clock::now();

    long long report_step = std::max(1LL, num_fotoni / 20);   // report ogni 5%

    for (long long i = 0; i < num_fotoni; i++) {
        if ((i + 1) % report_step == 0) {
            auto tempo_attuale     = std::chrono::high_resolution_clock::now();
            double tempo_esecuzione = std::chrono::duration<double>(tempo_attuale - tempo_inizio_esatto).count();
            double rate  = (i + 1) / tempo_esecuzione;
            double tempo_necessrio_fotoni_rimanenti   = (num_fotoni - i - 1) / rate;
            printf(" [%5.1f%%]  %.0f fotoni/s  Estimated Time of Arrival %.0fs\n", 100.0 * (i + 1) / num_fotoni, rate, tempo_necessrio_fotoni_rimanenti);
        }

        Particle p = sample_source(spettro, rng);
        transport_photon(p, phantom, dose, rng);
    }

    auto tempo_fine_esatto = std::chrono::high_resolution_clock::now();
    double tempo_esecuzione = std::chrono::duration<double>(tempo_fine_esatto - tempo_inizio_esatto).count();

    print_dose_stats(dose, num_fotoni, tempo_esecuzione);

    compute_pdd(dose, pdd, coordinate_cm);
    compute_lateral_profile(dose, profilo_dose, coordinate_cm_laterali, 10.0);
    print_pdd_table(coordinate_cm, pdd, phantom_label);

    const char *pdd_file;
    const char *profilo_file;
    const char *slice_file;
    const char *bin_file;

    if (tipo_phantom == 0){
      pdd_file = "./CPU_Parallelo/pdd_water_BL.csv";
      profilo_file = "./CPU_Parallelo/profile_water_BL.csv";
      slice_file = "./CPU_Parallelo/dose_slice_water_BL.csv";
      bin_file = "./CPU_Parallelo/dose_water_BL.bin";
    } else{
      pdd_file = "./CPU_Parallelo/pdd_hetero_BL.csv";
      profilo_file = "./CPU_Parallelo/profile_hetero_BL.csv";
      slice_file = "./CPU_Parallelo/dose_slice_hetero_BL.csv";
      bin_file = "./CPU_Parallelo/dose_hetero_BL.bin";
    }
    save_pdd_csv(coordinate_cm, pdd, pdd_file);
    save_profile_csv(coordinate_cm_laterali, profilo_dose, profilo_file);
    save_dose_slice_csv(dose, slice_file);
    save_dose_binary(dose, bin_file);

    delete[] phantom;
    delete[] dose;
    delete[] pdd;
    delete[] coordinate_cm;
    delete[] profilo_dose;
    delete[] coordinate_cm_laterali;

    printf("  Simulazione completata.\n");

    return 0;
}


Writing ./CPU_Parallelo/BeerLambert.cpp


## Compilazione

Compilazione main completo

In [ ]:
!g++ -O2 -std=c++17 -pthread -o ./CPU_Parallelo/mc_rt_cpu_parallelo ./CPU_Parallelo/main.cpp -lm

Compilazione main per validazione Beer Lambert

In [ ]:
!g++ -O2 -std=c++17 -o ./CPU_Parallelo/test_beer_lambert_parallelo ./CPU_Parallelo/BeerLambert.cpp -lm

## Esecuzione

In [ ]:
numero_fotoni = 1000000

In [ ]:
!./CPU_Parallelo/mc_rt_cpu_parallelo $numero_fotoni 0 42

  Monte Carlo per Radioterapia — CPU Parallelo (std::thread)

  Parametri:
  Phantom    : 100x100x100 voxel  |  voxel 3mm  |  30^3 cm^3
  Materiale  : Acqua omogenea
  N fotoni   : 1000000
  Seed       : 42
  ECUT       : 10 keV
  Thread     : 2

Costruzione phantom con acqua
 [  5.0%]  84748 fotoni/s  ETA 11s
 [ 10.0%]  92923 fotoni/s  ETA 10s
 [ 15.0%]  96658 fotoni/s  ETA 9s
 [ 20.0%]  96743 fotoni/s  ETA 8s
 [ 25.0%]  98270 fotoni/s  ETA 8s
 [ 30.0%]  87632 fotoni/s  ETA 8s
 [ 35.0%]  79142 fotoni/s  ETA 8s
 [ 40.0%]  71343 fotoni/s  ETA 8s
 [ 45.0%]  70883 fotoni/s  ETA 8s
 [ 50.0%]  73191 fotoni/s  ETA 7s
 [ 55.0%]  75069 fotoni/s  ETA 6s
 [ 60.0%]  76600 fotoni/s  ETA 5s
 [ 65.0%]  78135 fotoni/s  ETA 4s
 [ 70.0%]  79069 fotoni/s  ETA 4s
 [ 75.0%]  76692 fotoni/s  ETA 3s
 [ 80.0%]  75080 fotoni/s  ETA 3s
 [ 85.0%]  72308 fotoni/s  ETA 2s
 [ 90.0%]  71755 fotoni/s  ETA 1s
 [ 95.0%]  72558 fotoni/s  ETA 1s
 [100.0%]  73816 fotoni/s  ETA 0s

 Statistiche simulazione: 
  Particelle 

In [ ]:
!./CPU_Parallelo/mc_rt_cpu_parallelo $numero_fotoni 1 42

  Monte Carlo per Radioterapia — CPU Parallelo (std::thread)

  Parametri:
  Phantom    : 100x100x100 voxel  |  voxel 3mm  |  30^3 cm^3
  Materiale  : Acqua + Osso
  N fotoni   : 1000000
  Seed       : 42
  ECUT       : 10 keV
  Thread     : 2

Costruzione phantom eterogeneo
Inserto osseo: 4096 voxel = 110.6 cm³  ( volume teorico 125 cm³)
 [  5.0%]  97399 fotoni/s  ETA 10s
 [ 10.0%]  96223 fotoni/s  ETA 9s
 [ 15.0%]  72124 fotoni/s  ETA 12s
 [ 20.0%]  53459 fotoni/s  ETA 15s
 [ 25.0%]  42106 fotoni/s  ETA 18s
 [ 30.0%]  43057 fotoni/s  ETA 16s
 [ 35.0%]  43833 fotoni/s  ETA 15s
 [ 40.0%]  46788 fotoni/s  ETA 13s
 [ 45.0%]  49725 fotoni/s  ETA 11s
 [ 50.0%]  52203 fotoni/s  ETA 10s
 [ 55.0%]  54440 fotoni/s  ETA 8s
 [ 60.0%]  56339 fotoni/s  ETA 7s
 [ 65.0%]  58125 fotoni/s  ETA 6s
 [ 70.0%]  59699 fotoni/s  ETA 5s
 [ 75.0%]  61216 fotoni/s  ETA 4s
 [ 80.0%]  62465 fotoni/s  ETA 3s
 [ 85.0%]  63784 fotoni/s  ETA 2s
 [ 90.0%]  64943 fotoni/s  ETA 2s
 [ 95.0%]  66121 fotoni/s  ETA 1s
 [10

In [ ]:
!./CPU_Parallelo/test_beer_lambert_parallelo $numero_fotoni 0 42

  Monte Carlo per Radioterapia — CPU Sequenziale

  Parametri:
  Phantom    : 100x100x100 voxel  |  voxel 3mm  |  30³ cm³
  Materiale  : Acqua omogenea
  N fotoni   : 1000000
  Seed       : 42
  ECUT       : 10 keV

Costruzione phantom con acqua 
 Avvio simulazione 
 [  5.0%]  7004083 fotoni/s  Estimated Time of Arrival 0s
 [ 10.0%]  7184156 fotoni/s  Estimated Time of Arrival 0s
 [ 15.0%]  7300630 fotoni/s  Estimated Time of Arrival 0s
 [ 20.0%]  7447231 fotoni/s  Estimated Time of Arrival 0s
 [ 25.0%]  7473405 fotoni/s  Estimated Time of Arrival 0s
 [ 30.0%]  7550290 fotoni/s  Estimated Time of Arrival 0s
 [ 35.0%]  7583394 fotoni/s  Estimated Time of Arrival 0s
 [ 40.0%]  7532711 fotoni/s  Estimated Time of Arrival 0s
 [ 45.0%]  7536976 fotoni/s  Estimated Time of Arrival 0s
 [ 50.0%]  7492038 fotoni/s  Estimated Time of Arrival 0s
 [ 55.0%]  7493327 fotoni/s  Estimated Time of Arrival 0s
 [ 60.0%]  7492617 fotoni/s  Estimated Time of Arrival 0s
 [ 65.0%]  7490282 fotoni/s  Estimate

# Codice GPU V1

### physics.cuh

In [2]:
%%writefile ./physics.cuh

#pragma once

#ifdef __CUDACC__
#define HD __host__ __device__
#else
#define HD
#endif

#include <cmath>

// ======================================================
// COSTANTI FISICHE (Usa static const per l'accessibilità ovunque)
// ======================================================
static const double ME_C2 = 0.51099895;
static const double PI    = 3.14159265358979323846;
static const double ECUT  = 0.010;
static const double PCUT  = 0.100;

// ======================================================
// GEOMETRIA PHANTOM (Dimensioni fisse per il compilatore)
// ======================================================
static const int NX = 100;
static const int NY = 100;
static const int NZ = 100;
static const double VOXEL_CM   = 0.30;
static const double PHANTOM_CM = 30.0; // NX * VOXEL_CM

// ======================================================
// MATERIALI ED ENERGIA
// ======================================================
#define MAT_WATER 0
#define MAT_BONE  1
#define MAT_LUNG  2
#define MAT_AIR   3
#define N_MAT     4
static const int N_ENERGY = 28;

// ======================================================
// TABELLE DEVICE (__constant__)
// Dichiarate globalmente per essere visibili ai kernel
// ======================================================
#ifdef __CUDACC__
__constant__ double d_DENSITY[N_MAT];
__constant__ double d_ENERGY_GRID[N_ENERGY];
__constant__ double d_MU_TOTAL[N_MAT][N_ENERGY];
__constant__ double d_MU_PE[N_MAT][N_ENERGY];
__constant__ double d_MU_COMPTON[N_MAT][N_ENERGY];
__constant__ double d_MU_PAIR[N_MAT][N_ENERGY];
#endif

// ======================================================
// FUNZIONE DI CARICAMENTO (Da chiamare nel main)
// ======================================================
#ifdef __CUDACC__
inline void upload_physics_tables(const double* h_dens, const double* h_grid,
                                 const double* h_tot, const double* h_pe,
                                 const double* h_comp, const double* h_pair) {
    cudaMemcpyToSymbol(d_DENSITY,     h_dens, sizeof(double) * N_MAT);
    cudaMemcpyToSymbol(d_ENERGY_GRID, h_grid, sizeof(double) * N_ENERGY);
    cudaMemcpyToSymbol(d_MU_TOTAL,    h_tot,  sizeof(double) * N_MAT * N_ENERGY);
    cudaMemcpyToSymbol(d_MU_PE,       h_pe,   sizeof(double) * N_MAT * N_ENERGY);
    cudaMemcpyToSymbol(d_MU_COMPTON,  h_comp, sizeof(double) * N_MAT * N_ENERGY);
    cudaMemcpyToSymbol(d_MU_PAIR,     h_pair, sizeof(double) * N_MAT * N_ENERGY);
}
#endif

// ======================================================
// INTERPOLAZIONE E FISICA (Solo Device)
// ======================================================

HD inline double interp_mu(double energia_mev, const double* tabella) {
#ifdef __CUDA_ARCH__
    // Logica per GPU: usa d_ENERGY_GRID
    if (energia_mev <= d_ENERGY_GRID[0]) return tabella[0];
    if (energia_mev >= d_ENERGY_GRID[N_ENERGY-1]) return tabella[N_ENERGY-1];

    int lo = 0, hi = N_ENERGY - 1;
    while (hi - lo > 1) {
        int mid = (lo + hi) / 2;
        if (d_ENERGY_GRID[mid] <= energia_mev) lo = mid;
        else hi = mid;
    }
    double t = (energia_mev - d_ENERGY_GRID[lo]) / (d_ENERGY_GRID[hi] - d_ENERGY_GRID[lo]);
    return tabella[lo] * (1.0 - t) + tabella[hi] * t;
#else
    // Fallback per Host (opzionale, se serve testing su CPU)
    return 0.0;
#endif
}

// Helper per accedere alle righe delle matrici in __constant__
HD inline double get_mu_total(double energia, int mat) {
#ifdef __CUDA_ARCH__
    return interp_mu(energia, d_MU_TOTAL[mat]) * d_DENSITY[mat];
#else
    return 0.0;
#endif
}

HD inline double get_mu_pe(double energia, int mat) {
#ifdef __CUDA_ARCH__
    return interp_mu(energia, d_MU_PE[mat]) * d_DENSITY[mat];
#else
    return 0.0;
#endif
}

HD inline double get_mu_compton(double energia, int mat) {
#ifdef __CUDA_ARCH__
    return interp_mu(energia, d_MU_COMPTON[mat]) * d_DENSITY[mat];
#else
    return 0.0;
#endif
}

HD inline int select_interaction(double energia, int materiale, double xi) {
    double mu_tot = get_mu_total(energia, materiale);
    if (mu_tot <= 0.0) return 1;

    double p_pe = get_mu_pe(energia, materiale) / mu_tot;
    double p_comp = get_mu_compton(energia, materiale) / mu_tot;

    if (xi < p_pe) return 0;
    if (xi < p_pe + p_comp) return 1;
    return 2;
}

// ======================================================
// GEOMETRIA E INDICIZZAZIONE
// ======================================================
HD inline int phantom_idx(int ix, int iy, int iz) {
    return ix + NX * (iy + NY * iz);
}

HD inline bool inside(double x, double y, double z) {
    return x >= 0.0 && x < PHANTOM_CM &&
           y >= 0.0 && y < PHANTOM_CM &&
           z >= 0.0 && z < PHANTOM_CM;
}

HD inline int vox(double c) {
    int v = (int)(c / VOXEL_CM);
    if (v < 0) v = 0;
    if (v >= NX) v = NX - 1;
    return v;
}

Writing ./physics.cuh


### phantom

In [3]:
%%writefile ./phantom.cuh

#pragma once

#ifdef __CUDACC__
#define HD __host__ __device__
#else
#define HD
#endif

#include <cstdio>
#include "physics.cuh"

#ifdef __CUDACC__
#include <cuda_runtime.h>
#endif

// ======================================================
// PHANTOM GPU / CPU COMPATIBILE
// ======================================================
//
// Nota importante:
// - build_phantom_* : SOLO HOST (preparazione dati)
// - init_dose_host  : CPU
// - init_dose_gpu   : GPU con cudaMemset
//
// Non ha senso costruire il phantom dentro il device.
// Si prepara su host e poi si copia su GPU.
// ======================================================


// ------------------------------------------------------
// Phantom omogeneo (solo acqua)
// HOST ONLY
// ------------------------------------------------------
inline void build_phantom_water(int *phantom)
{
    const int total = NX * NY * NZ;

    for (int i = 0; i < total; i++)
        phantom[i] = MAT_WATER;
}


// ------------------------------------------------------
// Phantom eterogeneo con inserto osseo centrale
// HOST ONLY
// ------------------------------------------------------
inline void build_phantom_hetero(int *phantom)
{
    build_phantom_water(phantom);

    const int cx = NX / 2;
    const int cy = NY / 2;
    const int cz = NZ / 2;

    const int semi_lato = (int)(2.5 / VOXEL_CM + 0.5);

    int count = 0;

    for (int iz = cz - semi_lato; iz < cz + semi_lato; iz++)
    for (int iy = cy - semi_lato; iy < cy + semi_lato; iy++)
    for (int ix = cx - semi_lato; ix < cx + semi_lato; ix++)
    {
        if (ix >= 0 && ix < NX &&
            iy >= 0 && iy < NY &&
            iz >= 0 && iz < NZ)
        {
            phantom[phantom_idx(ix, iy, iz)] = MAT_BONE;
            count++;
        }
    }

    double volume = count * VOXEL_CM * VOXEL_CM * VOXEL_CM;

    printf("Inserto osseo: %d voxel = %.2f cm^3 (teorico 125 cm^3)\n",
           count, volume);
}


// ------------------------------------------------------
// Inizializzazione dose CPU
// HOST ONLY
// ------------------------------------------------------
inline void init_dose_host(double *dose)
{
    const int total = NX * NY * NZ;

    for (int i = 0; i < total; i++)
        dose[i] = 0.0;
}


#ifdef __CUDACC__

// ------------------------------------------------------
// Inizializzazione dose GPU
// ------------------------------------------------------
inline void init_dose_gpu(double *d_dose)
{
    cudaMemset(d_dose, 0, NX * NY * NZ * sizeof(double));
}

#endif

Writing ./phantom.cuh


### random

In [4]:
%%writefile ./random.cuh

#pragma once

#ifdef __CUDACC__
#define HD __host__ __device__
#else
#define HD
#endif

#include <cstdint>
#include <cmath>

// ======================================================
// RANDOM.H  VERSIONE GPU / CPU
// RNG per Monte Carlo parallelo CUDA
// ======================================================


// ======================================================
// XOSHIRO256++
// Un RNG eccellente per HPC / Monte Carlo
// Ogni thread CUDA avrà il suo stato indipendente.
// ======================================================

struct Xoshiro256
{
    uint64_t s[4];

    // -----------------------------------------
    // rotate left
    // -----------------------------------------
    HD static inline uint64_t rotl(uint64_t x, int k)
    {
        return (x << k) | (x >> (64 - k));
    }

    // -----------------------------------------
    // splitmix64 per seed iniziale
    // -----------------------------------------
    HD static inline uint64_t splitmix64(uint64_t &x)
    {
        x += 0x9e3779b97f4a7c15ULL;

        uint64_t z = x;

        z = (z ^ (z >> 30)) * 0xbf58476d1ce4e5b9ULL;
        z = (z ^ (z >> 27)) * 0x94d049bb133111ebULL;

        return z ^ (z >> 31);
    }

    // -----------------------------------------
    // costruttore seed
    // -----------------------------------------
    HD explicit Xoshiro256(uint64_t seed = 1ULL)
    {
        uint64_t x = seed;

        s[0] = splitmix64(x);
        s[1] = splitmix64(x);
        s[2] = splitmix64(x);
        s[3] = splitmix64(x);
    }

    // -----------------------------------------
    // prossimo uint64
    // -----------------------------------------
    HD inline uint64_t next_u64()
    {
        const uint64_t result = rotl(s[1] * 5ULL, 7) * 9ULL;

        const uint64_t t = s[1] << 17;

        s[2] ^= s[0];
        s[3] ^= s[1];
        s[1] ^= s[2];
        s[0] ^= s[3];

        s[2] ^= t;
        s[3] = rotl(s[3], 45);

        return result;
    }

    // -----------------------------------------
    // uniforme (0,1)
    // -----------------------------------------
    HD inline double operator()()
    {
        double r;

        do
        {
            r = (double)(next_u64() >> 11) *
                (1.0 / 9007199254740992.0);   // 2^53
        }
        while (r <= 0.0);

        return r;
    }
};


// ======================================================
// SPETTRO 6 MV
// ======================================================

static const int SPECTRUM_BINS = 24;

static const double SPECTRUM_E[SPECTRUM_BINS] =
{
    0.25, 0.50, 0.75, 1.00, 1.25, 1.50,
    1.75, 2.00, 2.25, 2.50, 2.75, 3.00,
    3.25, 3.50, 3.75, 4.00, 4.25, 4.50,
    4.75, 5.00, 5.25, 5.50, 5.75, 6.00
};

static const double SPECTRUM_FLUENCE[SPECTRUM_BINS] =
{
    0.0243, 0.0676, 0.0862, 0.0929, 0.0919, 0.0868,
    0.0794, 0.0712, 0.0628, 0.0548, 0.0471, 0.0399,
    0.0334, 0.0276, 0.0224, 0.0178, 0.0138, 0.0104,
    0.0075, 0.0052, 0.0034, 0.0020, 0.0010, 0.0004
};


// ======================================================
// SPECTRUM
// ======================================================

struct Spectrum
{
    double cdf[SPECTRUM_BINS];
    double energie[SPECTRUM_BINS];
    double bin_width;

    // -----------------------------------------
    // costruttore
    // -----------------------------------------
    HD Spectrum()
    {
        double sum = 0.0;

        for (int i = 0; i < SPECTRUM_BINS; i++)
            sum += SPECTRUM_FLUENCE[i];

        cdf[0] = SPECTRUM_FLUENCE[0] / sum;

        for (int i = 1; i < SPECTRUM_BINS; i++)
            cdf[i] = cdf[i - 1] + SPECTRUM_FLUENCE[i] / sum;

        cdf[SPECTRUM_BINS - 1] = 1.0;

        for (int i = 0; i < SPECTRUM_BINS; i++)
            energie[i] = SPECTRUM_E[i];

        bin_width = 0.25;
    }

    // -----------------------------------------
    // sample energia
    // -----------------------------------------
    HD inline double sample(Xoshiro256 &rng) const
    {
        const double xi = rng();

        int lo = 0;
        int hi = SPECTRUM_BINS - 1;

        while (lo < hi)
        {
            int mid = (lo + hi) / 2;

            if (cdf[mid] < xi)
                lo = mid + 1;
            else
                hi = mid;
        }

        double E = energie[lo];

        E += (rng() - 0.5) * bin_width;

        if (E < 0.01) E = 0.01;
        if (E > 6.00) E = 6.00;

        return E;
    }
};

Writing ./random.cuh


### output

In [5]:
%%writefile ./output.cuh

#pragma once

#ifdef __CUDACC__
#define HD __host__ __device__
#else
#define HD
#endif

#include <cstdio>
#include <cmath>
#include <fstream>
#include "physics.cuh"

// ======================================================
// OUTPUT.H  VERSIONE GPU / CPU
//
// IMPORTANTE:
// Le analisi e il salvataggio file si fanno su HOST.
// Dopo il kernel CUDA copi dose GPU -> CPU e usi queste.
//
// Nessun __device__ sulle funzioni I/O.
// ======================================================


// ======================================================
// utility min semplice compatibile CUDA
// ======================================================
HD inline int imin(int a, int b)
{
    return (a < b) ? a : b;
}


// ======================================================
// PDD
// ======================================================
inline void compute_pdd(
    const double *dose,
    double *pdd,
    double *profondita,
    int semiampiezza_media = 8)
{
    const int cx = NX / 2;
    const int cy = NY / 2;

    double max_dose = 0.0;

    for (int iz = 0; iz < NZ; iz++)
    {
        double somma = 0.0;
        int count = 0;

        for (int ix = cx - semiampiezza_media;
             ix <= cx + semiampiezza_media; ix++)
        {
            for (int iy = cy - semiampiezza_media;
                 iy <= cy + semiampiezza_media; iy++)
            {
                if (ix >= 0 && ix < NX &&
                    iy >= 0 && iy < NY)
                {
                    somma += dose[phantom_idx(ix, iy, iz)];
                    count++;
                }
            }
        }

        if (count > 0)
            pdd[iz] = somma / count;
        else
            pdd[iz] = 0.0;

        profondita[iz] = (iz + 0.5) * VOXEL_CM;

        if (pdd[iz] > max_dose)
            max_dose = pdd[iz];
    }

    if (max_dose > 0.0)
    {
        for (int iz = 0; iz < NZ; iz++)
            pdd[iz] = 100.0 * pdd[iz] / max_dose;
    }
}


// ======================================================
// PROFILO LATERALE
// ======================================================
inline void compute_lateral_profile(
    const double *dose,
    double *profile,
    double *coord_cm,
    double depth_cm = 10.0,
    int semiampiezza = 2)
{
    int iz = imin((int)(depth_cm / VOXEL_CM), NZ - 1);

    const int cy = NY / 2;

    double maxdose = 0.0;

    for (int ix = 0; ix < NX; ix++)
    {
        double sum = 0.0;
        int count = 0;

        for (int iy = cy - semiampiezza;
             iy <= cy + semiampiezza; iy++)
        {
            if (iy >= 0 && iy < NY)
            {
                sum += dose[phantom_idx(ix, iy, iz)];
                count++;
            }
        }

        if (count > 0)
            profile[ix] = sum / count;
        else
            profile[ix] = 0.0;

        coord_cm[ix] =
            (ix + 0.5) * VOXEL_CM - PHANTOM_CM / 2.0;

        if (profile[ix] > maxdose)
            maxdose = profile[ix];
    }

    if (maxdose > 0.0)
    {
        for (int ix = 0; ix < NX; ix++)
            profile[ix] = 100.0 * profile[ix] / maxdose;
    }
}


// ======================================================
// SALVA PDD CSV
// ======================================================
inline void save_pdd_csv(
    const double *depth,
    const double *pdd,
    const char *filename)
{
    std::ofstream f(filename);

    f << "depth_cm,dose_percent\n";

    for (int iz = 0; iz < NZ; iz++)
        f << depth[iz] << "," << pdd[iz] << "\n";

    f.close();

    printf("Salvato: %s\n", filename);
}


// ======================================================
// SALVA PROFILO CSV
// ======================================================
inline void save_profile_csv(
    const double *x,
    const double *dose,
    const char *filename)
{
    std::ofstream f(filename);

    f << "position_cm,dose_percent\n";

    for (int i = 0; i < NX; i++)
        f << x[i] << "," << dose[i] << "\n";

    f.close();

    printf("Salvato: %s\n", filename);
}


// ======================================================
// SLICE CENTRALE
// ======================================================
inline void save_dose_slice_csv(
    const double *dose,
    const char *filename)
{
    std::ofstream f(filename);

    const int iy = NY / 2;

    for (int iz = 0; iz < NZ; iz++)
    {
        for (int ix = 0; ix < NX; ix++)
        {
            f << dose[phantom_idx(ix, iy, iz)];

            if (ix < NX - 1)
                f << ",";
        }

        f << "\n";
    }

    f.close();

    printf("Salvato: %s\n", filename);
}


// ======================================================
// BINARY 3D
// ======================================================
inline void save_dose_binary(
    const double *dose,
    const char *filename)
{
    FILE *f = fopen(filename, "wb");

    if (!f)
    {
        printf("Errore apertura %s\n", filename);
        return;
    }

    fwrite(dose, sizeof(double), NX * NY * NZ, f);

    fclose(f);

    printf("Salvato: %s\n", filename);
}


// ======================================================
// TABELLA PDD
// ======================================================
inline void print_pdd_table(
    const double *z,
    const double *pdd,
    const char *label)
{
    printf("\nPDD - %s\n", label);
    printf("-----------------------------\n");

    double refs[] =
    {
        1.5, 3.0, 5.0, 10.0,
        15.0, 20.0, 25.0
    };

    for (int i = 0; i < 7; i++)
    {
        int k = (int)(refs[i] / VOXEL_CM);

        if (k >= 0 && k < NZ)
            printf("%6.1f cm   %8.3f %%\n",
                   z[k], pdd[k]);
    }
}


// ======================================================
// STATISTICHE
// ======================================================
inline void print_dose_stats(
    const double *dose,
    long long N,
    double sec)
{
    double maxdose = 0.0;
    double etot = 0.0;
    int hit = 0;

    for (int i = 0; i < NX * NY * NZ; i++)
    {
        if (dose[i] > 0.0)
        {
            hit++;
            etot += dose[i];

            if (dose[i] > maxdose)
                maxdose = dose[i];
        }
    }

    printf("\nStatistiche simulazione\n");
    printf("-----------------------\n");
    printf("Particelle      : %lld\n", N);
    printf("Tempo [s]       : %.3f\n", sec);
    printf("Throughput MP/s : %.3f\n", N / sec / 1e6);
    printf("Voxel colpiti   : %d / %d\n", hit, NX*NY*NZ);
    printf("Energia totale  : %.6e MeV\n", etot);
    printf("Energia/partic. : %.6e MeV\n", etot / (double)N);
    printf("Dose max        : %.6e\n", maxdose);
}

Writing ./output.cuh


### compton

In [104]:
%%writefile ./compton.cuh

#ifdef __CUDACC__
#define HD __host__ __device__
#else
#define HD
#endif

#pragma once

#include <math.h>
#include "physics.cuh"

// ============================================================
// Utility compatibili CUDA
// ============================================================
HD inline double gpu_min(double a, double b) { return (a < b) ? a : b; }
HD inline double gpu_max(double a, double b) { return (a > b) ? a : b; }

// ============================================================
// KAHN COMPTON SAMPLING (GPU SAFE)
// ============================================================
HD inline void kahn_compton(
    double energia_mev,
    double xi_1,
    double xi_2,
    double xi_3,
    double &cos_theta,
    double &energia_scatter
) {
    double alpha   = energia_mev / ME_C2;
    double tau_min = 1.0 / (1.0 + 2.0 * alpha);

    double area_ramo_1 = log(1.0 / tau_min);
    double area_ramo_2 = 0.5 * (1.0 - tau_min * tau_min);
    double area_totale = area_ramo_1 + area_ramo_2;

    double tau;

    if (xi_1 * area_totale < area_ramo_1) {
        // ramo logaritmico
        tau = pow(tau_min, 1.0 - xi_2);
    } else {
        // ramo quadratico
        double tau_min2 = tau_min * tau_min;
        double tau2 = tau_min2 + xi_2 * (1.0 - tau_min2);
        tau = sqrt(gpu_max(tau2, 1e-30));
    }

    tau = gpu_min(gpu_max(tau, tau_min), 1.0);

    cos_theta = 1.0 - (1.0 - tau) / (alpha * tau);
    cos_theta = gpu_min(gpu_max(cos_theta, -1.0), 1.0);

    energia_scatter = tau * energia_mev;

    // probabilità di accettazione Klein-Nishina
    double sin2_theta = gpu_max(0.0, 1.0 - cos_theta * cos_theta);
    double corr = (tau * sin2_theta) / (1.0 + tau * tau);

    double p_acc = 1.0 - corr;
    p_acc = gpu_min(gpu_max(p_acc, 0.0), 1.0);

    // rejection
    if (xi_3 > p_acc) {
        cos_theta = 2.0;   // flag rigetto
    }
}

// ============================================================
// WRAPPER rejection loop
// ============================================================
template<typename RNG>
HD inline void sample_compton(
    double energia_mev,
    RNG &rng,
    double &cos_theta,
    double &energia_scatter
) {
    while (true) {
        double xi1 = rng();
        double xi2 = rng();
        double xi3 = rng();

        kahn_compton(
            energia_mev,
            xi1, xi2, xi3,
            cos_theta,
            energia_scatter
        );

        if (cos_theta <= 1.0)
            return;
    }
}

// ============================================================
// ROTAZIONE DIREZIONE FOTONE DOPO SCATTERING
// ============================================================
HD inline void rotate_direction(
    double &ux,
    double &uy,
    double &uz,
    double cos_theta,
    double phi
) {
    double sin_theta = sqrt(gpu_max(0.0, 1.0 - cos_theta * cos_theta));

    double cos_phi = cos(phi);
    double sin_phi = sin(phi);

    double ux_new, uy_new, uz_new;

    // caso quasi parallelo asse z
    if (fabs(uz) > 0.99999) {

        double segno = (uz >= 0.0) ? 1.0 : -1.0;

        ux_new = sin_theta * cos_phi;
        uy_new = sin_theta * sin_phi * segno;
        uz_new = cos_theta * segno;

    } else {

        double proj_xy = sqrt(1.0 - uz * uz);

        ux_new =
            sin_theta *
            (ux * uz * cos_phi - uy * sin_phi) / proj_xy
            + ux * cos_theta;

        uy_new =
            sin_theta *
            (uy * uz * cos_phi + ux * sin_phi) / proj_xy
            + uy * cos_theta;

        uz_new =
            -sin_theta * cos_phi * proj_xy
            + uz * cos_theta;
    }

    // normalizzazione
    double norm = sqrt(
        ux_new * ux_new +
        uy_new * uy_new +
        uz_new * uz_new
    );

    if (norm > 0.0) {
        ux = ux_new / norm;
        uy = uy_new / norm;
        uz = uz_new / norm;
    }
}

Overwriting ./compton.cuh


### main

In [55]:
# ── Crea directory di lavoro ──────────────────────────────────
import os
os.makedirs('GPU_CUDA', exist_ok=True)
print('Directory GPU_CUDA creata.')

Directory GPU_CUDA creata.


In [131]:
%%writefile GPU_CUDA/physics.cuh
#pragma once

// ============================================================
//  physics.cuh  —  Costanti fisiche e funzioni di lookup
//  Tutte le funzioni sono __host__ __device__ così possono
//  essere chiamate sia da CPU (main) che da GPU (kernel).
// ============================================================

// COSTANTI FISICHE
static const double ME_C2    = 0.51099895;
static const double PI       = 3.14159265358979323846;
static const double ECUT     = 0.010;
static const double PCUT     = 0.100;

// GEOMETRIA PHANTOM
static const int    NX = 100, NY = 100, NZ = 100;
static const double VOXEL_CM  = 0.30;
static const double PHANTOM_CM = NX * VOXEL_CM;

// MATERIALI
#define MAT_WATER 0
#define MAT_BONE  1
#define MAT_LUNG  2
#define MAT_AIR   3
#define N_MAT     4

static const double DENSITY[N_MAT] = { 1.000, 1.850, 0.260, 0.001205 };
static const int N_ENERGY = 28;
static const double ENERGY_GRID[28] = {
    0.010, 0.015, 0.020, 0.030, 0.040, 0.050, 0.060, 0.080, 0.100,
    0.150, 0.200, 0.300, 0.400, 0.500, 0.600, 0.800, 1.000, 1.250,
    1.500, 2.000, 3.000, 4.000, 5.000, 6.000, 8.000, 10.000, 15.000, 20.000
};

__device__ __constant__ double d_DENSITY[N_MAT]     = { 1.000, 1.850, 0.260, 0.001205 };
__device__ __constant__ double d_ENERGY_GRID[28]    = {
    0.010, 0.015, 0.020, 0.030, 0.040, 0.050, 0.060, 0.080, 0.100,
    0.150, 0.200, 0.300, 0.400, 0.500, 0.600, 0.800, 1.000, 1.250,
    1.500, 2.000, 3.000, 4.000, 5.000, 6.000, 8.000, 10.000, 15.000, 20.000
};

__device__ __constant__ double d_MU_TOTAL[N_MAT][28] = {
    { 5.329, 1.673, 0.8096, 0.3756, 0.2683, 0.2269, 0.2059, 0.1837, 0.1707,
      0.1505, 0.1370, 0.1186, 0.1061, 0.09687, 0.09007, 0.07865, 0.07072, 0.06323,
      0.05754, 0.04942, 0.03969, 0.03403, 0.03031, 0.02770, 0.02429, 0.02219, 0.01941, 0.01813 },
    { 19.89, 7.131, 3.085, 1.012, 0.5475, 0.3941, 0.3178, 0.2595, 0.2368,
      0.1958, 0.1698, 0.1393, 0.1222, 0.1107, 0.1018, 0.08795, 0.07838, 0.06945,
      0.06283, 0.05351, 0.04257, 0.03624, 0.03209, 0.02913, 0.02536, 0.02296, 0.01978, 0.01832 },
    { 5.208, 1.638, 0.7933, 0.3681, 0.2629, 0.2224, 0.2018, 0.1800, 0.1673,
      0.1475, 0.1342, 0.1162, 0.1040, 0.09493, 0.08827, 0.07708, 0.06930, 0.06196,
      0.05639, 0.04843, 0.03889, 0.03335, 0.02970, 0.02715, 0.02380, 0.02175, 0.01902, 0.01776 },
    { 36.01, 12.28, 5.279, 1.625, 0.6918, 0.3954, 0.2885, 0.2085, 0.1875,
      0.1504, 0.1340, 0.1123, 0.09921, 0.09076, 0.08414, 0.07285, 0.06529, 0.05817,
      0.05298, 0.04534, 0.03597, 0.03054, 0.02699, 0.02453, 0.02131, 0.01931, 0.01673, 0.01551 }
};

__device__ __constant__ double d_MU_PE[N_MAT][28] = {
    { 4.944, 1.374, 0.5195, 0.1036, 0.02407, 0.005800, 0.001334, 5.510e-5, 3.998e-5,
      2.799e-6, 2.200e-7, 1.400e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    { 19.35, 6.833, 2.818, 0.7469, 0.2837, 0.1152, 0.04660, 0.008680, 0.001900,
      1.800e-4, 2.000e-5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    { 4.845, 1.346, 0.5091, 0.1015, 0.02359, 0.005684, 0.001307, 5.400e-5, 3.918e-5,
      2.743e-6, 2.156e-7, 1.372e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    { 35.52, 11.99, 5.012, 1.379, 0.4529, 0.1581, 0.05757, 0.008251, 0.001581,
      8.208e-5, 7.636e-6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 }
};

__device__ __constant__ double d_MU_COMPTON[N_MAT][28] = {
    { 0.3854, 0.2988, 0.2672, 0.2651, 0.2595, 0.2476, 0.2329, 0.1984, 0.1661,
      0.1505, 0.1370, 0.1186, 0.1061, 0.09687, 0.09007, 0.07865, 0.07072, 0.06323,
      0.05754, 0.04942, 0.03969, 0.03403, 0.03031, 0.02770, 0.02429, 0.02219, 0.01878, 0.01719 },
    { 0.4869, 0.2684, 0.2503, 0.2465, 0.2429, 0.2310, 0.2172, 0.1848, 0.1548,
      0.1400, 0.1275, 0.1103, 0.09870, 0.09010, 0.08377, 0.07313, 0.06575, 0.05862,
      0.05338, 0.04579, 0.03667, 0.03133, 0.02784, 0.02539, 0.02217, 0.02016, 0.01702, 0.01552 },
    { 0.3777, 0.2928, 0.2619, 0.2598, 0.2543, 0.2426, 0.2282, 0.1944, 0.1628,
      0.1475, 0.1342, 0.1162, 0.1040, 0.09493, 0.08827, 0.07708, 0.06930, 0.06196,
      0.05639, 0.04843, 0.03889, 0.03335, 0.02970, 0.02715, 0.02380, 0.02175, 0.01840, 0.01684 },
    { 0.3779, 0.2933, 0.2624, 0.2602, 0.2547, 0.2430, 0.2285, 0.1946, 0.1630,
      0.1477, 0.1344, 0.1163, 0.1041, 0.09516, 0.08844, 0.07723, 0.06942, 0.06207,
      0.05649, 0.04852, 0.03894, 0.03339, 0.02973, 0.02718, 0.02382, 0.02177, 0.01843, 0.01686 }
};

__device__ __constant__ double d_MU_PAIR[N_MAT][28] = {
    { 0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0, 0.000630, 0.000940 },
    { 0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0, 0.002760, 0.002800 },
    { 0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0, 0.000617, 0.000921 },
    { 0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0, 0.000589, 0.000879 }
};

static const double MU_TOTAL[N_MAT][28] = {
    { 5.329, 1.673, 0.8096, 0.3756, 0.2683, 0.2269, 0.2059, 0.1837, 0.1707,
      0.1505, 0.1370, 0.1186, 0.1061, 0.09687, 0.09007, 0.07865, 0.07072, 0.06323,
      0.05754, 0.04942, 0.03969, 0.03403, 0.03031, 0.02770, 0.02429, 0.02219, 0.01941, 0.01813 },
    { 19.89, 7.131, 3.085, 1.012, 0.5475, 0.3941, 0.3178, 0.2595, 0.2368,
      0.1958, 0.1698, 0.1393, 0.1222, 0.1107, 0.1018, 0.08795, 0.07838, 0.06945,
      0.06283, 0.05351, 0.04257, 0.03624, 0.03209, 0.02913, 0.02536, 0.02296, 0.01978, 0.01832 },
    { 5.208, 1.638, 0.7933, 0.3681, 0.2629, 0.2224, 0.2018, 0.1800, 0.1673,
      0.1475, 0.1342, 0.1162, 0.1040, 0.09493, 0.08827, 0.07708, 0.06930, 0.06196,
      0.05639, 0.04843, 0.03889, 0.03335, 0.02970, 0.02715, 0.02380, 0.02175, 0.01902, 0.01776 },
    { 36.01, 12.28, 5.279, 1.625, 0.6918, 0.3954, 0.2885, 0.2085, 0.1875,
      0.1504, 0.1340, 0.1123, 0.09921, 0.09076, 0.08414, 0.07285, 0.06529, 0.05817,
      0.05298, 0.04534, 0.03597, 0.03054, 0.02699, 0.02453, 0.02131, 0.01931, 0.01673, 0.01551 }
};

__device__ __forceinline__
double d_interp_mu(double e, const double* __restrict__ tab) {
    if (e <= d_ENERGY_GRID[0])         return tab[0];
    if (e >= d_ENERGY_GRID[27]) return tab[27];
    int lo = 0, hi = 27;
    while (hi - lo > 1) { int m=(lo+hi)>>1; if(d_ENERGY_GRID[m]<=e) lo=m; else hi=m; }
    double t = (e - d_ENERGY_GRID[lo]) / (d_ENERGY_GRID[hi] - d_ENERGY_GRID[lo]);
    return tab[lo]*(1.0-t) + tab[hi]*t;
}
__device__ __forceinline__ double d_get_mu_total(double e, int mat)   { return d_interp_mu(e, d_MU_TOTAL[mat])   * d_DENSITY[mat]; }
__device__ __forceinline__ double d_get_mu_pe(double e, int mat)      { return d_interp_mu(e, d_MU_PE[mat])      * d_DENSITY[mat]; }
__device__ __forceinline__ double d_get_mu_compton(double e, int mat) { return d_interp_mu(e, d_MU_COMPTON[mat]) * d_DENSITY[mat]; }

__device__ __forceinline__
int d_select_interaction(double e, int mat, double xi) {
    double mu = d_get_mu_total(e, mat);
    if (mu <= 0.0) return 1;
    double p_pe = d_get_mu_pe(e, mat) / mu;
    double p_co = d_get_mu_compton(e, mat) / mu;
    if (xi < p_pe)        return 0;
    if (xi < p_pe + p_co) return 1;
    return 2;
}

__host__ __device__ __forceinline__ int phantom_idx(int ix, int iy, int iz) { return ix + NX*(iy + NY*iz); }
__host__ __device__ __forceinline__ bool inside(double x, double y, double z) {
    return x>=0.0 && x<PHANTOM_CM && y>=0.0 && y<PHANTOM_CM && z>=0.0 && z<PHANTOM_CM;
}
__host__ __device__ __forceinline__ int vox(double c) {
    int v=(int)(c/VOXEL_CM); if(v<0) v=0; if(v>=NX) v=NX-1; return v;
}

inline double interp_mu(double e, const double tab[28]) {
    if(e<=ENERGY_GRID[0]) return tab[0];
    if(e>=ENERGY_GRID[27]) return tab[27];
    int lo=0,hi=27;
    while(hi-lo>1){int m=(lo+hi)/2; if(ENERGY_GRID[m]<=e) lo=m; else hi=m;}
    double t=(e-ENERGY_GRID[lo])/(ENERGY_GRID[hi]-ENERGY_GRID[lo]);
    return tab[lo]*(1.0-t)+tab[hi]*t;
}
inline double get_mu_total(double e, int mat) { return interp_mu(e, MU_TOTAL[mat])*DENSITY[mat]; }

Overwriting GPU_CUDA/physics.cuh


In [122]:
%%writefile GPU_CUDA/random.cuh
#pragma once
#include <cstdint>

struct Xoshiro256 {
    uint64_t s[4];
    __host__ __device__ static uint64_t splitmix64(uint64_t &x) {
        x += 0x9e3779b97f4a7c15ULL;
        uint64_t z = x;
        z = (z^(z>>30))*0xbf58476d1ce4e5b9ULL;
        z = (z^(z>>27))*0x94d049bb133111ebULL;
        return z^(z>>31);
    }
    __host__ __device__ explicit Xoshiro256(uint64_t seed) {
        s[0]=splitmix64(seed); s[1]=splitmix64(seed);
        s[2]=splitmix64(seed); s[3]=splitmix64(seed);
    }
    __host__ __device__ Xoshiro256() {}
    __host__ __device__ static uint64_t rotl(uint64_t x, int k) { return (x<<k)|(x>>(64-k)); }
    __host__ __device__ uint64_t next() {
        const uint64_t r = rotl(s[1]*5,7)*9;
        const uint64_t t = s[1]<<17;
        s[2]^=s[0]; s[3]^=s[1]; s[1]^=s[2]; s[0]^=s[3]; s[2]^=t;
        s[3]=rotl(s[3],45); return r;
    }
    __host__ __device__ double operator()() {
        double r;
        do { r=(double)(next()>>11)*(1.0/(double)(1ULL<<53)); } while(r<=0.0);
        return r;
    }
};

static const int SPECTRUM_BINS = 24;
static const double SPECTRUM_E[24]       = { 0.25,0.50,0.75,1.00,1.25,1.50,1.75,2.00,2.25,2.50,2.75,3.00,3.25,3.50,3.75,4.00,4.25,4.50,4.75,5.00,5.25,5.50,5.75,6.00 };
static const double SPECTRUM_FLUENCE[24] = { 0.0243,0.0676,0.0862,0.0929,0.0919,0.0868,0.0794,0.0712,0.0628,0.0548,0.0471,0.0399,0.0334,0.0276,0.0224,0.0178,0.0138,0.0104,0.0075,0.0052,0.0034,0.0020,0.0010,0.0004 };

__device__ __constant__ double d_spectrum_cdf[24];
__device__ __constant__ double d_spectrum_e  [24];
__device__ __constant__ double d_spectrum_dE;

__device__ __forceinline__
double d_sample_spectrum(Xoshiro256 &rng) {
    double xi = rng();
    int lo=0, hi=SPECTRUM_BINS-1;
    while(lo<hi){ int m=(lo+hi)>>1; if(d_spectrum_cdf[m]<xi) lo=m+1; else hi=m; }
    double e = d_spectrum_e[lo] + (rng()-0.5)*d_spectrum_dE;
    if(e<0.01) e=0.01; if(e>6.00) e=6.00; return e;
}

struct Spectrum {
    double cdf[24]; double energie[24]; double dE;
    Spectrum() {
        double s=0; for(int i=0;i<24;i++) s+=SPECTRUM_FLUENCE[i];
        cdf[0]=SPECTRUM_FLUENCE[0]/s;
        for(int i=1;i<24;i++) cdf[i]=cdf[i-1]+SPECTRUM_FLUENCE[i]/s;
        cdf[23]=1.0; for(int i=0;i<24;i++) energie[i]=SPECTRUM_E[i]; dE=0.25;
    }
    double sample(Xoshiro256 &rng) const {
        double xi=rng(); int lo=0,hi=23;
        while(lo<hi){int m=(lo+hi)/2; if(cdf[m]<xi) lo=m+1; else hi=m;}
        double e=energie[lo]+(rng()-0.5)*dE;
        if(e<0.01) e=0.01; if(e>6.00) e=6.00; return e;
    }
};

inline void upload_spectrum_to_device() {
    double cdf_h[24], e_h[24]; double s=0;
    for(int i=0;i<24;i++) s+=SPECTRUM_FLUENCE[i];
    cdf_h[0]=SPECTRUM_FLUENCE[0]/s;
    for(int i=1;i<24;i++) cdf_h[i]=cdf_h[i-1]+SPECTRUM_FLUENCE[i]/s;
    cdf_h[23]=1.0; for(int i=0;i<24;i++) e_h[i]=SPECTRUM_E[i];
    double dE=0.25;
    cudaMemcpyToSymbol(d_spectrum_cdf,cdf_h,sizeof(cdf_h));
    cudaMemcpyToSymbol(d_spectrum_e,  e_h,  sizeof(e_h));
    cudaMemcpyToSymbol(d_spectrum_dE, &dE,  sizeof(double));
}

Overwriting GPU_CUDA/random.cuh


In [58]:
%%writefile GPU_CUDA/compton.cuh
#pragma once
#include "physics.cuh"
#include "random.cuh"
__device__ __forceinline__
void d_kahn_compton(double e, double xi1, double xi2, double xi3,
                    double &cos_theta, double &e_scatter) {
    double alpha   = e / ME_C2;
    double tau_min = 1.0 / (1.0 + 2.0*alpha);
    double a1 = log(1.0/tau_min);
    double a2 = (1.0 - tau_min*tau_min)*0.5;
    double tau;
    if (xi1*(a1+a2) < a1) {
        tau = pow(tau_min, 1.0-xi2);
    } else {
        double tmin2 = tau_min*tau_min;
        tau = sqrt(fmax(tmin2 + xi2*(1.0-tmin2), 1e-30));
    }
    tau = fmin(fmax(tau,tau_min),1.0);
    cos_theta = 1.0 - (1.0-tau)/(alpha*tau);
    cos_theta = fmin(fmax(cos_theta,-1.0),1.0);
    e_scatter = tau*e;
    double sin2 = fmax(0.0, 1.0-cos_theta*cos_theta);
    double corr = (tau*sin2)/(1.0+tau*tau);
    double p_acc = fmax(0.0, fmin(1.0, 1.0-corr));
    if (xi3 > p_acc) cos_theta = 2.0;
}

__device__ __forceinline__
void d_sample_compton(double e, Xoshiro256 &rng,
                      double &cos_theta, double &e_scatter) {
    while(true) {
        d_kahn_compton(e, rng(), rng(), rng(), cos_theta, e_scatter);
        if (cos_theta <= 1.0) return;
    }
}

__device__ __forceinline__
void d_rotate_direction(double &ux, double &uy, double &uz,
                        double cos_theta, double phi) {
    double sin_t = sqrt(fmax(0.0, 1.0-cos_theta*cos_theta));
    double cp = cos(phi), sp = sin(phi);
    double nx, ny, nz;
    if (fabs(uz) > 0.99999) {
        double sgn = (uz>0.0) ? 1.0 : -1.0;
        nx = sin_t*cp; ny = sin_t*sp*sgn; nz = cos_theta*sgn;
    } else {
        double proj = sqrt(1.0-uz*uz);
        nx = sin_t*(ux*uz*cp - uy*sp)/proj + ux*cos_theta;
        ny = sin_t*(uy*uz*cp + ux*sp)/proj + uy*cos_theta;
        nz = -sin_t*cp*proj               + uz*cos_theta;
    }
    double norm = sqrt(nx*nx+ny*ny+nz*nz);
    if (norm>0.0) { ux=nx/norm; uy=ny/norm; uz=nz/norm; }
}

Overwriting GPU_CUDA/compton.cuh


In [123]:
%%writefile GPU_CUDA/phantom.cuh
#pragma once
#include <cstdio>
#include <cstring>
#include "physics.cuh"

inline void build_phantom_water(int *p) {
    for(int i=0;i<NX*NY*NZ;i++) p[i]=MAT_WATER;
}
inline void build_phantom_hetero(int *p) {
    build_phantom_water(p);
    int cx=NX/2,cy=NY/2,cz=NZ/2,r=(int)(2.5/VOXEL_CM+0.5),cnt=0;
    for(int iz=cz-r;iz<cz+r;iz++)
    for(int iy=cy-r;iy<cy+r;iy++)
    for(int ix=cx-r;ix<cx+r;ix++) {
        if(ix>=0&&ix<NX&&iy>=0&&iy<NY&&iz>=0&&iz<NZ) { p[phantom_idx(ix,iy,iz)]=MAT_BONE; cnt++; }
    }
    printf("Inserto osseo: %d voxel = %.1f cm3\n",cnt,cnt*VOXEL_CM*VOXEL_CM*VOXEL_CM);
}

Overwriting GPU_CUDA/phantom.cuh


In [124]:
%%writefile GPU_CUDA/output.cuh
#pragma once
#include <cstdio>
#include <cmath>
#include <fstream>
#include <algorithm>
#include "physics.cuh"

inline void compute_pdd(const double *dose, double *pdd, double *depth, int semi=8) {
    int cx=NX/2,cy=NY/2; double dmax=0;
    for(int iz=0;iz<NZ;iz++) {
        double acc=0; int n=0;
        for(int ix=cx-semi;ix<=cx+semi;ix++)
        for(int iy=cy-semi;iy<=cy+semi;iy++)
            if(ix>=0&&ix<NX&&iy>=0&&iy<NY){acc+=dose[phantom_idx(ix,iy,iz)];n++;}
        pdd[iz]=(n>0)?acc/n:0.0; depth[iz]=(iz+0.5)*VOXEL_CM;
        if(pdd[iz]>dmax) dmax=pdd[iz];
    }
    if(dmax>0) for(int iz=0;iz<NZ;iz++) pdd[iz]=pdd[iz]/dmax*100.0;
}
inline void compute_lateral_profile(const double *dose, double *prof, double *coord,
                                    double depth_cm=10.0, int semi=2) {
    int iz=std::min((int)(depth_cm/VOXEL_CM),NZ-1),cy=NY/2; double dmax=0;
    for(int ix=0;ix<NX;ix++) {
        double acc=0;int n=0;
        for(int iy=cy-semi;iy<=cy+semi;iy++)
            if(iy>=0&&iy<NY){acc+=dose[phantom_idx(ix,iy,iz)];n++;}
        prof[ix]=(n>0)?acc/n:0.0; coord[ix]=(ix+0.5)*VOXEL_CM-PHANTOM_CM/2.0;
        if(prof[ix]>dmax) dmax=prof[ix];
    }
    if(dmax>0) for(int ix=0;ix<NX;ix++) prof[ix]=prof[ix]/dmax*100.0;
}
inline void save_pdd_csv(const double *d, const double *pdd, const char *fn) {
    std::ofstream f(fn); f<<"depth_cm,dose_percent\n";
    for(int i=0;i<NZ;i++) f<<d[i]<<","<<pdd[i]<<"\n"; printf("  Salvato: %s\n",fn);
}
inline void save_profile_csv(const double *c, const double *p, const char *fn) {
    std::ofstream f(fn); f<<"position_cm,dose_percent\n";
    for(int i=0;i<NX;i++) f<<c[i]<<","<<p[i]<<"\n"; printf("  Salvato: %s\n",fn);
}
inline void save_dose_slice_csv(const double *dose, const char *fn) {
    std::ofstream f(fn); int iy=NY/2;
    for(int iz=0;iz<NZ;iz++) {
        for(int ix=0;ix<NX;ix++){f<<dose[phantom_idx(ix,iy,iz)];if(ix<NX-1)f<<",";}
        f<<"\n";
    } printf("  Salvato: %s\n",fn);
}
inline void save_dose_binary(const double *dose, const char *fn) {
    FILE *f=fopen(fn,"wb"); if(!f){printf("ERRORE: %s\n",fn);return;}
    fwrite(dose,sizeof(double),NX*NY*NZ,f); fclose(f);
    printf("  Salvato: %s\n",fn);
}
inline void print_pdd_table(const double *depth, const double *pdd, const char *label) {
    printf("\n  PDD — %s\n",label);
    printf("  %-20s  %10s  %s\n","Profondita [cm]","Dose [%]","Riferimento");
    double refs[]={1.5,3.0,5.0,10.0,15.0,20.0,25.0};
    const char *notes[]={"build-up","d_max 6MV","","D10","","D20",""};
    for(int r=0;r<7;r++){int k=(int)(refs[r]/VOXEL_CM);if(k>=0&&k<NZ) printf("  %-20.1f  %10.2f  %s\n",depth[k],pdd[k],notes[r]);}
}
inline void print_dose_stats(const double *dose, long long N, double t) {
    double dmax=0,etot=0; int hit=0;
    for(int i=0;i<NX*NY*NZ;i++){if(dose[i]>0){hit++;etot+=dose[i];if(dose[i]>dmax)dmax=dose[i];}}
    printf("\n  Statistiche simulazione:\n");
    printf("  Particelle : %lld\n",N);
    printf("  Tempo      : %.2f s\n",t);
    printf("  Throughput : %.3f MP/s\n",N/t/1e6);
    printf("  Voxel>0    : %d/%d (%.1f%%)\n",hit,NX*NY*NZ,100.0*hit/(NX*NY*NZ));
    printf("  E totale   : %.4e MeV\n",etot);
    printf("  Dose max   : %.6e\n",dmax);
}

Overwriting GPU_CUDA/output.cuh


In [125]:
%%writefile GPU_CUDA/main.cu

#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>
#include <cuda_runtime.h>

#include "physics.cuh"
#include "compton.cuh"
#include "random.cuh"
#include "phantom.cuh"
#include "output.cuh"

// ──────────────────────────────────────────────────────────────
//  Macro CUDA_CHECK
// ──────────────────────────────────────────────────────────────
#define CUDA_CHECK(call)                                                        \
    do {                                                                        \
        cudaError_t err = (call);                                               \
        if (err != cudaSuccess) {                                               \
            fprintf(stderr, "CUDA error %s:%d — %s\n",                         \
                    __FILE__, __LINE__, cudaGetErrorString(err));               \
            exit(EXIT_FAILURE);                                                 \
        }                                                                       \
    } while (0)

// ──────────────────────────────────────────────────────────────
//  Struttura particella (in local memory per thread)
// ──────────────────────────────────────────────────────────────
struct Particle {
    double x, y, z;
    double ux, uy, uz;
    double energia;
};

// ──────────────────────────────────────────────────────────────
//  Distanza al prossimo confine di voxel
// ──────────────────────────────────────────────────────────────
__device__ __forceinline__
double d_boundary_distance(double x,  double y,  double z,
                           double ux, double uy, double uz,
                           int ix, int iy, int iz)
{
    double dmin = 1.0e30;
    if (fabs(ux) > 1.0e-12) {
        double wall = (ux > 0.0) ? (ix + 1) * VOXEL_CM : ix * VOXEL_CM;
        double d    = (wall - x) / ux;
        if (d > 1.0e-10) dmin = fmin(dmin, d);
    }
    if (fabs(uy) > 1.0e-12) {
        double wall = (uy > 0.0) ? (iy + 1) * VOXEL_CM : iy * VOXEL_CM;
        double d    = (wall - y) / uy;
        if (d > 1.0e-10) dmin = fmin(dmin, d);
    }
    if (fabs(uz) > 1.0e-12) {
        double wall = (uz > 0.0) ? (iz + 1) * VOXEL_CM : iz * VOXEL_CM;
        double d    = (wall - z) / uz;
        if (d > 1.0e-10) dmin = fmin(dmin, d);
    }
    return dmin;
}

// ──────────────────────────────────────────────────────────────
//  Trasporto fotone (device)
// ──────────────────────────────────────────────────────────────
__device__
void d_transport_photon(Particle fotone_iniziale,
                        const int * __restrict__ phantom,
                        double *dose,
                        Xoshiro256 &rng)
{
    Particle stack[64];
    int stack_top = 0;
    stack[stack_top++] = fotone_iniziale;

    while (stack_top > 0) {
        Particle p = stack[--stack_top];

        for (int step = 0; step < 100000; step++) {

            // Cutoff energetico
            if (p.energia < ECUT) {
                if (inside(p.x, p.y, p.z))
                    atomicAdd(&dose[phantom_idx(vox(p.x), vox(p.y), vox(p.z))], p.energia);
                break;
            }

            if (!inside(p.x, p.y, p.z)) break;

            int ix  = vox(p.x), iy = vox(p.y), iz = vox(p.z);
            int mat = phantom[phantom_idx(ix, iy, iz)];
            double mu = d_get_mu_total(p.energia, mat);
            if (mu <= 0.0) break;

            double d_teo  = -log(rng()) / mu;
            double d_fis  = d_boundary_distance(p.x, p.y, p.z, p.ux, p.uy, p.uz, ix, iy, iz);

            if (d_teo <= d_fis) {
                p.x += p.ux * d_teo;
                p.y += p.uy * d_teo;
                p.z += p.uz * d_teo;

                if (!inside(p.x, p.y, p.z)) break;

                ix  = vox(p.x); iy = vox(p.y); iz = vox(p.z);
                mat = phantom[phantom_idx(ix, iy, iz)];
                int id_vox = phantom_idx(ix, iy, iz);

                int tipo = d_select_interaction(p.energia, mat, rng());

                // FOTOELETTRICO
                if (tipo == 0) {
                    atomicAdd(&dose[id_vox], p.energia);
                    break;
                }
                // COMPTON (Kahn)
                else if (tipo == 1) {
                    double cos_theta, e_scatter;
                    d_sample_compton(p.energia, rng, cos_theta, e_scatter);

                    double e_ceduta = p.energia - e_scatter;
                    if (e_ceduta > 0.0) atomicAdd(&dose[id_vox], e_ceduta);

                    p.energia = e_scatter;
                    double phi = 2.0 * PI * rng();
                    d_rotate_direction(p.ux, p.uy, p.uz, cos_theta, phi);

                    if (p.energia < ECUT) {
                        atomicAdd(&dose[id_vox], p.energia);
                        break;
                    }
                }
                // PRODUZIONE DI COPPIE
                else {
                    double e_cin = p.energia - 2.0 * ME_C2;
                    if (e_cin > 0.0) atomicAdd(&dose[id_vox], e_cin);

                    if (ME_C2 > ECUT && stack_top + 2 <= 62) {
                        double cos_a = 2.0 * rng() - 1.0;
                        double phi_a = 2.0 * PI * rng();
                        double sin_a = sqrt(fmax(0.0, 1.0 - cos_a * cos_a));

                        Particle g1, g2;
                        g1.x = g2.x = p.x; g1.y = g2.y = p.y; g1.z = g2.z = p.z;
                        g1.ux =  sin_a * cos(phi_a); g1.uy =  sin_a * sin(phi_a); g1.uz =  cos_a;
                        g2.ux = -g1.ux;              g2.uy = -g1.uy;              g2.uz = -g1.uz;
                        g1.energia = g2.energia = ME_C2;
                        stack[stack_top++] = g1;
                        stack[stack_top++] = g2;
                    }
                    break;
                }

            } else {
                // Avanza al bordo voxel con epsilon
                p.x += p.ux * (d_fis + 1.0e-7);
                p.y += p.uy * (d_fis + 1.0e-7);
                p.z += p.uz * (d_fis + 1.0e-7);
            }
        }
    }
}

// ──────────────────────────────────────────────────────────────
//  KERNEL PRINCIPALE
//
//  FIX DISTRIBUZIONE FOTONI (causa Test 8 e Test 9 falliti):
//  ─────────────────────────────────────────────────────────────
//  Bug: la vecchia formula
//      n_loc = base + (tid < resto ? 1 : 0)
//  assegnava sempre il fotone extra ai thread con tid < resto,
//  cioè ai thread "bassi" (tipicamente sui primi SM). Questo
//  creava un bias spaziale nella sorgente: i fotoni extra erano
//  tutti campionati da RNG con seed piccoli e consecutivi,
//  producendo una sovradensità nella regione centrale del campo
//  10x10 cm. Effetto visibile: asimmetria nel profilo laterale
//  (Test 8) e PDD leggermente alterata (Test 9 gamma index).
//
//  Fix: usare la formula continua start/end:
//      start = (N * tid)       / nthreads
//      end   = (N * (tid+1))   / nthreads
//      n_loc = end - start
//  Questa distribuisce i fotoni "extra" in modo sparso lungo
//  tutto l'intervallo di tid, senza concentrarli in testa.
//  La somma su tutti i thread è esattamente N per costruzione.
// ──────────────────────────────────────────────────────────────
__global__
void mc_kernel(const int * __restrict__ phantom,
               double *dose,
               uint64_t seed_base,
               long long num_fotoni)
{
    long long tid      = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long nthreads = (long long)gridDim.x * blockDim.x;

    // ── FIX: distribuzione continua, non base+resto ───────────
    // Garantisce sum(n_loc) == num_fotoni esattamente,
    // e distribuisce i fotoni "dispari" in modo uniforme
    // lungo tutto lo spazio dei tid anziché concentrarli
    // sui primi thread.
    long long start = (num_fotoni * tid)           / nthreads;
    long long end   = (num_fotoni * (tid + 1LL))   / nthreads;
    long long n_loc = end - start;

    if (n_loc <= 0) return;

    // Seed decorrelato: XOR con offset di Weyl, poi Xoshiro256
    // che internamente applica splitmix64 x4.
    uint64_t my_seed = seed_base ^ ((uint64_t)tid * 0x9e3779b97f4a7c15ULL);
    Xoshiro256 rng(my_seed);

    const double FIELD_HALF = 5.0;
    const double cx_src     = PHANTOM_CM / 2.0;
    const double cy_src     = PHANTOM_CM / 2.0;

    for (long long i = 0; i < n_loc; i++) {
        Particle p;
        p.x       = cx_src + (rng() * 2.0 - 1.0) * FIELD_HALF;
        p.y       = cy_src + (rng() * 2.0 - 1.0) * FIELD_HALF;
        p.z       = 1.0e-7;
        p.ux      = 0.0;
        p.uy      = 0.0;
        p.uz      = 1.0;
        p.energia = d_sample_spectrum(rng);
        d_transport_photon(p, phantom, dose, rng);
    }
}

// ──────────────────────────────────────────────────────────────
//  MAIN
// ──────────────────────────────────────────────────────────────
int main(int argc, char *argv[]) {

    long long num_fotoni   = 1000000LL;
    int       tipo_phantom = 0;
    uint64_t  seed         = 42ULL;

    if (argc > 1) num_fotoni   = std::atoll(argv[1]);
    if (argc > 2) tipo_phantom = std::atoi(argv[2]);
    if (argc > 3) seed         = (uint64_t)std::atoll(argv[3]);

    const char *phantom_label = (tipo_phantom == 0) ? "Acqua omogenea" : "Acqua + Osso";

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("  Monte Carlo per Radioterapia — GPU CUDA\n");
    printf("  GPU: %s  |  SM: %d.%d  |  VRAM: %.0f MB\n",
           prop.name, prop.major, prop.minor, prop.totalGlobalMem / 1.0e6);
    printf("\n  Parametri:\n");
    printf("  Phantom    : %dx%dx%d voxel  |  voxel %.0fmm  |  %.0f cm\n",
           NX, NY, NZ, VOXEL_CM * 10.0, PHANTOM_CM);
    printf("  Materiale  : %s\n", phantom_label);
    printf("  N fotoni   : %lld\n", num_fotoni);
    printf("  Seed       : %llu\n", (unsigned long long)seed);
    printf("  ECUT       : %.0f keV\n\n", ECUT * 1000.0);

    // ── Griglia CUDA ──────────────────────────────────────────
    // 256 thread/blocco: 8 warp, buona occupancy
    // 8 blocchi/SM: sufficiente latency hiding per kernel con
    //               molto lavoro irregolare (transport loop)
    const int THREADS_PER_BLOCK = 256;
    int n_sm     = prop.multiProcessorCount;
    int n_blocks = n_sm * 8;

    long long max_threads = (long long)n_blocks * THREADS_PER_BLOCK;
    if (max_threads > num_fotoni) {
        n_blocks = (int)((num_fotoni + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK);
        if (n_blocks < 1) n_blocks = 1;
    }

    printf("  Griglia CUDA: %d blocchi x %d thread = %lld thread totali\n",
           n_blocks, THREADS_PER_BLOCK,
           (long long)n_blocks * THREADS_PER_BLOCK);
    printf("  Fotoni/thread (media): %.1f\n\n",
           (double)num_fotoni / ((double)n_blocks * THREADS_PER_BLOCK));

    // ── Allocazione host ──────────────────────────────────────
    int    *h_phantom = new int   [NX * NY * NZ];
    double *h_dose    = new double[NX * NY * NZ]();
    double *pdd       = new double[NZ];
    double *depth_cm  = new double[NZ];
    double *profilo   = new double[NX];
    double *coord_lat = new double[NX];

    if (tipo_phantom == 0) {
        printf("  Costruzione phantom: acqua omogenea\n");
        build_phantom_water(h_phantom);
    } else {
        printf("  Costruzione phantom: acqua + osso\n");
        build_phantom_hetero(h_phantom);
    }

    upload_spectrum_to_device();

    // ── Allocazione device e trasferimento dati ───────────────
    int    *d_phantom;
    double *d_dose;
    size_t  sz_ph   = (size_t)NX * NY * NZ * sizeof(int);
    size_t  sz_dose = (size_t)NX * NY * NZ * sizeof(double);

    CUDA_CHECK(cudaMalloc(&d_phantom, sz_ph));
    CUDA_CHECK(cudaMalloc(&d_dose,    sz_dose));
    CUDA_CHECK(cudaMemcpy(d_phantom, h_phantom, sz_ph, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_dose, 0, sz_dose));

    // ── Lancio kernel ─────────────────────────────────────────
    printf("  Avvio kernel GPU...\n");
    auto t0 = std::chrono::high_resolution_clock::now();

    mc_kernel<<<n_blocks, THREADS_PER_BLOCK>>>(d_phantom, d_dose, seed, num_fotoni);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    auto t1 = std::chrono::high_resolution_clock::now();
    double tempo = std::chrono::duration<double>(t1 - t0).count();

    // ── Raccolta risultati e analisi ──────────────────────────
    CUDA_CHECK(cudaMemcpy(h_dose, d_dose, sz_dose, cudaMemcpyDeviceToHost));

    print_dose_stats(h_dose, num_fotoni, tempo);
    compute_pdd(h_dose, pdd, depth_cm);
    compute_lateral_profile(h_dose, profilo, coord_lat, 10.0);
    print_pdd_table(depth_cm, pdd, phantom_label);

    const char *pdd_file, *prof_file, *slice_file, *bin_file;
    if (tipo_phantom == 0) {
        pdd_file   = "./GPU_CUDA/pdd_water.csv";
        prof_file  = "./GPU_CUDA/profile_water.csv";
        slice_file = "./GPU_CUDA/dose_slice_water.csv";
        bin_file   = "./GPU_CUDA/dose_water.bin";
    } else {
        pdd_file   = "./GPU_CUDA/pdd_hetero.csv";
        prof_file  = "./GPU_CUDA/profile_hetero.csv";
        slice_file = "./GPU_CUDA/dose_slice_hetero.csv";
        bin_file   = "./GPU_CUDA/dose_hetero.bin";
    }
    save_pdd_csv(depth_cm, pdd, pdd_file);
    save_profile_csv(coord_lat, profilo, prof_file);
    save_dose_slice_csv(h_dose, slice_file);
    save_dose_binary(h_dose, bin_file);

    // ── Cleanup ───────────────────────────────────────────────
    cudaFree(d_phantom);
    cudaFree(d_dose);
    delete[] h_phantom;
    delete[] h_dose;
    delete[] pdd;
    delete[] depth_cm;
    delete[] profilo;
    delete[] coord_lat;

    printf("\n  Simulazione GPU completata.\n");
    return 0;
}


Overwriting GPU_CUDA/main.cu


In [14]:
!nvidia-smi

Wed Apr 22 17:05:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [126]:
# ── Individua automaticamente la Compute Capability della GPU ─
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                        capture_output=True, text=True)
cc = result.stdout.strip().replace('.', '')
print(f'Compute Capability rilevata: {result.stdout.strip()}  →  -arch=sm_{cc}')

Compute Capability rilevata: 7.5  →  -arch=sm_75


In [132]:
# ── Compilazione ──────────────────────────────────────────────
import subprocess, os
result = subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],
                        capture_output=True,text=True)
cc = result.stdout.strip().replace('.', '')

cmd = ["nvcc", "-O3", f"-arch=sm_{cc}", "-std=c++17", "-o", "GPU_CUDA//mc_gpu", "GPU_CUDA//main.cu"]

print(f"Eseguo: {' '.join(cmd)}")

cp = subprocess.run(cmd, capture_output=True, text=True)

if cp.returncode == 0:
    print('\n✅ Compilazione OK!')
else:
    print('\n❌ Errore di compilazione:')
    print(cp.stderr) # Questo ti mostrerà l'errore esatto di NVCC

Eseguo: nvcc -O3 -arch=sm_75 -std=c++17 -o GPU_CUDA//mc_gpu GPU_CUDA//main.cu

❌ Errore di compilazione:
GPU_CUDA//physics.cuh(149): error: identifier "get_mu_pe" is undefined
  inline double get_mu_total(double e, int mat) {return (get_mu_pe(e, mat) + get_mu_compton(e, mat) + get_mu_pair(e, mat));}
                                                         ^

GPU_CUDA//physics.cuh(149): error: identifier "get_mu_compton" is undefined
  inline double get_mu_total(double e, int mat) {return (get_mu_pe(e, mat) + get_mu_compton(e, mat) + get_mu_pair(e, mat));}
                                                                             ^

GPU_CUDA//physics.cuh(149): error: identifier "get_mu_pair" is undefined
  inline double get_mu_total(double e, int mat) {return (get_mu_pe(e, mat) + get_mu_compton(e, mat) + get_mu_pair(e, mat));}
                                                                                                      ^

3 errors detected in the compilation of "GPU_CUDA//main

In [17]:
print(numero_fotoni)

100000000


In [128]:
# ── Esecuzione: phantom acqua, 1M fotoni ─────────────────────
# argv: N_fotoni  tipo_phantom(0=acqua,1=eterogeneo)  seed
!./GPU_CUDA/mc_gpu $numero_fotoni 0 42

  Monte Carlo per Radioterapia — GPU CUDA
  GPU: Tesla T4  |  SM: 7.5  |  VRAM: 15637 MB

  Parametri:
  Phantom    : 100x100x100 voxel  |  voxel 3mm  |  30 cm
  Materiale  : Acqua omogenea
  N fotoni   : 10000000
  Seed       : 42
  ECUT       : 10 keV

  Griglia CUDA: 320 blocchi x 256 thread = 81920 thread totali
  Fotoni/thread (media): 122.1

  Costruzione phantom: acqua omogenea
  Avvio kernel GPU...

  Statistiche simulazione:
  Particelle : 10000000
  Tempo      : 6.25 s
  Throughput : 1.600 MP/s
  Voxel>0    : 998594/1000000 (99.9%)
  E totale   : 1.0503e+07 MeV
  Dose max   : 1.885537e+02

  PDD — Acqua omogenea
  Profondita [cm]         Dose [%]  Riferimento
  1.6                        97.32  build-up
  3.1                        92.22  d_max 6MV
  5.0                        89.33  
  10.0                       74.64  D10
  15.1                       61.39  
  19.9                       50.41  D20
  25.1                       39.86  
  Salvato: ./GPU_CUDA/pdd_water.csv
  Sa

In [113]:
# ── Esecuzione: phantom eterogeneo, 2M fotoni ────────────────
!./GPU_CUDA/mc_gpu $numero_fotoni 1 42

  Monte Carlo per Radioterapia — GPU CUDA
  GPU: Tesla T4  |  SM: 7.5  |  VRAM: 15637 MB

  Parametri:
  Phantom    : 100x100x100 voxel  |  voxel 3mm  |  30 cm
  Materiale  : Acqua + Osso
  N fotoni   : 10000000
  Seed       : 42
  ECUT       : 10 keV

  Griglia CUDA: 320 blocchi x 256 thread = 81920 thread totali
  Fotoni/thread (media): 122.1

  Costruzione phantom: acqua + osso
Inserto osseo: 4096 voxel = 110.6 cm3
  Avvio kernel GPU...

  Statistiche simulazione:
  Particelle : 10000000
  Tempo      : 6.60 s
  Throughput : 1.514 MP/s
  Voxel>0    : 998998/1000000 (99.9%)
  E totale   : 1.0821e+07 MeV
  Dose max   : 2.414792e+02

  PDD — Acqua + Osso
  Profondita [cm]         Dose [%]  Riferimento
  1.6                        75.84  build-up
  3.1                        73.08  d_max 6MV
  5.0                        69.17  
  10.0                       58.80  D10
  15.1                       85.93  
  19.9                       34.28  D20
  25.1                       26.96  
  Salvat

In [114]:
# ── Visualizzazione PDD ───────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

pdd = pd.read_csv('./GPU_CUDA/pdd_water.csv')

plt.figure(figsize=(8, 5))
plt.plot(pdd['depth_cm'], pdd['dose_percent'], 'b-', linewidth=2)
plt.xlabel('Profondità [cm]')
plt.ylabel('Dose [%]')
plt.title('PDD — Fascio 6MV in acqua (GPU CUDA)')
plt.grid(True, alpha=0.3)
plt.axvline(x=3.0, color='r', linestyle='--', alpha=0.5, label='d_max 6MV (~3cm)')
plt.legend()
plt.tight_layout()
plt.savefig('./GPU_CUDA/pdd_plot.png', dpi=150)
plt.show()
print('Plot salvato.')

Plot salvato.


In [115]:
# ── Heatmap dose (slice centrale XZ) ─────────────────────────
import numpy as np
import matplotlib.pyplot as plt

dose_slice = pd.read_csv('./GPU_CUDA/dose_slice_water.csv', header=None).values

plt.figure(figsize=(7, 7))
plt.imshow(dose_slice, aspect='auto', cmap='hot',
           extent=[0, 30, 30, 0])
plt.colorbar(label='Dose (u.a.)')
plt.xlabel('X [cm]')
plt.ylabel('Profondità Z [cm]')
plt.title('Distribuzione di dose — slice XZ centrale')
plt.tight_layout()
plt.savefig('./GPU_CUDA/dose_heatmap.png', dpi=150)
plt.show()

### Nuova sezione

In [39]:
%%writefile mc_gpu.cu

// ============================================================
//  mc_gpu.cu  —  Monte Carlo Radioterapia — GPU CUDA
//  FILE MONOLITICO: nessun header esterno (.cuh)
//  Tutto il codice è in questo unico file.
//
//  COMPILAZIONE:
//    !nvcc -O3 -arch=sm_75 -std=c++17 -o mc_gpu mc_gpu.cu
//  (rilevare CC: !nvidia-smi --query-gpu=compute_cap --format=csv,noheader)
//
//  ESECUZIONE:
//    !./mc_gpu 100000000 0 42   <- 100M fotoni, acqua
//    !./mc_gpu 100000000 1 42   <- 100M fotoni, eterogeneo
// ============================================================

#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>
#include <fstream>
#include <algorithm>
#include <cstring>
#include <cstdint>
#include <cuda_runtime.h>

// ══════════════════════════════════════════════════════════════
//  SEZIONE 1: COSTANTI FISICHE E GEOMETRIA
// ══════════════════════════════════════════════════════════════

static const double ME_C2     = 0.51099895;
static const double PI        = 3.14159265358979323846;
static const double ECUT      = 0.010;   // MeV (10 keV)
static const double PCUT      = 0.100;   // MeV

static const int    NX = 100, NY = 100, NZ = 100;
static const double VOXEL_CM   = 0.30;
static const double PHANTOM_CM = NX * VOXEL_CM;  // 30.0 cm

#define MAT_WATER 0
#define MAT_BONE  1
#define MAT_LUNG  2
#define MAT_AIR   3
#define N_MAT     4
static const int N_ENERGY = 28;

// ── Densità ──────────────────────────────────────────────────
static const double DENSITY[N_MAT] = { 1.000, 1.850, 0.260, 0.001205 };
__device__ __constant__ double d_DENSITY[N_MAT];

// ── Griglia energetica ────────────────────────────────────────
static const double ENERGY_GRID[28] = {
    0.010, 0.015, 0.020, 0.030, 0.040, 0.050, 0.060, 0.080, 0.100,
    0.150, 0.200, 0.300, 0.400, 0.500, 0.600, 0.800, 1.000, 1.250,
    1.500, 2.000, 3.000, 4.000, 5.000, 6.000, 8.000, 10.000, 15.000, 20.000
};
__device__ __constant__ double d_ENERGY_GRID[28];

// ── Tabelle μ/ρ [cm^2/g] ─────────────────────────────────────
static const double MU_TOTAL[N_MAT][28] = {
    { 5.329, 1.673, 0.8096, 0.3756, 0.2683, 0.2269, 0.2059, 0.1837, 0.1707,
      0.1505, 0.1370, 0.1186, 0.1061, 0.09687, 0.09007, 0.07865, 0.07072, 0.06323,
      0.05754, 0.04942, 0.03969, 0.03403, 0.03031, 0.02770, 0.02429, 0.02219, 0.01941, 0.01813 },
    { 19.89, 7.131, 3.085, 1.012, 0.5475, 0.3941, 0.3178, 0.2595, 0.2368,
      0.1958, 0.1698, 0.1393, 0.1222, 0.1107, 0.1018, 0.08795, 0.07838, 0.06945,
      0.06283, 0.05351, 0.04257, 0.03624, 0.03209, 0.02913, 0.02536, 0.02296, 0.01978, 0.01832 },
    { 5.208, 1.638, 0.7933, 0.3681, 0.2629, 0.2224, 0.2018, 0.1800, 0.1673,
      0.1475, 0.1342, 0.1162, 0.1040, 0.09493, 0.08827, 0.07708, 0.06930, 0.06196,
      0.05639, 0.04843, 0.03889, 0.03335, 0.02970, 0.02715, 0.02380, 0.02175, 0.01902, 0.01776 },
    { 36.01, 12.28, 5.279, 1.625, 0.6918, 0.3954, 0.2885, 0.2085, 0.1875,
      0.1504, 0.1340, 0.1123, 0.09921, 0.09076, 0.08414, 0.07285, 0.06529, 0.05817,
      0.05298, 0.04534, 0.03597, 0.03054, 0.02699, 0.02453, 0.02131, 0.01931, 0.01673, 0.01551 }
};
static const double MU_PE[N_MAT][28] = {
    { 4.944, 1.374, 0.5195, 0.1036, 0.02407, 0.005800, 0.001334, 5.510e-5, 3.998e-5,
      2.799e-6, 2.200e-7, 1.400e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    { 19.35, 6.833, 2.818, 0.7469, 0.2837, 0.1152, 0.04660, 0.008680, 0.001900,
      1.800e-4, 2.000e-5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    { 4.845, 1.346, 0.5091, 0.1015, 0.02359, 0.005684, 0.001307, 5.400e-5, 3.918e-5,
      2.743e-6, 2.156e-7, 1.372e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    { 35.52, 11.99, 5.012, 1.379, 0.4529, 0.1581, 0.05757, 0.008251, 0.001581,
      8.208e-5, 7.636e-6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 }
};
static const double MU_COMPTON[N_MAT][28] = {
    { 0.3854, 0.2988, 0.2672, 0.2651, 0.2595, 0.2476, 0.2329, 0.1984, 0.1661,
      0.1505, 0.1370, 0.1186, 0.1061, 0.09687, 0.09007, 0.07865, 0.07072, 0.06323,
      0.05754, 0.04942, 0.03969, 0.03403, 0.03031, 0.02770, 0.02429, 0.02219, 0.01878, 0.01719 },
    { 0.4869, 0.2684, 0.2503, 0.2465, 0.2429, 0.2310, 0.2172, 0.1848, 0.1548,
      0.1400, 0.1275, 0.1103, 0.09870, 0.09010, 0.08377, 0.07313, 0.06575, 0.05862,
      0.05338, 0.04579, 0.03667, 0.03133, 0.02784, 0.02539, 0.02217, 0.02016, 0.01702, 0.01552 },
    { 0.3777, 0.2928, 0.2619, 0.2598, 0.2543, 0.2426, 0.2282, 0.1944, 0.1628,
      0.1475, 0.1342, 0.1162, 0.1040, 0.09493, 0.08827, 0.07708, 0.06930, 0.06196,
      0.05639, 0.04843, 0.03889, 0.03335, 0.02970, 0.02715, 0.02380, 0.02175, 0.01840, 0.01684 },
    { 0.3779, 0.2933, 0.2624, 0.2602, 0.2547, 0.2430, 0.2285, 0.1946, 0.1630,
      0.1477, 0.1344, 0.1163, 0.1041, 0.09516, 0.08844, 0.07723, 0.06942, 0.06207,
      0.05649, 0.04852, 0.03894, 0.03339, 0.02973, 0.02718, 0.02382, 0.02177, 0.01843, 0.01686 }
};
static const double MU_PAIR[N_MAT][28] = {
    { 0,0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 0.000630, 0.000940 },
    { 0,0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 0.002760, 0.002800 },
    { 0,0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 0.000617, 0.000921 },
    { 0,0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 0.000589, 0.000879 }
};

// Tabelle device in constant memory (una copia sola, caricata da main)
__device__ __constant__ double d_MU_TOTAL  [N_MAT][28];
__device__ __constant__ double d_MU_PE     [N_MAT][28];
__device__ __constant__ double d_MU_COMPTON[N_MAT][28];

// ══════════════════════════════════════════════════════════════
//  SEZIONE 2: SPETTRO 6MV
// ══════════════════════════════════════════════════════════════

static const int SPECTRUM_BINS = 24;
static const double SPECTRUM_E[24] = {
    0.25,0.50,0.75,1.00,1.25,1.50,1.75,2.00,
    2.25,2.50,2.75,3.00,3.25,3.50,3.75,4.00,
    4.25,4.50,4.75,5.00,5.25,5.50,5.75,6.00
};
static const double SPECTRUM_FLUENCE[24] = {
    0.0243,0.0676,0.0862,0.0929,0.0919,0.0868,0.0794,0.0712,
    0.0628,0.0548,0.0471,0.0399,0.0334,0.0276,0.0224,0.0178,
    0.0138,0.0104,0.0075,0.0052,0.0034,0.0020,0.0010,0.0004
};
__device__ __constant__ double d_spectrum_cdf[24];
__device__ __constant__ double d_spectrum_e  [24];
__device__ __constant__ double d_spectrum_dE;

// ══════════════════════════════════════════════════════════════
//  SEZIONE 3: RNG XOSHIRO256**  (host + device)
// ══════════════════════════════════════════════════════════════

struct Xoshiro256 {
    uint64_t s[4];

    __host__ __device__
    static uint64_t splitmix64(uint64_t &x) {
        x += 0x9e3779b97f4a7c15ULL;
        uint64_t z = x;
        z = (z ^ (z >> 30)) * 0xbf58476d1ce4e5b9ULL;
        z = (z ^ (z >> 27)) * 0x94d049bb133111ebULL;
        return z ^ (z >> 31);
    }

    __host__ __device__
    explicit Xoshiro256(uint64_t seed) {
        s[0] = splitmix64(seed);
        s[1] = splitmix64(seed);
        s[2] = splitmix64(seed);
        s[3] = splitmix64(seed);
    }

    __host__ __device__ Xoshiro256() {}

    __host__ __device__
    static uint64_t rotl(uint64_t x, int k) { return (x << k) | (x >> (64 - k)); }

    __host__ __device__
    uint64_t next() {
        const uint64_t result = rotl(s[1] * 5, 7) * 9;
        const uint64_t t = s[1] << 17;
        s[2] ^= s[0]; s[3] ^= s[1]; s[1] ^= s[2]; s[0] ^= s[3];
        s[2] ^= t;
        s[3] = rotl(s[3], 45);
        return result;
    }

    __host__ __device__
    double operator()() {
        double r;
        do { r = (double)(next() >> 11) * (1.0 / (double)(1ULL << 53)); } while (r <= 0.0);
        return r;
    }
};

// ══════════════════════════════════════════════════════════════
//  SEZIONE 4: FUNZIONI HOST — GEOMETRIA E FISICA (identiche al CPU)
// ══════════════════════════════════════════════════════════════

// Indice lineare phantom (row-major): ix varia più veloce
__host__ __device__ __forceinline__
int phantom_idx(int ix, int iy, int iz) { return ix + NX * (iy + NY * iz); }

__host__ __device__ __forceinline__
bool inside(double x, double y, double z) {
    return x >= 0.0 && x < PHANTOM_CM &&
           y >= 0.0 && y < PHANTOM_CM &&
           z >= 0.0 && z < PHANTOM_CM;
}

__host__ __device__ __forceinline__
int vox(double coord) {
    int v = (int)(coord / VOXEL_CM);
    if (v < 0)   v = 0;
    if (v >= NX) v = NX - 1;
    return v;
}

// Interpolazione lineare su griglia energetica (host)
inline double h_interp_mu(double e, const double tab[28]) {
    if (e <= ENERGY_GRID[0])    return tab[0];
    if (e >= ENERGY_GRID[27])   return tab[27];
    int lo = 0, hi = 27;
    while (hi - lo > 1) { int m=(lo+hi)/2; if(ENERGY_GRID[m]<=e) lo=m; else hi=m; }
    double t = (e - ENERGY_GRID[lo]) / (ENERGY_GRID[hi] - ENERGY_GRID[lo]);
    return tab[lo]*(1.0-t) + tab[hi]*t;
}
inline double get_mu_total(double e, int mat) { return h_interp_mu(e, MU_TOTAL[mat]) * DENSITY[mat]; }

// ══════════════════════════════════════════════════════════════
//  SEZIONE 5: FUNZIONI DEVICE — FISICA GPU
// ══════════════════════════════════════════════════════════════

// Interpolazione device (legge da constant memory)
__device__ __forceinline__
double d_interp_mu(double e, const double* __restrict__ tab) {
    if (e <= d_ENERGY_GRID[0])  return tab[0];
    if (e >= d_ENERGY_GRID[27]) return tab[27];
    int lo = 0, hi = 27;
    while (hi - lo > 1) { int m=(lo+hi)>>1; if(d_ENERGY_GRID[m]<=e) lo=m; else hi=m; }
    double t = (e - d_ENERGY_GRID[lo]) / (d_ENERGY_GRID[hi] - d_ENERGY_GRID[lo]);
    return tab[lo]*(1.0-t) + tab[hi]*t;
}
__device__ __forceinline__ double d_get_mu_total(double e, int mat)   { return d_interp_mu(e, d_MU_TOTAL[mat])   * d_DENSITY[mat]; }
__device__ __forceinline__ double d_get_mu_pe(double e, int mat)      { return d_interp_mu(e, d_MU_PE[mat])      * d_DENSITY[mat]; }
__device__ __forceinline__ double d_get_mu_compton(double e, int mat) { return d_interp_mu(e, d_MU_COMPTON[mat]) * d_DENSITY[mat]; }

__device__ __forceinline__
int d_select_interaction(double e, int mat, double xi) {
    double mu = d_get_mu_total(e, mat);
    if (mu <= 0.0) return 1;
    double p_pe = d_get_mu_pe(e, mat) / mu;
    double p_co = d_get_mu_compton(e, mat) / mu;
    if (xi < p_pe)        return 0;
    if (xi < p_pe + p_co) return 1;
    return 2;
}

// Campionamento spettro 6MV (device, da constant memory)
__device__ __forceinline__
double d_sample_spectrum(Xoshiro256 &rng) {
    double xi = rng();
    int lo = 0, hi = SPECTRUM_BINS - 1;
    while (lo < hi) { int m=(lo+hi)>>1; if(d_spectrum_cdf[m]<xi) lo=m+1; else hi=m; }
    double e = d_spectrum_e[lo] + (rng() - 0.5) * d_spectrum_dE;
    if (e < 0.01) e = 0.01;
    if (e > 6.00) e = 6.00;
    return e;
}

// Campionamento Compton — metodo di Kahn (identico al CPU)
__device__ __forceinline__
void d_kahn_compton(double energia_mev, double xi_1, double xi_2, double xi_3,
                    double &cos_theta, double &energia_scatter) {
    double alpha   = energia_mev / ME_C2;
    double tau_min = 1.0 / (1.0 + 2.0 * alpha);
    double area_1  = log(1.0 / tau_min);
    double area_2  = (1.0 - tau_min * tau_min) * 0.5;
    double tau;

    if (xi_1 * (area_1 + area_2) < area_1) {
        tau = pow(tau_min, 1.0 - xi_2);
    } else {
        double tmin2 = tau_min * tau_min;
        tau = sqrt(fmax(tmin2 + xi_2 * (1.0 - tmin2), 1e-30));
    }
    tau = fmin(fmax(tau, tau_min), 1.0);

    cos_theta = 1.0 - (1.0 - tau) / (alpha * tau);
    cos_theta = fmin(fmax(cos_theta, -1.0), 1.0);
    energia_scatter = tau * energia_mev;

    double sin2  = fmax(0.0, 1.0 - cos_theta * cos_theta);
    double corr  = (tau * sin2) / (1.0 + tau * tau);
    double p_acc = fmax(0.0, fmin(1.0, 1.0 - corr));
    if (xi_3 > p_acc) cos_theta = 2.0;  // segnale di rejection
}

__device__ __forceinline__
void d_sample_compton(double energia_mev, Xoshiro256 &rng,
                      double &cos_theta, double &energia_scatter) {
    while (true) {
        d_kahn_compton(energia_mev, rng(), rng(), rng(), cos_theta, energia_scatter);
        if (cos_theta <= 1.0) return;
    }
}

// Rotazione direzione dopo scattering (identica al CPU)
__device__ __forceinline__
void d_rotate_direction(double &ux, double &uy, double &uz,
                        double cos_theta, double phi) {
    double sin_theta = sqrt(fmax(0.0, 1.0 - cos_theta * cos_theta));
    double cos_phi   = cos(phi);
    double sin_phi   = sin(phi);
    double ux_new, uy_new, uz_new;

    if (fabs(uz) > 0.99999) {
        double segno = (uz > 0.0) ? 1.0 : -1.0;
        ux_new = sin_theta * cos_phi;
        uy_new = sin_theta * sin_phi * segno;
        uz_new = cos_theta * segno;
    } else {
        double proj_xy = sqrt(1.0 - uz * uz);
        ux_new = sin_theta * (ux * uz * cos_phi - uy * sin_phi) / proj_xy + ux * cos_theta;
        uy_new = sin_theta * (uy * uz * cos_phi + ux * sin_phi) / proj_xy + uy * cos_theta;
        uz_new = -sin_theta * cos_phi * proj_xy                            + uz * cos_theta;
    }
    double norm = sqrt(ux_new*ux_new + uy_new*uy_new + uz_new*uz_new);
    if (norm > 0.0) { ux = ux_new/norm; uy = uy_new/norm; uz = uz_new/norm; }
}

// Distanza al prossimo confine di voxel (identica al CPU)
__device__ __forceinline__
double d_boundary_distance(double x, double y, double z,
                           double ux, double uy, double uz,
                           int ix, int iy, int iz) {
    double dmin = 1.0e30;
    if (fabs(ux) > 1.0e-12) { double w=(ux>0)?(ix+1)*VOXEL_CM:ix*VOXEL_CM; double d=(w-x)/ux; if(d>1.0e-10) dmin=fmin(dmin,d); }
    if (fabs(uy) > 1.0e-12) { double w=(uy>0)?(iy+1)*VOXEL_CM:iy*VOXEL_CM; double d=(w-y)/uy; if(d>1.0e-10) dmin=fmin(dmin,d); }
    if (fabs(uz) > 1.0e-12) { double w=(uz>0)?(iz+1)*VOXEL_CM:iz*VOXEL_CM; double d=(w-z)/uz; if(d>1.0e-10) dmin=fmin(dmin,d); }
    return dmin;
}

// ══════════════════════════════════════════════════════════════
//  SEZIONE 6: STRUTTURA PARTICELLA E TRASPORTO
// ══════════════════════════════════════════════════════════════

struct Particle {
    double x, y, z;
    double ux, uy, uz;
    double energia;
};

// Trasporto fotone — logica identica al CPU sequenziale
// Unica differenza: dose[] aggiornata con atomicAdd invece di +=
__device__
void d_transport_photon(Particle fotone_iniziale,
                        const int * __restrict__ phantom,
                        double *dose,
                        Xoshiro256 &rng)
{
    Particle stack[64];
    int num_particelle_stack = 0;
    stack[num_particelle_stack++] = fotone_iniziale;

    while (num_particelle_stack > 0) {
        Particle particella_corrente = stack[--num_particelle_stack];

        for (int step = 0; step < 100000; step++) {

            // Cutoff energetico
            if (particella_corrente.energia < ECUT) {
                if (inside(particella_corrente.x, particella_corrente.y, particella_corrente.z)) {
                    int ix = vox(particella_corrente.x);
                    int iy = vox(particella_corrente.y);
                    int iz = vox(particella_corrente.z);
                    atomicAdd(&dose[phantom_idx(ix, iy, iz)], particella_corrente.energia);
                }
                break;
            }

            // Verifica bounds
            if (!inside(particella_corrente.x, particella_corrente.y, particella_corrente.z))
                break;

            int ix = vox(particella_corrente.x);
            int iy = vox(particella_corrente.y);
            int iz = vox(particella_corrente.z);
            int materiale = phantom[phantom_idx(ix, iy, iz)];
            double mu = d_get_mu_total(particella_corrente.energia, materiale);
            if (mu <= 0.0) break;

            // Campiona cammino libero medio
            double xi = rng();
            double distanza_teorica = -log(xi) / mu;
            double distanza_fisica  = d_boundary_distance(
                particella_corrente.x, particella_corrente.y, particella_corrente.z,
                particella_corrente.ux, particella_corrente.uy, particella_corrente.uz,
                ix, iy, iz);

            if (distanza_teorica <= distanza_fisica) {
                // Sposta al punto di interazione
                particella_corrente.x += particella_corrente.ux * distanza_teorica;
                particella_corrente.y += particella_corrente.uy * distanza_teorica;
                particella_corrente.z += particella_corrente.uz * distanza_teorica;

                if (!inside(particella_corrente.x, particella_corrente.y, particella_corrente.z))
                    break;

                // Ricalcola voxel
                ix = vox(particella_corrente.x);
                iy = vox(particella_corrente.y);
                iz = vox(particella_corrente.z);
                materiale = phantom[phantom_idx(ix, iy, iz)];
                int id_voxel = phantom_idx(ix, iy, iz);
                int tipo_interazione = d_select_interaction(particella_corrente.energia, materiale, rng());

                // FOTOELETTRICO
                if (tipo_interazione == 0) {
                    atomicAdd(&dose[id_voxel], particella_corrente.energia);
                    break;
                }
                // COMPTON (Kahn)
                else if (tipo_interazione == 1) {
                    double cos_theta;
                    double energia_scatter;
                    d_sample_compton(particella_corrente.energia, rng, cos_theta, energia_scatter);

                    double energia_ceduta = particella_corrente.energia - energia_scatter;
                    if (energia_ceduta > 0.0)
                        atomicAdd(&dose[id_voxel], energia_ceduta);

                    particella_corrente.energia = energia_scatter;
                    double phi = 2.0 * PI * rng();
                    d_rotate_direction(particella_corrente.ux, particella_corrente.uy, particella_corrente.uz, cos_theta, phi);

                    if (particella_corrente.energia < ECUT) {
                        atomicAdd(&dose[id_voxel], particella_corrente.energia);
                        break;
                    }
                }
                // PRODUZIONE DI COPPIE
                else {
                    double energia_cinetica_residua = particella_corrente.energia - 2.0 * ME_C2;
                    if (energia_cinetica_residua > 0.0)
                        atomicAdd(&dose[id_voxel], energia_cinetica_residua);

                    if (ME_C2 > ECUT && num_particelle_stack + 2 <= 62) {
                        double cos_theta = 2.0 * rng() - 1.0;
                        double phi_a     = 2.0 * PI * rng();
                        double sen_theta = sqrt(fmax(0.0, 1.0 - cos_theta * cos_theta));

                        Particle fotone_secondario_1, fotone_secondario_2;
                        fotone_secondario_1.x = fotone_secondario_2.x = particella_corrente.x;
                        fotone_secondario_1.y = fotone_secondario_2.y = particella_corrente.y;
                        fotone_secondario_1.z = fotone_secondario_2.z = particella_corrente.z;
                        fotone_secondario_1.ux =  sen_theta * cos(phi_a);
                        fotone_secondario_1.uy =  sen_theta * sin(phi_a);
                        fotone_secondario_1.uz =  cos_theta;
                        fotone_secondario_2.ux = -fotone_secondario_1.ux;
                        fotone_secondario_2.uy = -fotone_secondario_1.uy;
                        fotone_secondario_2.uz = -fotone_secondario_1.uz;
                        fotone_secondario_1.energia = fotone_secondario_2.energia = ME_C2;

                        stack[num_particelle_stack++] = fotone_secondario_1;
                        stack[num_particelle_stack++] = fotone_secondario_2;
                    }
                    break;
                }

            } else {
                double eps = 1.0e-7;
                particella_corrente.x += particella_corrente.ux * (distanza_fisica + eps);
                particella_corrente.y += particella_corrente.uy * (distanza_fisica + eps);
                particella_corrente.z += particella_corrente.uz * (distanza_fisica + eps);
            }
        }
    }
}

// ══════════════════════════════════════════════════════════════
//  SEZIONE 7: KERNEL PRINCIPALE
// ══════════════════════════════════════════════════════════════

__global__
void mc_kernel(const int * __restrict__ phantom,
               double *dose,
               uint64_t seed_base,
               long long num_fotoni)
{
    long long tid      = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long nthreads = (long long)gridDim.x * blockDim.x;

    long long base  = num_fotoni / nthreads;
    long long resto = num_fotoni % nthreads;
    long long n_loc = base + (tid < resto ? 1LL : 0LL);

    if (n_loc == 0) return;

    // Seed unico per thread — XOR con sequenza di Weyl
    // Il costruttore Xoshiro256 chiama splitmix64 x4 internamente.
    // NON pre-applicare splitmix64 qui: causerebbe correlazione tra thread.
    uint64_t my_seed = seed_base ^ ((uint64_t)tid * 0x9e3779b97f4a7c15ULL);
    Xoshiro256 rng(my_seed);

    // Campionamento sorgente — identico al CPU
    static const double FIELD_HALF = 5.0;
    const double cx = PHANTOM_CM / 2.0;
    const double cy = PHANTOM_CM / 2.0;

    for (long long i = 0; i < n_loc; i++) {
        Particle p;
        p.x      = cx + (rng() * 2.0 - 1.0) * FIELD_HALF;
        p.y      = cy + (rng() * 2.0 - 1.0) * FIELD_HALF;
        p.z      = 1.0e-7;
        p.ux     = 0.0;
        p.uy     = 0.0;
        p.uz     = 1.0;
        p.energia = d_sample_spectrum(rng);
        d_transport_photon(p, phantom, dose, rng);
    }
}

// ══════════════════════════════════════════════════════════════
//  SEZIONE 8: OUTPUT (identico al CPU — gira su host)
// ══════════════════════════════════════════════════════════════

inline void compute_pdd(const double *dose, double *pdd, double *profondita, int semiampiezza = 8) {
    int cx = NX / 2, cy = NY / 2;
    double max_dose = 0.0;
    for (int iz = 0; iz < NZ; iz++) {
        double acc = 0.0; int n = 0;
        for (int ix = cx - semiampiezza; ix <= cx + semiampiezza; ix++)
        for (int iy = cy - semiampiezza; iy <= cy + semiampiezza; iy++) {
            if (ix >= 0 && ix < NX && iy >= 0 && iy < NY) { acc += dose[phantom_idx(ix, iy, iz)]; n++; }
        }
        pdd[iz] = (n > 0) ? acc / n : 0.0;
        profondita[iz] = (iz + 0.5) * VOXEL_CM;
        if (pdd[iz] > max_dose) max_dose = pdd[iz];
    }
    if (max_dose > 0.0)
        for (int iz = 0; iz < NZ; iz++) pdd[iz] = pdd[iz] / max_dose * 100.0;
}

inline void compute_lateral_profile(const double *dose, double *profilo, double *coordinate,
                                    double profondita_scelta = 10.0, int semiampiezza_media = 2) {
    int iz = std::min((int)(profondita_scelta / VOXEL_CM), NZ - 1);
    int centro_asse_y = NY / 2;
    double dose_massima_profilo = 0.0;
    for (int ix = 0; ix < NX; ix++) {
        double somma = 0.0; int n = 0;
        for (int iy = centro_asse_y - semiampiezza_media; iy <= centro_asse_y + semiampiezza_media; iy++) {
            if (iy >= 0 && iy < NY) { somma += dose[phantom_idx(ix, iy, iz)]; n++; }
        }
        profilo[ix]    = (n > 0) ? somma / n : 0.0;
        coordinate[ix] = (ix + 0.5) * VOXEL_CM - PHANTOM_CM / 2.0;
        if (profilo[ix] > dose_massima_profilo) dose_massima_profilo = profilo[ix];
    }
    if (dose_massima_profilo > 0.0)
        for (int ix = 0; ix < NX; ix++) profilo[ix] = profilo[ix] / dose_massima_profilo * 100.0;
}

inline void save_pdd_csv(const double *profondita, const double *pdd, const char *fn) {
    std::ofstream f(fn); f << "depth_cm,dose_percent\n";
    for (int iz = 0; iz < NZ; iz++) f << profondita[iz] << "," << pdd[iz] << "\n";
    printf("  Salvato: %s\n", fn);
}
inline void save_profile_csv(const double *coord, const double *profilo, const char *fn) {
    std::ofstream f(fn); f << "position_cm,dose_percent\n";
    for (int ix = 0; ix < NX; ix++) f << coord[ix] << "," << profilo[ix] << "\n";
    printf("  Salvato: %s\n", fn);
}
inline void save_dose_slice_csv(const double *dose, const char *fn) {
    std::ofstream f(fn); int iy = NY / 2;
    for (int iz = 0; iz < NZ; iz++) {
        for (int ix = 0; ix < NX; ix++) { f << dose[phantom_idx(ix, iy, iz)]; if (ix < NX-1) f << ","; }
        f << "\n";
    }
    printf("  Salvato: %s\n", fn);
}
inline void save_dose_binary(const double *dose, const char *fn) {
    FILE *f = fopen(fn, "wb");
    if (!f) { printf("ERRORE: impossibile aprire %s\n", fn); return; }
    fwrite(dose, sizeof(double), NX * NY * NZ, f);
    fclose(f);
    printf("  Salvato: %s  (%d float64)\n", fn, NX * NY * NZ);
}
inline void print_pdd_table(const double *profondita, const double *pdd, const char *label) {
    printf("\n  PDD — %s\n", label);
    printf("  %-20s  %10s  %s\n", "Profondita [cm]", "Dose [%]", "Riferimento");
    double refs[] = {1.5,3.0,5.0,10.0,15.0,20.0,25.0};
    const char *notes[] = {"build-up","d_max 6MV","","D10","","D20",""};
    for (int r = 0; r < 7; r++) {
        int k = (int)(refs[r] / VOXEL_CM);
        if (k >= 0 && k < NZ) printf("  %-20.1f  %10.2f  %s\n", profondita[k], pdd[k], notes[r]);
    }
}
inline void print_dose_stats(const double *dose, long long N, double t) {
    double dmax = 0.0, etot = 0.0; int hit = 0;
    for (int i = 0; i < NX*NY*NZ; i++) {
        if (dose[i] > 0.0) { hit++; etot += dose[i]; if (dose[i] > dmax) dmax = dose[i]; }
    }
    printf("\n  Statistiche simulazione:\n");
    printf("  Particelle simulate : %lld\n", N);
    printf("  Tempo totale        : %.2f s\n", t);
    printf("  Throughput          : %.3f MP/s\n", N / t / 1.0e6);
    printf("  Tempo/particella    : %.1f us\n", t / N * 1.0e6);
    printf("  Voxel con dose>0    : %d / %d (%.1f%%)\n", hit, NX*NY*NZ, 100.0*hit/(NX*NY*NZ));
    printf("  Energia totale dep. : %.4e MeV\n", etot);
    printf("  Energia/particella  : %.4e MeV\n", N > 0 ? etot/N : 0.0);
    printf("  Dose massima (u.a.) : %.6e\n", dmax);
}

// ══════════════════════════════════════════════════════════════
//  SEZIONE 9: PHANTOM (identico al CPU)
// ══════════════════════════════════════════════════════════════

inline void build_phantom_water(int *phantom) {
    for (int i = 0; i < NX*NY*NZ; i++) phantom[i] = MAT_WATER;
}
inline void build_phantom_hetero(int *phantom) {
    build_phantom_water(phantom);
    int cx = NX/2, cy = NY/2, cz = NZ/2;
    int meta_lato_inserto = (int)(2.5 / VOXEL_CM + 0.5);
    int count = 0;
    for (int iz = cz - meta_lato_inserto; iz < cz + meta_lato_inserto; iz++)
    for (int iy = cy - meta_lato_inserto; iy < cy + meta_lato_inserto; iy++)
    for (int ix = cx - meta_lato_inserto; ix < cx + meta_lato_inserto; ix++) {
        if (ix>=0 && ix<NX && iy>=0 && iy<NY && iz>=0 && iz<NZ) {
            phantom[phantom_idx(ix, iy, iz)] = MAT_BONE;
            count++;
        }
    }
    double vol = count * VOXEL_CM * VOXEL_CM * VOXEL_CM;
    printf("  Inserto osseo: %d voxel = %.1f cm3 (teorico 125 cm3)\n", count, vol);
}

// ══════════════════════════════════════════════════════════════
//  SEZIONE 10: MAIN
// ══════════════════════════════════════════════════════════════

#define CUDA_CHECK(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error %s:%d — %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(EXIT_FAILURE); \
    } \
} while(0)

// Carica tutte le costanti fisiche nella constant memory device
static void upload_constants() {
    CUDA_CHECK(cudaMemcpyToSymbol(d_DENSITY,    DENSITY,    sizeof(DENSITY)));
    CUDA_CHECK(cudaMemcpyToSymbol(d_ENERGY_GRID,ENERGY_GRID,sizeof(ENERGY_GRID)));
    CUDA_CHECK(cudaMemcpyToSymbol(d_MU_TOTAL,   MU_TOTAL,   sizeof(MU_TOTAL)));
    CUDA_CHECK(cudaMemcpyToSymbol(d_MU_PE,      MU_PE,      sizeof(MU_PE)));
    CUDA_CHECK(cudaMemcpyToSymbol(d_MU_COMPTON, MU_COMPTON, sizeof(MU_COMPTON)));

    // Spettro 6MV
    double cdf_h[SPECTRUM_BINS], e_h[SPECTRUM_BINS];
    double somma = 0.0;
    for (int i = 0; i < SPECTRUM_BINS; i++) somma += SPECTRUM_FLUENCE[i];
    cdf_h[0] = SPECTRUM_FLUENCE[0] / somma;
    for (int i = 1; i < SPECTRUM_BINS; i++)
        cdf_h[i] = cdf_h[i-1] + SPECTRUM_FLUENCE[i] / somma;
    cdf_h[SPECTRUM_BINS-1] = 1.0;
    for (int i = 0; i < SPECTRUM_BINS; i++) e_h[i] = SPECTRUM_E[i];
    double dE = 0.25;
    CUDA_CHECK(cudaMemcpyToSymbol(d_spectrum_cdf, cdf_h, sizeof(cdf_h)));
    CUDA_CHECK(cudaMemcpyToSymbol(d_spectrum_e,   e_h,   sizeof(e_h)));
    CUDA_CHECK(cudaMemcpyToSymbol(d_spectrum_dE,  &dE,   sizeof(double)));
}

int main(int argc, char *argv[]) {
    long long num_fotoni   = 1000000LL;
    int       tipo_phantom = 0;
    uint64_t  seed         = 42ULL;

    if (argc > 1) num_fotoni   = std::atoll(argv[1]);
    if (argc > 2) tipo_phantom = std::atoi(argv[2]);
    if (argc > 3) seed         = (uint64_t)std::atoll(argv[3]);

    const char *phantom_label = (tipo_phantom == 0) ? "Acqua omogenea" : "Acqua + Osso";

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("  Monte Carlo per Radioterapia — GPU CUDA\n");
    printf("  GPU: %s  |  SM: %d.%d  |  VRAM: %.0f MB\n",
           prop.name, prop.major, prop.minor, prop.totalGlobalMem / 1.0e6);
    printf("\n  Parametri:\n");
    printf("  Phantom    : %dx%dx%d voxel  |  voxel %.0fmm  |  %.0f cm\n",
           NX, NY, NZ, VOXEL_CM * 10.0, PHANTOM_CM);
    printf("  Materiale  : %s\n", phantom_label);
    printf("  N fotoni   : %lld\n", num_fotoni);
    printf("  Seed       : %llu\n", (unsigned long long)seed);
    printf("  ECUT       : %.0f keV\n\n", ECUT * 1000.0);

    const int THREADS_PER_BLOCK = 256;
    int n_sm     = prop.multiProcessorCount;
    int n_blocks = n_sm * 8;
    if ((long long)n_blocks * THREADS_PER_BLOCK > num_fotoni)
        n_blocks = (int)((num_fotoni + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK);
    if (n_blocks < 1) n_blocks = 1;

    printf("  Griglia CUDA: %d blocchi x %d thread = %lld thread totali\n",
           n_blocks, THREADS_PER_BLOCK, (long long)n_blocks * THREADS_PER_BLOCK);
    printf("  Fotoni/thread (media): %.1f\n\n",
           (double)num_fotoni / ((double)n_blocks * THREADS_PER_BLOCK));

    int    *h_phantom = new int   [NX * NY * NZ];
    double *h_dose    = new double[NX * NY * NZ]();
    double *pdd       = new double[NZ];
    double *depth_cm  = new double[NZ];
    double *profilo   = new double[NX];
    double *coord_lat = new double[NX];

    if (tipo_phantom == 0) { printf("  Costruzione phantom: acqua omogenea\n"); build_phantom_water(h_phantom); }
    else                   { printf("  Costruzione phantom: acqua + osso\n");    build_phantom_hetero(h_phantom); }

    // Carica TUTTE le costanti fisiche in constant memory
    upload_constants();

    int    *d_phantom; double *d_dose;
    CUDA_CHECK(cudaMalloc(&d_phantom, (size_t)NX*NY*NZ*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_dose,    (size_t)NX*NY*NZ*sizeof(double)));
    CUDA_CHECK(cudaMemcpy(d_phantom, h_phantom, (size_t)NX*NY*NZ*sizeof(int), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_dose, 0, (size_t)NX*NY*NZ*sizeof(double)));

    printf("  Avvio kernel GPU...\n");
    auto t0 = std::chrono::high_resolution_clock::now();

    mc_kernel<<<n_blocks, THREADS_PER_BLOCK>>>(d_phantom, d_dose, seed, num_fotoni);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    auto t1 = std::chrono::high_resolution_clock::now();
    double tempo = std::chrono::duration<double>(t1 - t0).count();

    CUDA_CHECK(cudaMemcpy(h_dose, d_dose, (size_t)NX*NY*NZ*sizeof(double), cudaMemcpyDeviceToHost));

    print_dose_stats(h_dose, num_fotoni, tempo);
    compute_pdd(h_dose, pdd, depth_cm);
    compute_lateral_profile(h_dose, profilo, coord_lat, 10.0);
    print_pdd_table(depth_cm, pdd, phantom_label);

    const char *pdd_file, *prof_file, *slice_file, *bin_file;
    if (tipo_phantom == 0) {
        pdd_file   = "./GPU_CUDA/pdd_water.csv";
        prof_file  = "./GPU_CUDA/profile_water.csv";
        slice_file = "./GPU_CUDA/dose_slice_water.csv";
        bin_file   = "./GPU_CUDA/dose_water.bin";
    } else {
        pdd_file   = "./GPU_CUDA/pdd_hetero.csv";
        prof_file  = "./GPU_CUDA/profile_hetero.csv";
        slice_file = "./GPU_CUDA/dose_slice_hetero.csv";
        bin_file   = "./GPU_CUDA/dose_hetero.bin";
    }
    save_pdd_csv(depth_cm, pdd, pdd_file);
    save_profile_csv(coord_lat, profilo, prof_file);
    save_dose_slice_csv(h_dose, slice_file);
    save_dose_binary(h_dose, bin_file);

    cudaFree(d_phantom); cudaFree(d_dose);
    delete[] h_phantom; delete[] h_dose;
    delete[] pdd; delete[] depth_cm; delete[] profilo; delete[] coord_lat;

    printf("\n  Simulazione GPU completata.\n");
    return 0;
}


Writing mc_gpu.cu


In [40]:
!nvcc -O3 -arch=sm_75 -std=c++17 -o mc_gpu mc_gpu.cu

mc_gpu.cu(33): warning #177-D: variable "PCUT" was declared but never referenced
  static const double PCUT = 0.100;
                      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

mc_gpu.cu(44): warning #177-D: variable "N_ENERGY" was declared but never referenced
  static const int N_ENERGY = 28;
                   ^

mc_gpu.cu(101): warning #177-D: variable "MU_PAIR" was declared but never referenced
  static const double MU_PAIR[4][28] = {
                      ^



In [43]:
!./mc_gpu 100000000 0 42

  Monte Carlo per Radioterapia — GPU CUDA
  GPU: Tesla T4  |  SM: 7.5  |  VRAM: 15637 MB

  Parametri:
  Phantom    : 100x100x100 voxel  |  voxel 3mm  |  30 cm
  Materiale  : Acqua omogenea
  N fotoni   : 100000000
  Seed       : 42
  ECUT       : 10 keV

  Griglia CUDA: 320 blocchi x 256 thread = 81920 thread totali
  Fotoni/thread (media): 1220.7

  Costruzione phantom: acqua omogenea
  Avvio kernel GPU...

  Statistiche simulazione:
  Particelle simulate : 100000000
  Tempo totale        : 59.18 s
  Throughput          : 1.690 MP/s
  Tempo/particella    : 0.6 us
  Voxel con dose>0    : 1000000 / 1000000 (100.0%)
  Energia totale dep. : 1.0507e+08 MeV
  Energia/particella  : 1.0507e+00 MeV
  Dose massima (u.a.) : 1.477742e+03

  PDD — Acqua omogenea
  Profondita [cm]         Dose [%]  Riferimento
  1.6                        97.45  build-up
  3.1                        94.22  d_max 6MV
  5.0                        89.66  
  10.0                       74.57  D10
  15.1                

# Codice GPU V2

# Codice GPU V3

# Codice GPU V4

# Test

## Costanti e metodi di utilità

### Versione CPU sequenziale

In [77]:
import os, subprocess, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from pathlib import Path

NX = NY = NZ = 100
VOXEL_CM = 0.30
PHANTOM_CM = NX * VOXEL_CM
MASSA_ELETTRONE_MEV = 0.51099895
N = numero_fotoni

ESEGUIBILE_MC = "./CPU_Sequenziale/mc_rt_cpu_sequenziale"
DOSE_WATER = "./CPU_Sequenziale/dose_water.bin"
DOSE_HETERO = "./CPU_Sequenziale/dose_hetero.bin"
PDD_WATER = "./CPU_Sequenziale/pdd_water.csv"
PDD_HETERO = "./CPU_Sequenziale/pdd_hetero.csv"

PDD_WATER_BL     = "./CPU_Sequenziale/pdd_water_BL.csv"


# Definizione spettro 6MV (Sheikh-Bagheri e Rogers)
ENERGIE_MEV = np.array([0.25,0.50,0.75,1.00,1.25,1.50,1.75,2.00,
                   2.25,2.50,2.75,3.00,3.25,3.50,3.75,4.00,
                   4.25,4.50,4.75,5.00,5.25,5.50,5.75,6.00])
PESI_SPETTRO = np.array([0.0243,0.0676,0.0862,0.0929,0.0919,0.0868,0.0794,0.0712,
                   0.0628,0.0548,0.0471,0.0399,0.0334,0.0276,0.0224,0.0178,
                   0.0138,0.0104,0.0075,0.0052,0.0034,0.0020,0.0010,0.0004])
ENERGIA_MEDIA_6MV = float(np.sum(ENERGIE_MEV * PESI_SPETTRO) / np.sum(PESI_SPETTRO))

# Definizione riferimento PDD letteratura (Sheikh-Bagheri e Rogers)
PROFONDITA_RIF = np.array([0.15,0.30,0.45,0.60,0.75,0.90,1.05,1.20,1.35,1.50,
                    1.80,2.10,2.40,3.00,3.60,4.20,5.00,6.00,7.00,8.00,
                    9.00,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,
                    19.0,20.0,21.0,22.0,23.0,24.0,25.0])
DOSE_RIF = np.array([74.0,82.0,88.0,93.0,96.5,98.5,99.5,100.0,99.8,99.5,
                    98.5,97.5,96.5,94.5,92.5,90.5,87.5,83.5,79.5,76.0,
                    73.0,70.0,67.0,64.0,61.0,58.5,56.0,53.5,51.0,48.5,
                    46.5,44.5,42.5,40.5,38.5,37.0,35.5])

def lancia_simulazione(n_fotoni, tipo_phantom=0, seed=42):
    comando = [ESEGUIBILE_MC, str(int(n_fotoni)), str(tipo_phantom), str(seed)]
    inizio = time.time()
    risultato = subprocess.run(comando, capture_output=True, text=True)
    durata = time.time() - inizio
    if risultato.returncode != 0:
        print(f"Errore nella simulazione: {risultato.stderr[:200]}")
        return False
    return True


def load_pdd(filename):
    if not os.path.exists(filename):
        print(f"  [ERRORE] file non trovato: {filename}"); return None, None
    data = np.loadtxt(filename, delimiter=',', skiprows=1)
    return data[:, 0], data[:, 1]

def load_dose_bin(nome_file):
    if not os.path.exists(nome_file):
        print(f"File non trovato: {nome_file}")
        return None
    dati_piatti = np.fromfile(nome_file, dtype=np.float64)
    return dati_piatti.reshape((NZ, NY, NX))

def save_fig(fig, name):
    path = f"./CPU_Sequenziale/{name}.png"
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f"    Figura salvata: {path}")

def calcola_coseno_medio_teorico(energia_mev, num_punti=20_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    coseni = np.linspace(-1.0, 1.0, num_punti)
    tau = 1.0 / (1.0 + alfa * (1.0 - coseni))
    probabilita = 0.5 * tau**2 * (tau + 1.0/tau - 1.0 + coseni**2)
    area = np.trapezoid(probabilita, coseni)
    probabilita_norm = probabilita / area
    coseno_medio = np.trapezoid(coseni * probabilita_norm, coseni)
    return float(coseno_medio)

def simula_urti_compton(energia_mev, generatore_casuale, n_campioni=50_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    tau_min = 1.0 / (1.0 + 2.0 * alfa)

    limite_accettazione_1 = np.log(1.0 / tau_min)
    limite_accettazione_2 = (1.0 - tau_min**2) * 0.5
    totale_limiti = limite_accettazione_1 + limite_accettazione_2

    coseni_generati = []
    tentativi_totali = 0

    while len(coseni_generati) < n_campioni:
        tentativi_totali += 1
        r1, r2, r3 = generatore_casuale.random(3)
        if r1 * totale_limiti < limite_accettazione_1:
            tau = tau_min**(1.0 - r2)
        else:
            tau = np.sqrt(tau_min**2 + r2 * (1.0 - tau_min**2))

        tau = np.clip(tau, tau_min, 1.0)

        coseno = 1.0 - (1.0 - tau) / (alfa * tau)
        coseno = np.clip(coseno, -1.0, 1.0)

        seno2 = max(0.0, 1.0 - coseno**2)
        filtro = 1.0 - (tau * seno2) / (1.0 + tau**2)

        if r3 <= filtro:
            coseni_generati.append(coseno)

    efficienza = (len(coseni_generati) / tentativi_totali) * 100
    return np.array(coseni_generati), efficienza

### Versione CPU Parallela

In [ ]:
import os, subprocess, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from pathlib import Path

NX = NY = NZ = 100
VOXEL_CM = 0.30
PHANTOM_CM = NX * VOXEL_CM
MASSA_ELETTRONE_MEV = 0.51099895
N = numero_fotoni

ESEGUIBILE_MC = "./CPU_Parallelo/mc_rt_cpu_parallelo"
DOSE_WATER = "./CPU_Parallelo/dose_water.bin"
DOSE_HETERO = "./CPU_Parallelo/dose_hetero.bin"
PDD_WATER = "./CPU_Parallelo/pdd_water.csv"
PDD_HETERO = "./CPU_Parallelo/pdd_hetero.csv"

PDD_WATER_BL = "./CPU_Parallelo/pdd_water_BL.csv"


# Definizione spettro 6MV (Sheikh-Bagheri e Rogers)
ENERGIE_MEV = np.array([0.25,0.50,0.75,1.00,1.25,1.50,1.75,2.00,
                   2.25,2.50,2.75,3.00,3.25,3.50,3.75,4.00,
                   4.25,4.50,4.75,5.00,5.25,5.50,5.75,6.00])
PESI_SPETTRO = np.array([0.0243,0.0676,0.0862,0.0929,0.0919,0.0868,0.0794,0.0712,
                   0.0628,0.0548,0.0471,0.0399,0.0334,0.0276,0.0224,0.0178,
                   0.0138,0.0104,0.0075,0.0052,0.0034,0.0020,0.0010,0.0004])
ENERGIA_MEDIA_6MV = float(np.sum(ENERGIE_MEV * PESI_SPETTRO) / np.sum(PESI_SPETTRO))

# Definizione riferimento PDD letteratura (Sheikh-Bagheri e Rogers)
PROFONDITA_RIF = np.array([0.15,0.30,0.45,0.60,0.75,0.90,1.05,1.20,1.35,1.50,
                    1.80,2.10,2.40,3.00,3.60,4.20,5.00,6.00,7.00,8.00,
                    9.00,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,
                    19.0,20.0,21.0,22.0,23.0,24.0,25.0])
DOSE_RIF = np.array([74.0,82.0,88.0,93.0,96.5,98.5,99.5,100.0,99.8,99.5,
                    98.5,97.5,96.5,94.5,92.5,90.5,87.5,83.5,79.5,76.0,
                    73.0,70.0,67.0,64.0,61.0,58.5,56.0,53.5,51.0,48.5,
                    46.5,44.5,42.5,40.5,38.5,37.0,35.5])

def lancia_simulazione(n_fotoni, tipo_phantom=0, seed=42):
    comando = [ESEGUIBILE_MC, str(int(n_fotoni)), str(tipo_phantom), str(seed)]
    inizio = time.time()
    risultato = subprocess.run(comando, capture_output=True, text=True)
    durata = time.time() - inizio
    if risultato.returncode != 0:
        print(f"Errore nella simulazione: {risultato.stderr[:200]}")
        return False
    return True


def load_pdd(filename):
    if not os.path.exists(filename):
        print(f"  [ERRORE] file non trovato: {filename}"); return None, None
    data = np.loadtxt(filename, delimiter=',', skiprows=1)
    return data[:, 0], data[:, 1]

def load_dose_bin(nome_file):
    if not os.path.exists(nome_file):
        print(f"File non trovato: {nome_file}")
        return None
    dati_piatti = np.fromfile(nome_file, dtype=np.float64)
    return dati_piatti.reshape((NZ, NY, NX))

def save_fig(fig, name):
    path = f"./CPU_Parallelo/{name}.png"
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f"    Figura salvata: {path}")

def calcola_coseno_medio_teorico(energia_mev, num_punti=20_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    coseni = np.linspace(-1.0, 1.0, num_punti)
    tau = 1.0 / (1.0 + alfa * (1.0 - coseni))
    probabilita = 0.5 * tau**2 * (tau + 1.0/tau - 1.0 + coseni**2)
    area = np.trapezoid(probabilita, coseni)
    probabilita_norm = probabilita / area
    coseno_medio = np.trapezoid(coseni * probabilita_norm, coseni)
    return float(coseno_medio)

def simula_urti_compton(energia_mev, generatore_casuale, n_campioni=50_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    tau_min = 1.0 / (1.0 + 2.0 * alfa)

    limite_accettazione_1 = np.log(1.0 / tau_min)
    limite_accettazione_2 = (1.0 - tau_min**2) * 0.5
    totale_limiti = limite_accettazione_1 + limite_accettazione_2

    coseni_generati = []
    tentativi_totali = 0

    while len(coseni_generati) < n_campioni:
        tentativi_totali += 1
        r1, r2, r3 = generatore_casuale.random(3)
        if r1 * totale_limiti < limite_accettazione_1:
            tau = tau_min**(1.0 - r2)
        else:
            tau = np.sqrt(tau_min**2 + r2 * (1.0 - tau_min**2))

        tau = np.clip(tau, tau_min, 1.0)

        coseno = 1.0 - (1.0 - tau) / (alfa * tau)
        coseno = np.clip(coseno, -1.0, 1.0)

        seno2 = max(0.0, 1.0 - coseno**2)
        filtro = 1.0 - (tau * seno2) / (1.0 + tau**2)

        if r3 <= filtro:
            coseni_generati.append(coseno)

    efficienza = (len(coseni_generati) / tentativi_totali) * 100
    return np.array(coseni_generati), efficienza

### Versione GPU V1

In [129]:
import os, subprocess, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from pathlib import Path

NX = NY = NZ = 100
VOXEL_CM = 0.30
PHANTOM_CM = NX * VOXEL_CM
MASSA_ELETTRONE_MEV = 0.51099895
N = numero_fotoni

ESEGUIBILE_MC = "./GPU_CUDA/mc_gpu"
DOSE_WATER = "./GPU_CUDA/dose_water.bin"
DOSE_HETERO = "./GPU_CUDA/dose_hetero.bin"
PDD_WATER = "./GPU_CUDA/pdd_water.csv"
PDD_HETERO = "./GPU_CUDA/pdd_hetero.csv"

PDD_WATER_BL = "./GPU_CUDA/pdd_water_BL.csv"


# Definizione spettro 6MV (Sheikh-Bagheri e Rogers)
ENERGIE_MEV = np.array([0.25,0.50,0.75,1.00,1.25,1.50,1.75,2.00,
                   2.25,2.50,2.75,3.00,3.25,3.50,3.75,4.00,
                   4.25,4.50,4.75,5.00,5.25,5.50,5.75,6.00])
PESI_SPETTRO = np.array([0.0243,0.0676,0.0862,0.0929,0.0919,0.0868,0.0794,0.0712,
                   0.0628,0.0548,0.0471,0.0399,0.0334,0.0276,0.0224,0.0178,
                   0.0138,0.0104,0.0075,0.0052,0.0034,0.0020,0.0010,0.0004])
ENERGIA_MEDIA_6MV = float(np.sum(ENERGIE_MEV * PESI_SPETTRO) / np.sum(PESI_SPETTRO))

# Definizione riferimento PDD letteratura (Sheikh-Bagheri e Rogers)
PROFONDITA_RIF = np.array([0.15,0.30,0.45,0.60,0.75,0.90,1.05,1.20,1.35,1.50,
                    1.80,2.10,2.40,3.00,3.60,4.20,5.00,6.00,7.00,8.00,
                    9.00,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,
                    19.0,20.0,21.0,22.0,23.0,24.0,25.0])
DOSE_RIF = np.array([74.0,82.0,88.0,93.0,96.5,98.5,99.5,100.0,99.8,99.5,
                    98.5,97.5,96.5,94.5,92.5,90.5,87.5,83.5,79.5,76.0,
                    73.0,70.0,67.0,64.0,61.0,58.5,56.0,53.5,51.0,48.5,
                    46.5,44.5,42.5,40.5,38.5,37.0,35.5])

def lancia_simulazione(n_fotoni, tipo_phantom=0, seed=42):
    comando = [ESEGUIBILE_MC, str(int(n_fotoni)), str(tipo_phantom), str(seed)]
    inizio = time.time()
    risultato = subprocess.run(comando, capture_output=True, text=True)
    durata = time.time() - inizio
    if risultato.returncode != 0:
        print(f"Errore nella simulazione: {risultato.stderr[:200]}")
        return False
    return True


def load_pdd(filename):
    if not os.path.exists(filename):
        print(f"  [ERRORE] file non trovato: {filename}"); return None, None
    data = np.loadtxt(filename, delimiter=',', skiprows=1)
    return data[:, 0], data[:, 1]

def load_dose_bin(nome_file):
    if not os.path.exists(nome_file):
        print(f"File non trovato: {nome_file}")
        return None
    dati_piatti = np.fromfile(nome_file, dtype=np.float64)
    return dati_piatti.reshape((NZ, NY, NX))

def save_fig(fig, name):
    path = f"./GPU_CUDA/{name}.png"
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f"    Figura salvata: {path}")

def calcola_coseno_medio_teorico(energia_mev, num_punti=20_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    coseni = np.linspace(-1.0, 1.0, num_punti)
    tau = 1.0 / (1.0 + alfa * (1.0 - coseni))
    probabilita = 0.5 * tau**2 * (tau + 1.0/tau - 1.0 + coseni**2)
    area = np.trapezoid(probabilita, coseni)
    probabilita_norm = probabilita / area
    coseno_medio = np.trapezoid(coseni * probabilita_norm, coseni)
    return float(coseno_medio)

def simula_urti_compton(energia_mev, generatore_casuale, n_campioni=50_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    tau_min = 1.0 / (1.0 + 2.0 * alfa)

    limite_accettazione_1 = np.log(1.0 / tau_min)
    limite_accettazione_2 = (1.0 - tau_min**2) * 0.5
    totale_limiti = limite_accettazione_1 + limite_accettazione_2

    coseni_generati = []
    tentativi_totali = 0

    while len(coseni_generati) < n_campioni:
        tentativi_totali += 1
        r1, r2, r3 = generatore_casuale.random(3)
        if r1 * totale_limiti < limite_accettazione_1:
            tau = tau_min**(1.0 - r2)
        else:
            tau = np.sqrt(tau_min**2 + r2 * (1.0 - tau_min**2))

        tau = np.clip(tau, tau_min, 1.0)

        coseno = 1.0 - (1.0 - tau) / (alfa * tau)
        coseno = np.clip(coseno, -1.0, 1.0)

        seno2 = max(0.0, 1.0 - coseno**2)
        filtro = 1.0 - (tau * seno2) / (1.0 + tau**2)

        if r3 <= filtro:
            coseni_generati.append(coseno)

    efficienza = (len(coseni_generati) / tentativi_totali) * 100
    return np.array(coseni_generati), efficienza

### Versione GPU V2

In [ ]:
import os, subprocess, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from pathlib import Path

NX = NY = NZ = 100
VOXEL_CM = 0.30
PHANTOM_CM = NX * VOXEL_CM
MASSA_ELETTRONE_MEV = 0.51099895
N = numero_fotoni

ESEGUIBILE_MC = "./GPU_V2/mc_rt_cpu_parallelo"
DOSE_WATER = "./GPU_V2/dose_water.bin"
DOSE_HETERO = "./GPU_V2/dose_hetero.bin"
PDD_WATER = "./GPU_V2/pdd_water.csv"
PDD_HETERO = "./GPU_V2/pdd_hetero.csv"

PDD_WATER_BL = "./GPU_V2/pdd_water_BL.csv"


# Definizione spettro 6MV (Sheikh-Bagheri e Rogers)
ENERGIE_MEV = np.array([0.25,0.50,0.75,1.00,1.25,1.50,1.75,2.00,
                   2.25,2.50,2.75,3.00,3.25,3.50,3.75,4.00,
                   4.25,4.50,4.75,5.00,5.25,5.50,5.75,6.00])
PESI_SPETTRO = np.array([0.0243,0.0676,0.0862,0.0929,0.0919,0.0868,0.0794,0.0712,
                   0.0628,0.0548,0.0471,0.0399,0.0334,0.0276,0.0224,0.0178,
                   0.0138,0.0104,0.0075,0.0052,0.0034,0.0020,0.0010,0.0004])
ENERGIA_MEDIA_6MV = float(np.sum(ENERGIE_MEV * PESI_SPETTRO) / np.sum(PESI_SPETTRO))

# Definizione riferimento PDD letteratura (Sheikh-Bagheri e Rogers)
PROFONDITA_RIF = np.array([0.15,0.30,0.45,0.60,0.75,0.90,1.05,1.20,1.35,1.50,
                    1.80,2.10,2.40,3.00,3.60,4.20,5.00,6.00,7.00,8.00,
                    9.00,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,
                    19.0,20.0,21.0,22.0,23.0,24.0,25.0])
DOSE_RIF = np.array([74.0,82.0,88.0,93.0,96.5,98.5,99.5,100.0,99.8,99.5,
                    98.5,97.5,96.5,94.5,92.5,90.5,87.5,83.5,79.5,76.0,
                    73.0,70.0,67.0,64.0,61.0,58.5,56.0,53.5,51.0,48.5,
                    46.5,44.5,42.5,40.5,38.5,37.0,35.5])

def lancia_simulazione(n_fotoni, tipo_phantom=0, seed=42):
    comando = [ESEGUIBILE_MC, str(int(n_fotoni)), str(tipo_phantom), str(seed)]
    inizio = time.time()
    risultato = subprocess.run(comando, capture_output=True, text=True)
    durata = time.time() - inizio
    if risultato.returncode != 0:
        print(f"Errore nella simulazione: {risultato.stderr[:200]}")
        return False
    return True


def load_pdd(filename):
    if not os.path.exists(filename):
        print(f"  [ERRORE] file non trovato: {filename}"); return None, None
    data = np.loadtxt(filename, delimiter=',', skiprows=1)
    return data[:, 0], data[:, 1]

def load_dose_bin(nome_file):
    if not os.path.exists(nome_file):
        print(f"File non trovato: {nome_file}")
        return None
    dati_piatti = np.fromfile(nome_file, dtype=np.float64)
    return dati_piatti.reshape((NZ, NY, NX))

def save_fig(fig, name):
    path = f"./GPU_V2/{name}.png"
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f"    Figura salvata: {path}")

def calcola_coseno_medio_teorico(energia_mev, num_punti=20_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    coseni = np.linspace(-1.0, 1.0, num_punti)
    tau = 1.0 / (1.0 + alfa * (1.0 - coseni))
    probabilita = 0.5 * tau**2 * (tau + 1.0/tau - 1.0 + coseni**2)
    area = np.trapezoid(probabilita, coseni)
    probabilita_norm = probabilita / area
    coseno_medio = np.trapezoid(coseni * probabilita_norm, coseni)
    return float(coseno_medio)

def simula_urti_compton(energia_mev, generatore_casuale, n_campioni=50_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    tau_min = 1.0 / (1.0 + 2.0 * alfa)

    limite_accettazione_1 = np.log(1.0 / tau_min)
    limite_accettazione_2 = (1.0 - tau_min**2) * 0.5
    totale_limiti = limite_accettazione_1 + limite_accettazione_2

    coseni_generati = []
    tentativi_totali = 0

    while len(coseni_generati) < n_campioni:
        tentativi_totali += 1
        r1, r2, r3 = generatore_casuale.random(3)
        if r1 * totale_limiti < limite_accettazione_1:
            tau = tau_min**(1.0 - r2)
        else:
            tau = np.sqrt(tau_min**2 + r2 * (1.0 - tau_min**2))

        tau = np.clip(tau, tau_min, 1.0)

        coseno = 1.0 - (1.0 - tau) / (alfa * tau)
        coseno = np.clip(coseno, -1.0, 1.0)

        seno2 = max(0.0, 1.0 - coseno**2)
        filtro = 1.0 - (tau * seno2) / (1.0 + tau**2)

        if r3 <= filtro:
            coseni_generati.append(coseno)

    efficienza = (len(coseni_generati) / tentativi_totali) * 100
    return np.array(coseni_generati), efficienza

### Versione GPU V3

In [ ]:
import os, subprocess, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from pathlib import Path

NX = NY = NZ = 100
VOXEL_CM = 0.30
PHANTOM_CM = NX * VOXEL_CM
MASSA_ELETTRONE_MEV = 0.51099895
N = numero_fotoni

ESEGUIBILE_MC = "./GPU_V3/mc_rt_cpu_parallelo"
DOSE_WATER = "./GPU_V3/dose_water.bin"
DOSE_HETERO = "./GPU_V3/dose_hetero.bin"
PDD_WATER = "./GPU_V3/pdd_water.csv"
PDD_HETERO = "./GPU_V3/pdd_hetero.csv"

PDD_WATER_BL = "./GPU_V3/pdd_water_BL.csv"


# Definizione spettro 6MV (Sheikh-Bagheri e Rogers)
ENERGIE_MEV = np.array([0.25,0.50,0.75,1.00,1.25,1.50,1.75,2.00,
                   2.25,2.50,2.75,3.00,3.25,3.50,3.75,4.00,
                   4.25,4.50,4.75,5.00,5.25,5.50,5.75,6.00])
PESI_SPETTRO = np.array([0.0243,0.0676,0.0862,0.0929,0.0919,0.0868,0.0794,0.0712,
                   0.0628,0.0548,0.0471,0.0399,0.0334,0.0276,0.0224,0.0178,
                   0.0138,0.0104,0.0075,0.0052,0.0034,0.0020,0.0010,0.0004])
ENERGIA_MEDIA_6MV = float(np.sum(ENERGIE_MEV * PESI_SPETTRO) / np.sum(PESI_SPETTRO))

# Definizione riferimento PDD letteratura (Sheikh-Bagheri e Rogers)
PROFONDITA_RIF = np.array([0.15,0.30,0.45,0.60,0.75,0.90,1.05,1.20,1.35,1.50,
                    1.80,2.10,2.40,3.00,3.60,4.20,5.00,6.00,7.00,8.00,
                    9.00,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,
                    19.0,20.0,21.0,22.0,23.0,24.0,25.0])
DOSE_RIF = np.array([74.0,82.0,88.0,93.0,96.5,98.5,99.5,100.0,99.8,99.5,
                    98.5,97.5,96.5,94.5,92.5,90.5,87.5,83.5,79.5,76.0,
                    73.0,70.0,67.0,64.0,61.0,58.5,56.0,53.5,51.0,48.5,
                    46.5,44.5,42.5,40.5,38.5,37.0,35.5])

def lancia_simulazione(n_fotoni, tipo_phantom=0, seed=42):
    comando = [ESEGUIBILE_MC, str(int(n_fotoni)), str(tipo_phantom), str(seed)]
    inizio = time.time()
    risultato = subprocess.run(comando, capture_output=True, text=True)
    durata = time.time() - inizio
    if risultato.returncode != 0:
        print(f"Errore nella simulazione: {risultato.stderr[:200]}")
        return False
    return True


def load_pdd(filename):
    if not os.path.exists(filename):
        print(f"  [ERRORE] file non trovato: {filename}"); return None, None
    data = np.loadtxt(filename, delimiter=',', skiprows=1)
    return data[:, 0], data[:, 1]

def load_dose_bin(nome_file):
    if not os.path.exists(nome_file):
        print(f"File non trovato: {nome_file}")
        return None
    dati_piatti = np.fromfile(nome_file, dtype=np.float64)
    return dati_piatti.reshape((NZ, NY, NX))

def save_fig(fig, name):
    path = f"./GPU_V3/{name}.png"
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f"    Figura salvata: {path}")

def calcola_coseno_medio_teorico(energia_mev, num_punti=20_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    coseni = np.linspace(-1.0, 1.0, num_punti)
    tau = 1.0 / (1.0 + alfa * (1.0 - coseni))
    probabilita = 0.5 * tau**2 * (tau + 1.0/tau - 1.0 + coseni**2)
    area = np.trapezoid(probabilita, coseni)
    probabilita_norm = probabilita / area
    coseno_medio = np.trapezoid(coseni * probabilita_norm, coseni)
    return float(coseno_medio)

def simula_urti_compton(energia_mev, generatore_casuale, n_campioni=50_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    tau_min = 1.0 / (1.0 + 2.0 * alfa)

    limite_accettazione_1 = np.log(1.0 / tau_min)
    limite_accettazione_2 = (1.0 - tau_min**2) * 0.5
    totale_limiti = limite_accettazione_1 + limite_accettazione_2

    coseni_generati = []
    tentativi_totali = 0

    while len(coseni_generati) < n_campioni:
        tentativi_totali += 1
        r1, r2, r3 = generatore_casuale.random(3)
        if r1 * totale_limiti < limite_accettazione_1:
            tau = tau_min**(1.0 - r2)
        else:
            tau = np.sqrt(tau_min**2 + r2 * (1.0 - tau_min**2))

        tau = np.clip(tau, tau_min, 1.0)

        coseno = 1.0 - (1.0 - tau) / (alfa * tau)
        coseno = np.clip(coseno, -1.0, 1.0)

        seno2 = max(0.0, 1.0 - coseno**2)
        filtro = 1.0 - (tau * seno2) / (1.0 + tau**2)

        if r3 <= filtro:
            coseni_generati.append(coseno)

    efficienza = (len(coseni_generati) / tentativi_totali) * 100
    return np.array(coseni_generati), efficienza

### Versione GPU V4

In [ ]:
import os, subprocess, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from pathlib import Path

NX = NY = NZ = 100
VOXEL_CM = 0.30
PHANTOM_CM = NX * VOXEL_CM
MASSA_ELETTRONE_MEV = 0.51099895
N = numero_fotoni

ESEGUIBILE_MC = "./GPU_V4/mc_rt_cpu_parallelo"
DOSE_WATER = "./GPU_V4/dose_water.bin"
DOSE_HETERO = "./GPU_V4/dose_hetero.bin"
PDD_WATER = "./GPU_V4/pdd_water.csv"
PDD_HETERO = "./GPU_V4/pdd_hetero.csv"

PDD_WATER_BL = "./GPU_V4/pdd_water_BL.csv"


# Definizione spettro 6MV (Sheikh-Bagheri e Rogers)
ENERGIE_MEV = np.array([0.25,0.50,0.75,1.00,1.25,1.50,1.75,2.00,
                   2.25,2.50,2.75,3.00,3.25,3.50,3.75,4.00,
                   4.25,4.50,4.75,5.00,5.25,5.50,5.75,6.00])
PESI_SPETTRO = np.array([0.0243,0.0676,0.0862,0.0929,0.0919,0.0868,0.0794,0.0712,
                   0.0628,0.0548,0.0471,0.0399,0.0334,0.0276,0.0224,0.0178,
                   0.0138,0.0104,0.0075,0.0052,0.0034,0.0020,0.0010,0.0004])
ENERGIA_MEDIA_6MV = float(np.sum(ENERGIE_MEV * PESI_SPETTRO) / np.sum(PESI_SPETTRO))

# Definizione riferimento PDD letteratura (Sheikh-Bagheri e Rogers)
PROFONDITA_RIF = np.array([0.15,0.30,0.45,0.60,0.75,0.90,1.05,1.20,1.35,1.50,
                    1.80,2.10,2.40,3.00,3.60,4.20,5.00,6.00,7.00,8.00,
                    9.00,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,
                    19.0,20.0,21.0,22.0,23.0,24.0,25.0])
DOSE_RIF = np.array([74.0,82.0,88.0,93.0,96.5,98.5,99.5,100.0,99.8,99.5,
                    98.5,97.5,96.5,94.5,92.5,90.5,87.5,83.5,79.5,76.0,
                    73.0,70.0,67.0,64.0,61.0,58.5,56.0,53.5,51.0,48.5,
                    46.5,44.5,42.5,40.5,38.5,37.0,35.5])

def lancia_simulazione(n_fotoni, tipo_phantom=0, seed=42):
    comando = [ESEGUIBILE_MC, str(int(n_fotoni)), str(tipo_phantom), str(seed)]
    inizio = time.time()
    risultato = subprocess.run(comando, capture_output=True, text=True)
    durata = time.time() - inizio
    if risultato.returncode != 0:
        print(f"Errore nella simulazione: {risultato.stderr[:200]}")
        return False
    return True


def load_pdd(filename):
    if not os.path.exists(filename):
        print(f"  [ERRORE] file non trovato: {filename}"); return None, None
    data = np.loadtxt(filename, delimiter=',', skiprows=1)
    return data[:, 0], data[:, 1]

def load_dose_bin(nome_file):
    if not os.path.exists(nome_file):
        print(f"File non trovato: {nome_file}")
        return None
    dati_piatti = np.fromfile(nome_file, dtype=np.float64)
    return dati_piatti.reshape((NZ, NY, NX))

def save_fig(fig, name):
    path = f"./GPU_V4/{name}.png"
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f"    Figura salvata: {path}")

def calcola_coseno_medio_teorico(energia_mev, num_punti=20_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    coseni = np.linspace(-1.0, 1.0, num_punti)
    tau = 1.0 / (1.0 + alfa * (1.0 - coseni))
    probabilita = 0.5 * tau**2 * (tau + 1.0/tau - 1.0 + coseni**2)
    area = np.trapezoid(probabilita, coseni)
    probabilita_norm = probabilita / area
    coseno_medio = np.trapezoid(coseni * probabilita_norm, coseni)
    return float(coseno_medio)

def simula_urti_compton(energia_mev, generatore_casuale, n_campioni=50_000):
    alfa = energia_mev / MASSA_ELETTRONE_MEV
    tau_min = 1.0 / (1.0 + 2.0 * alfa)

    limite_accettazione_1 = np.log(1.0 / tau_min)
    limite_accettazione_2 = (1.0 - tau_min**2) * 0.5
    totale_limiti = limite_accettazione_1 + limite_accettazione_2

    coseni_generati = []
    tentativi_totali = 0

    while len(coseni_generati) < n_campioni:
        tentativi_totali += 1
        r1, r2, r3 = generatore_casuale.random(3)
        if r1 * totale_limiti < limite_accettazione_1:
            tau = tau_min**(1.0 - r2)
        else:
            tau = np.sqrt(tau_min**2 + r2 * (1.0 - tau_min**2))

        tau = np.clip(tau, tau_min, 1.0)

        coseno = 1.0 - (1.0 - tau) / (alfa * tau)
        coseno = np.clip(coseno, -1.0, 1.0)

        seno2 = max(0.0, 1.0 - coseno**2)
        filtro = 1.0 - (tau * seno2) / (1.0 + tau**2)

        if r3 <= filtro:
            coseni_generati.append(coseno)

    efficienza = (len(coseni_generati) / tentativi_totali) * 100
    return np.array(coseni_generati), efficienza

## Test 1 - Analisi del Coefficiente di Attenuazione Lineare

Viene eseguita la validazione fisica della distribuzione di dose in acqua attraverso il calcolo del coefficiente di attenuazione lineare. Questo serve per verificare che il comportamento dei fotoni simulati segua correttamente la legge dell'attenuazione esponenziale nella regione di post-massimo.

Viene selezionata la regione (ROI) compresa tra 5.0 cm e 20.0 cm perchè è quella dove l'attenuazione è prevalentemente dominata dalla componente primaria del fascio. Viene eseguita una regressione lineare sul logaritmo della dose rispetto alla profondità. La pendenza della retta risultante identifica il coefficiente $\mu$ misurato. Viene, inoltre, calcolato il coefficiente di determinazione per valutare la bontà del fit e la coerenza del modello esponenziale con i dati simulati. Il valore ottenuto viene confrontato con un range di accettabilità fisica e confrontato con i dati standard NIST XCOM per energie monoenergetiche.

In [109]:
def test_1_coefficiente_attenuazione():

    profondita, dose = load_pdd(PDD_WATER)
    if profondita is None:
        return False

    filtro_zona_fit = (profondita >= 5.0) & (profondita <= 20.0) & (dose > 0)
    z_da_analizzare = profondita[filtro_zona_fit]
    dose_da_analizzare = dose[filtro_zona_fit]

    pendenza, intercetta = np.polyfit(z_da_analizzare, np.log(dose_da_analizzare), 1)
    mu_misurato = -pendenza

    MU_MIN = 0.030
    MU_MAX = 0.055

    test_superato = MU_MIN <= mu_misurato <= MU_MAX

    residui = np.log(dose_da_analizzare) - (pendenza * z_da_analizzare + intercetta)
    varianza_totale = np.log(dose_da_analizzare) - np.log(dose_da_analizzare).mean()

    somma_residui = np.sum(residui**2)
    somma_totale = np.sum(varianza_totale**2)
    r_quadrato = 1.0 - (somma_residui / somma_totale) if somma_totale > 0 else 0.0

    print(f"\n ANALISI DEL COEFFICIENTE DI ATTENUAZIONE ")
    print(f" Valore calcolato:  {mu_misurato:.5f} cm⁻¹")
    print(f" Qualità estrazione:   {r_quadrato:.4f}")


    if test_superato:
        print(f"\n Verifica limite fisico (Range atteso: {MU_MIN} - {MU_MAX} cm⁻¹) coerente")
    else:
        print(f" Valore è fuori range")

    print(f"\n Confronto con dati standard (NIST XCOM Monoenergetici):")
    dati_riferimento_nist = {
        "1.0 MeV": 0.07072,
        "1.5 MeV": 0.05754,
        "2.0 MeV": 0.04942,
        "3.0 MeV": 0.03969
    }

    for energia, mu_nist in dati_riferimento_nist.items():
        differenza_perc = abs(mu_misurato - mu_nist) / mu_nist * 100
        print(f" Rispetto a fotoni da {energia}:")
        print(f"  - Valore teorico: {mu_nist:.5f} cm⁻¹")
        print(f"  - Valore misurato: {mu_misurato:.5f} cm⁻¹")
        print(f"  - Differenza:     {differenza_perc:.1f}%")
    print("\n")

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.semilogy(profondita, dose, 'b-', label='Dose Simulata')
    z_grafico = np.linspace(0, 30, 100)
    dose_fit = np.exp(intercetta + pendenza * z_grafico)
    ax.semilogy(z_grafico, dose_fit, 'r--', label=f'Fit Esponenziale (μ={mu_misurato:.4f})')
    ax.axvspan(5.0, 20.0, color='green', alpha=0.1, label='Zona Analisi')
    ax.set_xlabel('Profondità (cm)')
    ax.set_ylabel('Dose (%)')
    ax.set_title(f'Validazione Attenuazione - {"Superato" if test_superato else "Non superato"}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_fig(fig, 'Test_1_Valutazione_Attenuazione')

    print(f"\n TEST 1 SUPERATO: {test_superato}")

    return test_superato


risultato = test_1_coefficiente_attenuazione()


 ANALISI DEL COEFFICIENTE DI ATTENUAZIONE 
 Valore calcolato:  0.03866 cm⁻¹
 Qualità estrazione:   0.9966

 Verifica limite fisico (Range atteso: 0.03 - 0.055 cm⁻¹) coerente

 Confronto con dati standard (NIST XCOM Monoenergetici):
 Rispetto a fotoni da 1.0 MeV:
  - Valore teorico: 0.07072 cm⁻¹
  - Valore misurato: 0.03866 cm⁻¹
  - Differenza:     45.3%
 Rispetto a fotoni da 1.5 MeV:
  - Valore teorico: 0.05754 cm⁻¹
  - Valore misurato: 0.03866 cm⁻¹
  - Differenza:     32.8%
 Rispetto a fotoni da 2.0 MeV:
  - Valore teorico: 0.04942 cm⁻¹
  - Valore misurato: 0.03866 cm⁻¹
  - Differenza:     21.8%
 Rispetto a fotoni da 3.0 MeV:
  - Valore teorico: 0.03969 cm⁻¹
  - Valore misurato: 0.03866 cm⁻¹
  - Differenza:     2.6%


    Figura salvata: ./GPU_CUDA/Test_1_Valutazione_Attenuazione.png

 TEST 1 SUPERATO: True


## Test 2 - Validazione della Legge di Beer Lambert

Viene eseguito un confronto tra i dati di dose ottenuti tramite simulazione Monte Carlo e il modello teorico basato sulla legge di Beer-Lambert. Il test verifica se il trasporto dei fotoni nel mezzo segue le leggi della fisica classica dell'attenuazione.

Viene generata una curva teorica ideale utilizzando il coefficiente di attenuazione standard ($\mu_{teorico} = 0.04942 \text{ cm}^{-1}$), corrispondente a fotoni monoenergetici da 2.0 MeV in acqua (dati NIST). Ogni punto viene considerato valido se l'errore assoluto rimane entro una tolleranza del 2.0%. Viene poi ricalcolato il coefficiente di attenuazione dalla simulazione nella zona di equilibrio e confrontato con il valore teorico. Il test viene superato solo se l'errore percentuale su $\mu$ è inferiore al 2%.

In [79]:
def test_2_validazione_BeerLambert():

    MU_TEORICO = 0.04942
    TOLLERANZA_DOSE = 2.0

    profondita, dose_mc = load_pdd(PDD_WATER_BL)
    if profondita is None:
        return False

    dose_teorica = 100.0 * np.exp(-MU_TEORICO * profondita)

    print(f"\n Confronto dose simulata e dose teorica:")
    print(f"{'Prof [cm]':>10} | {'Simulata [%]':>12} | {'Teorica [%]':>12} | {'Errore':>8}")
    print("-" * 55)

    esiti_punti = []
    profondita_controllo = [1.5, 3.0, 5.0, 10.0, 15.0, 20.0, 25.0]

    for z in profondita_controllo:
        idx = np.abs(profondita - z).argmin()
        d_sim = dose_mc[idx]
        d_teo = 100.0 * np.exp(-MU_TEORICO * z)
        differenza = abs(d_sim - d_teo)

        punto_ok = differenza < TOLLERANZA_DOSE
        esiti_punti.append(punto_ok)

        stato = "BUONO" if punto_ok else "ERRORE"
        print(f"{z:>10.1f} | {d_sim:>12.2f} | {d_teo:>12.2f} | {stato:>8} ({differenza:.3f})")

    filtro = (profondita >= 5.0) & (profondita <= 20.0)
    pendenza, _ = np.polyfit(profondita[filtro], np.log(dose_mc[filtro]), 1)
    mu_misurato = -pendenza
    errore_mu = abs(mu_misurato - MU_TEORICO) / MU_TEORICO * 100

    mu_ok = errore_mu < 2.0
    esiti_punti.append(mu_ok)

    print(f"\n Analisi Coefficiente (mu):")
    print(f"  Misurato: {mu_misurato:.5f} cm⁻¹")
    print(f"  Teorico:  {MU_TEORICO:.5f} cm⁻¹")
    print(f"  Errore:   {errore_mu:.2f}% ({'Nei limiti' if mu_ok else 'Fuori limite'})")
    print("\n")

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(profondita, dose_mc, 'b-', label='Simulazione Monte Carlo')
    ax.plot(profondita, dose_teorica, 'r--', label='Legge Teorica (Beer-Lambert)')
    ax.fill_between(profondita, dose_teorica - TOLLERANZA_DOSE, dose_teorica + TOLLERANZA_DOSE, color='red', alpha=0.1, label='Fascia Tolleranza ±2%')
    ax.set_xlabel('Profondità (cm)')
    ax.set_ylabel('Dose (%)')
    ax.set_title('Test 2: Validazione Fisica Beer Lambert')
    ax.legend()
    ax.grid(True, alpha=0.3)

    save_fig(fig, 'Test_2_Validazione_Beer_Lambert')

    test_superato = all(esiti_punti)
    print(f"\nTEST 2 SUPERATO: {test_superato}")

    return test_superato

risultato = test_2_validazione_BeerLambert()


  [ERRORE] file non trovato: ./CPU_Sequenziale/pdd_water_BL.csv


## Test 3 - Verifica delle sezioni d'urto e probabilità di interazione

Viene effettuata un'analisi  dei meccanismi di interazione radiazione-materia per fotoni da 2 MeV in acqua, sulla base dei dati del database NIST XCOM. Questo viene fatto per confermare che il trasporto dei fotoni nel simulatore rispetti la predominanza dei processi fisici attesi a questa specifica energia.

Per fare questo il coefficiente di attenuazione lineare totale ($\mu$) viene scomposto nei suoi tre contributi principali:
* Effetto Fotoelettrico
* Effetto Compton
* Produzione di Coppie

A 2 MeV in acqua, l'effetto Compton deve rappresentare la quasi totalità delle interazioni. Viene verificato che l'effetto fotoelettrico sia trascurabile e che la produzione di coppie sia coerente con l'energia del fascio impostata. Viene anche effettuato un controllo sulla conservazione della probabilità affinché la somma dei contributi parziali sia esattamente pari al 100%.

In [80]:
def test_3_verifica_sezioni_urto_nist():

    mu_fotoelettrico = 2.200e-7
    mu_compton = 0.04942
    mu_coppie = 0.0

    mu_totale = mu_fotoelettrico + mu_compton + mu_coppie

    perc_fotoelettrico = (mu_fotoelettrico / mu_totale) * 100
    perc_compton = (mu_compton / mu_totale) * 100
    perc_coppie = (mu_coppie / mu_totale) * 100

    print(f"ANALISI PROBABILITÀ DI INTERAZIONE (2 MeV in Acqua)\n")
    print(f"Coefficiente totale (mu): {mu_totale:.6f} cm²/g\n")
    print(f"Effetto Fotoelettrico   : {perc_fotoelettrico:.5f}%  (Atteso: quasi 0%)")
    print(f"Effetto Compton         : {perc_compton:.4f}% (Atteso: > 99.9%)")
    print(f"Produzione di Coppie    : {perc_coppie:.4f}%  (Atteso: 0.0%)")

    controlli = [
        ("Dominanza Compton (> 99.9%)      ",  perc_compton > 99.9),
        ("Fotoelettrico trascurabile       ",   perc_fotoelettrico < 0.01),
        ("Assenza produzione coppie        ",    perc_coppie == 0.0),
        ("Coerenza matematica (Somma=100%) ", abs(perc_fotoelettrico + perc_compton + perc_coppie - 100.0) < 1e-6)
    ]

    esiti = []
    print("\nVerifica coerenza dati:")
    for descrizione, superato in controlli:
        stato = "Coerente" if superato else "Non coerente"
        print(f"  {descrizione}: {stato}")
        esiti.append(superato)

    test_superato = all(esiti)
    print(f"\nTEST 3 SUPERATO: {test_superato}")

    return test_superato

risultato = test_3_verifica_sezioni_urto_nist()

ANALISI PROBABILITÀ DI INTERAZIONE (2 MeV in Acqua)

Coefficiente totale (mu): 0.049420 cm²/g

Effetto Fotoelettrico   : 0.00045%  (Atteso: quasi 0%)
Effetto Compton         : 99.9996% (Atteso: > 99.9%)
Produzione di Coppie    : 0.0000%  (Atteso: 0.0%)

Verifica coerenza dati:
  Dominanza Compton (> 99.9%)      : Coerente
  Fotoelettrico trascurabile       : Coerente
  Assenza produzione coppie        : Coerente
  Coerenza matematica (Somma=100%) : Coerente

TEST 3 SUPERATO: True


## Test 4 - Validazione stocastica della deviazione Compton


Viene verificata l'accuratezza dell'algoritmo di campionamento stocastico per lo scattering Compton. Nello specifico, confronta l'implementazione numerica basata sull'algoritmo di Kahn (metodo di rigetto) con la distribuzione teorica predetta dalla formula di Klein-Nishina.

Per diverse energie del fotone incidente (da 0.5 a 6.0 MeV), vengono generati 50.000 campioni dell'angolo di deviazione ($\cos\theta$) utilizzando il generatore di numeri casuali. Per ogni energia, poi, viene calcolata la media dei coseni campionati e confrontata con il valore medio analitico derivato dall'integrazione della sezione d'urto di Klein-Nishina. Il test è superato se l'errore relativo sulla media dei coseni rimane inferiore al 5% per tutti i livelli energetici testati.

In [81]:
def test_4_validazione_compton_kahn():

    energie_test = [0.5, 1.0, 2.0, 4.0, 6.0]
    generatore_casuale = np.random.default_rng(42)
    numero_campioni = 50_000
    limite_errore = 5.0

    print(f"ANALISI DEVIAZIONE COMPTON (Kahn e Klein-Nishina)\n")
    print(f"{'E [MeV]':>8} | {'<cos> MC':>12} | {'<cos> Teorico':>14} | {'Errore %':>10}  | {'Efficienza':>10}")
    print("-" * 67)

    esiti = []
    dati_per_grafico = {}

    for E in energie_test:
        media_teorica = calcola_coseno_medio_teorico(E)
        angoli_campionati, efficienza = simula_urti_compton(E, generatore_casuale, n_campioni=numero_campioni)

        media_mc = float(angoli_campionati.mean())
        errore_relativo = abs(media_mc - media_teorica) / abs(media_teorica) * 100.0

        ok = errore_relativo < limite_errore
        esiti.append(ok)
        dati_per_grafico[E] = angoli_campionati

        stato = "OK" if ok else "KO"
        print(f"{E:>8.1f} | {media_mc:>12.4f} | {media_teorica:>14.4f} | {errore_relativo:>9.2f}%  | {efficienza:.3f}")
    print("")

    E_plot = 2.0
    campioni_2mev = dati_per_grafico[E_plot]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(campioni_2mev, bins=50, density=True, alpha=0.5, color='steelblue', label=f'Simulazione Kahn ({numero_campioni} urti)')
    cos_grid = np.linspace(-1, 1, 400)

    alfa = E_plot / 0.511
    tau = 1.0 / (1.0 + alfa * (1.0 - cos_grid))
    pdf_teorica = 0.5 * tau**2 * (tau + 1.0/tau - 1.0 + cos_grid**2)
    pdf_teorica /= np.trapezoid(pdf_teorica, cos_grid)

    ax.plot(cos_grid, pdf_teorica, 'r-', lw=2, label='Teoria Klein-Nishina')
    ax.set_xlabel('Coseno dell\'angolo di deviazione (cos theta)')
    ax.set_ylabel('Probabilità (PDF)')
    ax.set_title(f'Confronto Distribuzione Angolare a {E_plot} MeV')
    ax.legend()
    ax.grid(True, alpha=0.3)

    save_fig(fig, 'Test_4_validazione_compton_kahn')

    test_superato = all(esiti)
    print(f"\nTEST 4 SUPERATO: {test_superato}")

    return test_superato

risultato = test_4_validazione_compton_kahn()

ANALISI DEVIAZIONE COMPTON (Kahn e Klein-Nishina)

 E [MeV] |     <cos> MC |  <cos> Teorico |   Errore %  | Efficienza
-------------------------------------------------------------------
     0.5 |       0.2863 |         0.2891 |      0.97%  | 73.958
     1.0 |       0.3613 |         0.3602 |      0.30%  | 79.916
     2.0 |       0.4232 |         0.4256 |      0.57%  | 86.032
     4.0 |       0.4838 |         0.4870 |      0.66%  | 90.843
     6.0 |       0.5222 |         0.5208 |      0.26%  | 93.223

    Figura salvata: ./CPU_Sequenziale/Test_4_validazione_compton_kahn.png

TEST 4 SUPERATO: True


## Test 5 - Analisi del bilancio energetico globale

Viene verificata la coerenza termodinamica della simulazione, analizzando come l'energia emessa dalla sorgente (spettro clinico da 6 MV) venga distribuita all'interno del phantom d'acqua. L'obiettivo è confermare che non vi siano perdite ingiustificate o creazioni di energia non fisica durante il trasporto dei voxel.

Viene calcoalta l'energia totale immessa nel sistema che viene confrontata con la somma integrale della dose depositata in tutto il volume 3D. In un phantom di dimensioni finite (30 cm), è normale che una parte dell'energia venga trasportata all'esterno dai fotoni trasmessi o diffusi (backscattering e leakage laterale). La frazione depositata deve rientrare in un intervallo realistico per un fascio da 6 MV.

In [117]:
def test_5_bilancio_energetico():

    print(f"ANALISI DEL BILANCIO ENERGETICO (Spettro 6MV)")

    energia_media_singolo_fotone = ENERGIA_MEDIA_6MV
    dose_3d = load_dose_bin(DOSE_WATER)
    if dose_3d is None:
        return False

    energia_totale_depositata = float(dose_3d.sum())
    energia_totale_emessa = N * energia_media_singolo_fotone
    frazione_depositata = energia_totale_depositata / energia_totale_emessa

    voxel_colpiti = int((dose_3d > 0).sum())
    percentuale_volume = (voxel_colpiti / dose_3d.size) * 100

    print(f"\nRisultati energetici:")
    print(f"  Energia emessa dalla sorgente:  {energia_totale_emessa:.4e} MeV")
    print(f"  Energia rimasta nel fantoccio:  {energia_totale_depositata:.4e} MeV")
    print(f"  Frazione di energia catturata:  {frazione_depositata*100:.1f}%")
    print(f"  Volume d'acqua interessato   :     {percentuale_volume:.1f}% dei voxel")

    controlli = [
        ("Deposizione di energia :", energia_totale_depositata > 0),
        ("Conservazione energia  :", frazione_depositata < 1.0),
        ("Range fisico atteso    :", 0.30 < frazione_depositata < 0.90)
    ]

    esiti = []
    print("\nVerifica coerenza fisica:")
    for descrizione, superato in controlli:
        stato = "Coerente" if superato else "Non coerente"
        print(f"  {descrizione} {stato}")
        esiti.append(superato)
    print()

    fig, (ax_profilo, ax_barre) = plt.subplots(1, 2, figsize=(12, 5))
    energia_per_fetta_z = dose_3d.sum(axis=(1, 2))
    profondita = np.linspace(0, 30, len(energia_per_fetta_z))
    ax_profilo.plot(profondita, energia_per_fetta_z, 'b-', lw=2)
    ax_profilo.set_xlabel('Profondità (cm)')
    ax_profilo.set_ylabel('Energia per fetta (MeV)')
    ax_profilo.set_title('Distribuzione energia lungo l\'asse')
    ax_profilo.grid(True, alpha=0.3)

    energia_persa = energia_totale_emessa - energia_totale_depositata
    ax_barre.bar(['Catturata', 'Uscita'], [energia_totale_depositata, energia_persa], color=['#4682B4', '#CD5C5C'], edgecolor='black', width=0.5)
    ax_barre.set_ylabel('Energia Totale (MeV)')
    ax_barre.set_title(f'Bilancio finale (Resa: {frazione_depositata*100:.1f}%)')
    ax_barre.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    save_fig(fig, 'Test_5_Validazione_bilancio_energia')

    test_superato = all(esiti)
    print(f"\nTEST 5 SUPERATO: {test_superato}")


    return test_superato
risultato = test_5_bilancio_energetico()

ANALISI DEL BILANCIO ENERGETICO (Spettro 6MV)

Risultati energetici:
  Energia emessa dalla sorgente:  1.9118e+07 MeV
  Energia rimasta nel fantoccio:  1.0503e+07 MeV
  Frazione di energia catturata:  54.9%
  Volume d'acqua interessato   :     99.9% dei voxel

Verifica coerenza fisica:
  Deposizione di energia : Coerente
  Conservazione energia  : Coerente
  Range fisico atteso    : Coerente

    Figura salvata: ./GPU_CUDA/Test_5_Validazione_bilancio_energia.png

TEST 5 SUPERATO: True


## Test 6 - Validazione della forma del PDD

Viene valutata la qualità dosimetrica della simulazione confrontando il Percent Depth Dose (PDD) ottenuto con i parametri clinici standard e i dati di riferimento presenti in letteratura (Sheikh-Bagheri). Questo perchè si vuole garantire che il fascio simulato sia rappresentativo di un acceleratore lineare clinico reale. Viene individuata la profondità del massimo di dose che per un fascio da 6 MV deve avere un picco che si trovi tra 0 e 2 cm. La curva simulata viene confrontata punto per punto con dati di riferimento tramite un'interpolazione cubica.

In [83]:
def test_6_forma_pdd_clinico():

    print(f"ANALISI CLINICA PDD (Spettro 6MV)")

    profondita, dose = load_pdd(PDD_WATER)
    if profondita is None:
        return False

    indice_picco = np.argmax(dose)
    z_picco = profondita[indice_picco]
    valore_picco = dose[indice_picco]

    dose_10 = dose[np.abs(profondita - 10.0).argmin()]
    dose_20 = dose[np.abs(profondita - 20.0).argmin()]

    rapporto_10_max = dose_10 / valore_picco
    rapporto_20_10  = dose_20 / dose_10

    controlli = [
        ("Profondità del picco (0-2 cm) :", 0.0 <= z_picco <= 2.0, f"{z_picco:.2f} cm"),
        ("Rapporto Dose 10cm / Max      :", 0.60 <= rapporto_10_max <= 0.82, f"{rapporto_10_max:.3f}"),
        ("Rapporto Dose 20cm / 10cm     :", 0.45 <= rapporto_20_10 <= 0.72, f"{rapporto_20_10:.3f}")
    ]

    esiti = []
    print("\nVerifica parametri PDD:")
    for desc, superato, valore in controlli:
        stato = "Coerente" if superato else "Non coerente"
        print(f"  {desc} {valore} -> {stato}")
        esiti.append(superato)


    f_riferimento = interp1d(PROFONDITA_RIF, DOSE_RIF, kind='cubic', fill_value='extrapolate')

    zona = (profondita >= 1.5) & (profondita <= 25.0)
    differenze = np.abs(dose[zona] - f_riferimento(profondita[zona]))
    errore_medio = differenze.mean()

    giudizio = "Corretto" if errore_medio < 3.0 else "Accettabile" if errore_medio < 6.0 else "Errato"
    print(f"\nConfronto con dati storici (Sheikh-Bagheri):")
    print(f"  Errore medio (MAE): {errore_medio:.2f}% -> {giudizio}")
    print()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(profondita, dose, 'b-', lw=2.5, label='La tua Simulazione')
    ax.plot(PROFONDITA_RIF, DOSE_RIF, 'go', ms=5, label='Riferimento Letteratura')
    ax.axvline(z_picco, color='gray', ls='--', alpha=0.5, label=f'Picco a {z_picco:.1f} cm')
    ax.set_xlabel('Profondità (cm)')
    ax.set_ylabel('Dose (%)')
    ax.set_title('Confronto PDD: Simulazione e dati clinici')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 110)

    save_fig(fig, 'Test_6_confronto_pdd_clinico')

    test_superato = all(esiti)
    print(f"\nTEST 6 SUPERATO: {test_superato}")

    return test_superato

risultato = test_6_forma_pdd_clinico()

ANALISI CLINICA PDD (Spettro 6MV)

Verifica parametri PDD:
  Profondità del picco (0-2 cm) : 0.75 cm -> Coerente
  Rapporto Dose 10cm / Max      : 0.773 -> Coerente
  Rapporto Dose 20cm / 10cm     : 0.644 -> Coerente

Confronto con dati storici (Sheikh-Bagheri):
  Errore medio (MAE): 3.79% -> Accettabile

    Figura salvata: ./CPU_Sequenziale/Test_6_confronto_pdd_clinico.png

TEST 6 SUPERATO: True


## Test 7 - Verifica dell'eterogeneità

Viene valutata la capacità di gestire variazioni di densità elettronica e numero atomico all'interno del volume. Viene analizzato il comportamento del fascio quando attraversa un inserto di osso compatto immerso in un phantom d'acqua.

Viene confrontato il profilo di dose in un fantoccio omogeneo (acqua) con uno eterogeneo contenente un inserto osseo posizionato tra 12.5 cm e 17.5 cm. All'interno dell'osso deve esserci una variazione della dose depositata rispetto all'acqua a causa del maggiore coefficiente di attenuazione e del diverso potere d'arresto del mezzo più denso. L'osso, attenuando il fascio più efficacemente dell'acqua, deve generare una riduzione significativa della dose a valle (un "effetto ombra" atteso > 3% a 20 cm di profondità).

In [84]:
def test_7_verifica_eterogeneita_osso():
    print(f"ANALISI ETEROGENEITÀ (Acqua-Osso)")

    prof_acqua, pdd_acqua = load_pdd(PDD_WATER)
    prof_osso, pdd_osso = load_pdd(PDD_HETERO)

    if prof_acqua is None or prof_osso is None:
      return False

    fetta_inizio, fetta_fine = int(12.5 / VOXEL_CM), int(17.5 / VOXEL_CM)
    dose_media_acqua = pdd_acqua[fetta_inizio:fetta_fine].mean()
    dose_media_osso  = pdd_osso[fetta_inizio:fetta_fine].mean()

    fetta_20cm = int(20.0 / VOXEL_CM)
    dose_dopo_acqua = pdd_acqua[fetta_20cm]
    dose_dopo_osso  = pdd_osso[fetta_20cm]

    diff_dentro = dose_media_osso - dose_media_acqua
    diff_dopo   = dose_dopo_acqua - dose_dopo_osso

    print(f"\nRisultati zona ossea (12.5-17.5 cm):")
    print(f"  Dose in acqua    : {dose_media_acqua:.2f}% | Dose in osso: {dose_media_osso:.2f}%")
    print(f"  Variazione locale: {diff_dentro:+.2f}% (Attesa: Positiva)")

    print(f"\nRisultati a valle (20 cm):")
    print(f"  Dose senza osso: {dose_dopo_acqua:.2f}% | Dose con osso: {dose_dopo_osso:.2f}%")
    print(f"  Effetto ombra  : {diff_dopo:+.2f}% (Atteso: > 3%)")

    controlli = [
        ("Maggiore assorbimento nell'osso", diff_dentro > 0),
        ("Presenza effetto ombra dopo l'osso", diff_dopo > 0),
        ("Entità dell'ombra significativa (>3%)", diff_dopo > 3.0)
    ]

    esiti = []
    print("\nVerifica fisica:")
    for desc, superato in controlli:
        stato = "Coerente" if superato else "Non coerente"
        print(f"  {desc} {stato}")
        esiti.append(superato)
    print()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(prof_acqua, pdd_acqua, 'b-', label='Tutta acqua')
    ax.plot(prof_osso, pdd_osso, 'r--', label='Con inserto osseo')
    ax.axvspan(12.5, 17.5, color='orange', alpha=0.2, label='Posizione Osso')
    ax.set_xlabel('Profondità (cm)')
    ax.set_ylabel('Dose (%)')
    ax.set_title('Test Eterogeneità: Effetto dell\'Osso sulla Dose')
    ax.legend()
    ax.grid(alpha=0.3)

    save_fig(fig, 'Test_7_Validazione_eterogeneita')

    test_superato = all(esiti)
    print(f"\nTEST 7 SUPERATO: {test_superato}")

    return test_superato

risultato = test_7_verifica_eterogeneita_osso()

ANALISI ETEROGENEITÀ (Acqua-Osso)
  [ERRORE] file non trovato: ./CPU_Sequenziale/pdd_hetero.csv


## Test 8 - Analisi della penombra laterale e diffusione compton

Viene analizzata la qualità del fascio lungo l'asse trasversale con concentrazione principale sul fenomeno della penombra. L'obiettivo è quantificare come la nitidezza dei bordi del campo di radiazione degradi all'aumentare della profondità a causa dello scattering dei fotoni (principalmente diffusione Compton) e del trasporto degli elettroni secondari.

In [118]:
def test_8_verifica_penombra_laterale():

    print(f"ANALISI DIFFUSIONE LATERALE (Penombra Compton)")

    dose_3d = load_dose_bin(DOSE_WATER)
    if dose_3d is None:
      return False

    profondita_controllo = [3.0, 5.0, 10.0, 15.0, 20.0]
    misure_penombra = []
    profili_grafico = {}

    posizioni_x = (np.arange(NX) + 0.5) * VOXEL_CM - (PHANTOM_CM / 2.0)

    print(f"\n{'Profondità [cm]':>15} | {'Penombra [cm]':>15} | {'Andamento'}")
    print("-" * 50)

    for z in profondita_controllo:
        fetta_z = min(int(z / VOXEL_CM), NZ - 1)
        profilo = dose_3d[fetta_z, NY//2-4 : NY//2+5, :].mean(axis=0)
        profilo_pulito = np.convolve(profilo, np.ones(3)/3, mode='same')
        profilo_norm = (profilo_pulito / profilo_pulito.max()) * 100

        idx_80 = np.where(profilo_norm >= 80)[0][-1]
        idx_20 = np.where(profilo_norm >= 20)[0][-1]

        distanza = posizioni_x[idx_20] - posizioni_x[idx_80]
        misure_penombra.append(distanza)
        profili_grafico[z] = profilo_norm

        trend = ""
        if len(misure_penombra) < 2:
            trend = "In crescita"
        elif distanza >= misure_penombra[-2] - 0.05:
            trend = "Normale"
        else:
            trend = "Anomalo"

        print(f"{z:>15.1f} | {distanza:>15.2f} | {trend}")

    rapporto_20_3 = misure_penombra[-1] / misure_penombra[0]
    test_superato = rapporto_20_3 > 1.10

    print(f"Risultati:")
    print(f"  Penombra a 3cm :  {misure_penombra[0]:.2f} cm")
    print(f"  Penombra a 20cm: {misure_penombra[-1]:.2f} cm")
    print(f"  Rapporto (20/3): {rapporto_20_3:.2f} (Target > 1.10)")
    print()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    for z in profondita_controllo:
        ax1.plot(posizioni_x, profili_grafico[z], label=f"z = {z} cm")
    ax1.set_title("Sfumatura del bordo (Penombra)")
    ax1.set_xlabel("Posizione laterale (cm)")
    ax1.set_ylabel("Dose %")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(profondita_controllo, misure_penombra, 'ro-', lw=2)
    ax2.set_title("Allargamento del fascio")
    ax2.set_xlabel("Profondità (cm)")
    ax2.set_ylabel("Larghezza penombra (cm)")
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    save_fig(fig, 'Test_8_Validazione_penombra_laterale')

    print(f"\nTEST 8 SUPERATO: {test_superato}")

    return test_superato
risultato = test_8_verifica_penombra_laterale()

ANALISI DIFFUSIONE LATERALE (Penombra Compton)

Profondità [cm] |   Penombra [cm] | Andamento
--------------------------------------------------
            3.0 |            0.60 | In crescita
            5.0 |            0.60 | Normale
           10.0 |            0.60 | Normale
           15.0 |            0.90 | Normale
           20.0 |            0.90 | Normale
Risultati:
  Penombra a 3cm :  0.60 cm
  Penombra a 20cm: 0.90 cm
  Rapporto (20/3): 1.50 (Target > 1.10)

    Figura salvata: ./GPU_CUDA/Test_8_Validazione_penombra_laterale.png

TEST 8 SUPERATO: True


## Test 9 - Analisi Gamma Index (2%/3mm)

Il Gamma Index è lo standard internazionale per il confronto quantitativo tra due distribuzioni di dose. Vengono integrate simultaneamente le differenze nella dose e le discrepanze spaziali fornendo un punteggio univoco che certifica l'accuratezza della simulazione rispetto allo standard di riferimento.

In [130]:
def test_9_gamma_index():

    profondita_mc, dose_mc_grezza = load_pdd(PDD_WATER)
    if profondita_mc is None:
        print("Errore: dati della simulazione non trovati.")
        return False

    profondita_rif = PROFONDITA_RIF
    dose_rif_storica = DOSE_RIF

    idx_10cm_mc = np.argmin(np.abs(profondita_mc - 10.0))
    idx_10cm_rif = np.argmin(np.abs(profondita_rif - 10.0))

    valore_10cm_mc = dose_mc_grezza[idx_10cm_mc]
    valore_10cm_rif = dose_rif_storica[idx_10cm_rif]

    dose_mc_percentuale = (dose_mc_grezza / valore_10cm_mc) * 100.0
    dose_rif_percentuale = (dose_rif_storica / valore_10cm_rif) * 100.0

    tolleranza_dose = 3.0
    tolleranza_distanza = 0.3

    def calcola_gamma_index(d_test, d_rif, z_test, z_rif):
        mc_su_griglia_rif = np.interp(z_rif, z_test, d_test)
        risultati = np.zeros(len(z_rif))
        for i in range(len(z_rif)):
            diff_dose_quadrata = ((mc_su_griglia_rif - d_rif[i]) / tolleranza_dose)**2
            diff_distanza_quadrata = ((z_rif - z_rif[i]) / tolleranza_distanza)**2
            risultati[i] = np.sqrt(np.min(diff_dose_quadrata + diff_distanza_quadrata))
        return risultati, mc_su_griglia_rif

    valori_gamma, mc_interpolata = calcola_gamma_index(dose_mc_percentuale, dose_rif_percentuale, profondita_mc, profondita_rif)

    maschera_zona_valida = (profondita_rif >= 3.0) & (profondita_rif <= 25.0)

    punti_ok = valori_gamma[maschera_zona_valida] <= 1.0
    percentuale_passati = np.mean(punti_ok) * 100

    soglia_minima = 70.0
    test_superato = percentuale_passati >= soglia_minima

    print("ANALISI GAMMA INDEX\n")
    print(f"Pass Rate Gamma (3%/3mm): {percentuale_passati:.1f}%")
    print(f"Soglia richiesta: {soglia_minima}%")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    ax1.plot(profondita_rif, dose_rif_percentuale, 'r--', label="Riferimento (Sheikh-Bagheri)")
    ax1.plot(profondita_mc, dose_mc_percentuale, 'b-', label="Nostro Simulatore (MC)")
    ax1.axvspan(0, 3, color='gray', alpha=0.1, label="Zona Build-up (Approssimata)")
    ax1.set_title("Andamento della Dose (Normalizzato a 10cm)")
    ax1.set_xlabel("Profondità [cm]")
    ax1.set_ylabel("Dose [%]")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    colori_barre = ['green' if g <= 1.0 else 'red' for g in valori_gamma[maschera_zona_valida]]
    ax2.bar(profondita_rif[maschera_zona_valida], valori_gamma[maschera_zona_valida], width=0.3, color=colori_barre)
    ax2.axhline(1.0, color='black', linestyle='--')
    ax2.set_title("Indice Gamma (zona > 3cm)")
    ax2.set_xlabel("Profondità [cm]")
    ax2.set_ylabel("Valore Gamma")
    ax2.grid(True, alpha=0.2)

    plt.tight_layout()
    save_fig(fig, 'Test_9_Validazione_PDD_Gamma')

    print(f"\nTEST 9 SUPERATO: {test_superato}")
    return test_superato

risultato = test_9_gamma_index()

ANALISI GAMMA INDEX

Pass Rate Gamma (3%/3mm): 54.2%
Soglia richiesta: 70.0%
    Figura salvata: ./GPU_CUDA/Test_9_Validazione_PDD_Gamma.png

TEST 9 SUPERATO: False


## Test 10 - Validazione della convergenza statistica

Viene verificata la robustezza stocastica del simulatore Monte Carlo per confermare che l'incertezza statistica della dose depositata diminuisca correttamente all'aumentare del numero di storie simulate, seguendo la legge fondamentale della statistica per i processi indipendenti.

Viene eseguita una serie di simulazioni con un numero crescente di fotoni. Per ogni configurazione, la simulazione viene ripetuta più volte tramite prove indipendenti utilizzando seed casuali diversi per calcolare la varianza dei risultati. Per ogni valore di N, viene calcolata la deviazione standard della dose media depositata per fotone che rappresenta l'errore statistico intrinseco della simulazione.

In [120]:
def test_10_convergenza_statistica(salta_lento: bool = False):

    configurazioni = [(10000, 20), (100000, 20), (1000000, 20)]
    if not salta_lento:
        configurazioni.append((10000000, 5))

    print(f" ANALISI DELLA CONVERGENZA STATISTICA\n")
    print(f"{'N Fotoni':>12} | {'Prove':>6} | {'Media E_dep':>14} | {'Incertezza (rho)':>14}")
    print("-" * 65)

    lista_sigma = []
    lista_N = []

    for N, n_ripetizioni in configurazioni:
        risultati_energia = []

        for i in range(n_ripetizioni):
            if lancia_simulazione(N, tipo_phantom=0, seed=500 + i):
                dati_dose = load_dose_bin(DOSE_WATER)
                if dati_dose is not None:
                    risultati_energia.append(dati_dose.sum() / N)

        if len(risultati_energia) < 3:
            continue

        media = np.mean(risultati_energia)
        dev_standard = np.std(risultati_energia, ddof=1)

        lista_sigma.append(dev_standard)
        lista_N.append(N)

        print(f"{N:>12,} | {len(risultati_energia):>6} | {media:>14.4e} | {dev_standard:>14.4e}")
        print()

    log_N = np.log10(lista_N)
    log_sigma = np.log10(lista_sigma)
    pendenza, intercetta = np.polyfit(log_N, log_sigma, 1)

    test_superato = abs(pendenza - (-0.5)) < 0.15

    print(f"\nAnalisi della pendenza:")
    print(f"  Pendenza misurata: {pendenza:.3f}")
    print(f"  Pendenza attesa:   -0.500")
    print(f"  Risultato:         {'Coerente' if test_superato else 'Non coerente'}")
    print()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.loglog(lista_N, lista_sigma, 'bo', label='Incertezza misurata (σ)')

    N_teorico = np.logspace(np.log10(min(lista_N)), np.log10(max(lista_N)), 100)
    sigma_teorica = lista_sigma[0] * np.sqrt(lista_N[0] / N_teorico)
    ax.loglog(N_teorico, sigma_teorica, 'r-', alpha=0.7, label='Teoria 1/√N (pendenza -0.5)')

    ax.set_xlabel('Numero di particelle (N)')
    ax.set_ylabel('Incertezza (MeV/fotone)')
    ax.set_title(f'Test di Convergenza: {"Superato" if test_superato else "Fallito"}')
    ax.legend()
    ax.grid(True, which="both", alpha=0.3)

    save_fig(fig, 'Test_10_Validazione_Convergenza')

    print(f"\nTEST 10 SUPERATO: {test_superato}")
    return test_superato
risultato = test_10_convergenza_statistica()

 ANALISI DELLA CONVERGENZA STATISTICA

    N Fotoni |  Prove |    Media E_dep | Incertezza (rho)
-----------------------------------------------------------------
      10,000 |     20 |     1.0500e+00 |     9.5552e-03

     100,000 |     20 |     1.0510e+00 |     3.3452e-03

   1,000,000 |     20 |     1.0507e+00 |     7.8740e-04

  10,000,000 |      5 |     1.0505e+00 |     3.5049e-04


Analisi della pendenza:
  Pendenza misurata: -0.493
  Pendenza attesa:   -0.500
  Risultato:         Coerente

    Figura salvata: ./GPU_CUDA/Test_10_Validazione_Convergenza.png

TEST 10 SUPERATO: True


# Valutazione tempi diverse versioni

In [ ]:
import subprocess
import csv
import time
import pandas as pd
import matplotlib.pyplot as plt

versioni = [
    {"nome": "CPU_Sequenziale_Acqua",       "comando": "./CPU_Sequenziale/mc_rt_cpu_sequenziale", "args": ["0", "42"], "cores": 1},
    {"nome": "CPU_Sequenziale_Acqua_Osso",  "comando": "./CPU_Sequenziale/./mc_rt_cpu_sequenziale", "args": ["1", "42"], "cores": 8},
    {"nome": "CPU_Parallelo_Acqua",         "comando": "./CPU_Parallelo/./mc_rt_cpu_parallelo", "args": ["0", "42"], "cores": 8},
    {"nome": "CPU_Parallelo_Acqua_Osso",    "comando": "./CPU_Parallelo/./mc_rt_cpu_parallelo", "args": ["1", "42"], "cores": 8},

]

fotoni_test = [10, 10**2, 10**3, 10**4, 10**5, 10**6, 10**7]
risultati = []

for n in fotoni_test:
    tempo_baseline = None
    for v in versioni:
        print(f"Test: {v['nome']}       | Fotoni: {n}")
        cmd = [v['comando'], str(n)] + v['args']

        start = time.perf_counter()
        subprocess.run(cmd, capture_output=True, text=True)
        fine = time.perf_counter()

        tempo_esecuzione = fine - start

        if v['nome'] == "CPU_Seq":
            tempo_baseline = tempo_esecuzione

        speedup = tempo_baseline / tempo_esecuzione if tempo_baseline else 1.0
        efficienza = speedup / v['cores']
        throughput = n / tempo_esecuzione

        risultati.append({
            "Fotoni": n,
            "Versione": v['nome'],
            "Tempo": tempo_esecuzione,
            "Speedup": speedup,
            "Efficienza": efficienza,
            "Throughput": throughput
        })

df = pd.DataFrame(risultati)
df.to_csv('benchmark_hpc.csv', index=False)
print("\nBenchmark completato e salvato in 'benchmark_hpc.csv'")

Test: CPU_Sequenziale_Acqua       | Fotoni: 10
Test: CPU_Sequenziale_Acqua_Osso       | Fotoni: 10
Test: CPU_Parallelo_Acqua       | Fotoni: 10
Test: CPU_Parallelo_Acqua_Osso       | Fotoni: 10
Test: CPU_Sequenziale_Acqua       | Fotoni: 100
Test: CPU_Sequenziale_Acqua_Osso       | Fotoni: 100
Test: CPU_Parallelo_Acqua       | Fotoni: 100
Test: CPU_Parallelo_Acqua_Osso       | Fotoni: 100
Test: CPU_Sequenziale_Acqua       | Fotoni: 1000
Test: CPU_Sequenziale_Acqua_Osso       | Fotoni: 1000
Test: CPU_Parallelo_Acqua       | Fotoni: 1000
Test: CPU_Parallelo_Acqua_Osso       | Fotoni: 1000
Test: CPU_Sequenziale_Acqua       | Fotoni: 10000
Test: CPU_Sequenziale_Acqua_Osso       | Fotoni: 10000
Test: CPU_Parallelo_Acqua       | Fotoni: 10000
Test: CPU_Parallelo_Acqua_Osso       | Fotoni: 10000
Test: CPU_Sequenziale_Acqua       | Fotoni: 100000
Test: CPU_Sequenziale_Acqua_Osso       | Fotoni: 100000
Test: CPU_Parallelo_Acqua       | Fotoni: 100000
Test: CPU_Parallelo_Acqua_Osso       | Foton

In [ ]:
def save_fig(fig, name):
    path = f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f"    Figura salvata: {path}")

In [ ]:
df = pd.read_csv('benchmark_hpc.csv')

fig, axs = plt.subplots(2, 2, figsize=(15, 12))

for nome in df['Versione'].unique():
    subset = df[df['Versione'] == nome]
    axs[0, 0].plot(subset['Fotoni'], subset['Tempo'], marker='o', label=nome)

axs[0, 0].set_title('Tempo di Esecuzione (Lower is better)')
axs[0, 0].set_xscale('log')
axs[0, 0].set_yscale('log')
axs[0, 0].set_ylabel('Secondi')
axs[0, 0].legend()
axs[0, 0].grid(True)

for nome in df['Versione'].unique():
    subset = df[df['Versione'] == nome]
    axs[0, 1].plot(subset['Fotoni'], subset['Speedup'], marker='s', label=nome)

axs[0, 1].set_title('Speedup (Higher is better)')
axs[0, 1].set_xscale('log')
axs[0, 1].set_ylabel('S = T_seq / T_par')
axs[0, 1].legend()
axs[0, 1].grid(True)

for nome in df['Versione'].unique():
    subset = df[df['Versione'] == nome]
    axs[1, 0].plot(subset['Fotoni'], subset['Throughput'], marker='^', label=nome)

axs[1, 0].set_title('Throughput: Fotoni al secondo')
axs[1, 0].set_xscale('log')
axs[1, 0].set_yscale('log')
axs[1, 0].set_ylabel('Fotoni/s')
axs[1, 0].legend()
axs[1, 0].grid(True)

df_max = df[df['Fotoni'] == 10**8]
axs[1, 1].bar(df_max['Versione'], df_max['Speedup'], color=['gray', 'blue', 'green'])
axs[1, 1].set_title('Strong Scaling (Fixed Load: 10^8 photons)')
axs[1, 1].set_ylabel('Speedup')

plt.tight_layout()
save_fig(fig, 'benchmark_plots')

    Figura salvata: benchmark_plots.png
